In [4]:
import numpy as np
from scipy.spatial.distance import euclidean
import pandas as pd
import copy
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

import sys
import argparse
import glob
import cmlreaders as cml
import json
from matplotlib.ticker import FuncFormatter
import re
from scipy import signal
from scipy.stats import zscore
import mne
from mne import create_info, EpochsArray
warnings.filterwarnings('ignore')

# ============================================================================
# IMPORT IRASA (assumes you have it available)
# ============================================================================

import cmlreaders as cml
from irasa.IRASA import IRASA
try:
    from statsmodels.formula.api import mixedlm
    from statsmodels.stats.multitest import fdrcorrection
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'statsmodels', '-q'])
    from statsmodels.formula.api import mixedlm
    from statsmodels.stats.multitest import fdrcorrection
import os
import pandas as pd
import numpy as np
import cmlreaders as cml
from scipy.spatial.distance import euclidean
import copy
import cmlreaders as cml

whole_df = cml.CMLReader.get_data_index()
exp = 'DBOY1'
subjects = whole_df.query('experiment == @exp')['subject'].unique()
subject = subjects[0]

sub_df = whole_df.query('experiment == @exp and subject == @subject')
session = sub_df['session'].unique()[0]

reader = cml.CMLReader(subject, exp, session=session)
evs = reader.load('task_events')

print(f"Subject: {subject}, Session: {session}")
print(f"Shape: {evs.shape}")
print(evs.head(20))

Subject: R1494D, Session: 0
Shape: (428, 28)
    eegoffset  correctPointingDirection                         eegfile  \
0      882107                 -999.0000  R1494D_DBOY1_0_28Jan20_2038.h5   
1      884123                 -999.0000  R1494D_DBOY1_0_28Jan20_2038.h5   
2      886040                 -999.0000  R1494D_DBOY1_0_28Jan20_2038.h5   
3      888173                 -999.0000  R1494D_DBOY1_0_28Jan20_2038.h5   
4      890173                 -999.0000  R1494D_DBOY1_0_28Jan20_2038.h5   
5      892174                 -999.0000  R1494D_DBOY1_0_28Jan20_2038.h5   
6      894122                 -999.0000  R1494D_DBOY1_0_28Jan20_2038.h5   
7      896207                 -999.0000  R1494D_DBOY1_0_28Jan20_2038.h5   
8      898172                 -999.0000  R1494D_DBOY1_0_28Jan20_2038.h5   
9      900206                 -999.0000  R1494D_DBOY1_0_28Jan20_2038.h5   
10     902123                 -999.0000  R1494D_DBOY1_0_28Jan20_2038.h5   
11     904256                 -999.0000  R1494D_DBOY1_0

In [5]:
import cmlreaders as cml
import pandas as pd
import numpy as np

pd.set_option('display.max_rows', None)

whole_df = cml.CMLReader.get_data_index()
exp = 'DBOY1'
subjects = whole_df.query('experiment == @exp')['subject'].unique()

print(f"Found {len(subjects)} subjects")

Found 44 subjects


In [9]:
import cmlreaders as cml
import pandas as pd
import numpy as np
from joblib import Parallel, delayed

pd.set_option('display.max_rows', None)

# ============================================================================
# LOAD IRASA CSV
# ============================================================================

df = pd.read_csv('ALL_SUBJECTS_irasa_clean.csv')

# ============================================================================
# FILTER TO GAMMA BAND AND BUILD LONG FORMAT
# ============================================================================

gamma     = df[df['freq_hz'] >= 40].copy()
gamma     = gamma[gamma['SP_clustering_from_prev'].notna()].copy()
gamma_avg = gamma.copy()

sp = gamma.copy()
sp['clustering_score'] = sp['SP_clustering_from_prev']
sp['clustering_type']  = 'SP'

tc = gamma.copy()
tc['clustering_score'] = tc['T_clustering_from_prev']
tc['clustering_type']  = 'T'

long = pd.concat([sp, tc], ignore_index=True)
long['clust_T']    = (long['clustering_type'] == 'T').astype(int)
long['region_RHP'] = (long['region'] == 'RHP').astype(int)
long['sess_id']    = long['subject'].astype(str) + '_' + long['session'].astype(str)

for col in ['encoding_osc_ssl', 'encoding_frac_ssl',
            'retrieval_osc_ssl', 'retrieval_frac_ssl']:
    subj_mean_map          = gamma_avg.groupby('subject')[col].mean()
    long[col + '_between'] = long['subject'].map(subj_mean_map)
    long[col + '_within']  = long[col] - long[col + '_between']

# ============================================================================
# PARALLEL WORKER — one subject × session
# ============================================================================

def process_session(subject, session, exp):
    import cmlreaders as cml
    import gc

    recall_rows = []
    nav_rows    = []

    try:
        reader = cml.CMLReader(subject, exp, session=session)
        evs    = reader.load('task_events').reset_index(drop=True)

        # ── GET SAMPLING RATE ─────────────────────────────────────────────
        sr = 1000  # default
        try:
            one_event = evs[evs['type'] == 'WORD'].iloc[[0]]
            eeg_small = reader.load('eeg', events=one_event, rel_start=0, rel_stop=1)
            sr        = float(eeg_small.samplerate)
            del eeg_small
            gc.collect()
        except Exception:
            sr = 1000
            print(f"  ! {subject} sess {session}: sr fallback to 1000")

        # ── LOOP OVER TRIALS ──────────────────────────────────────────────
        for trial, trial_evs in evs.groupby('trial'):
            if trial < 0:
                continue

            trial_evs = trial_evs.reset_index(drop=True)

            word_evs = trial_evs[trial_evs['type'] == 'WORD']
            n_words  = len(word_evs)
            if n_words == 0:
                continue

            rec_words   = trial_evs[trial_evs['type'] == 'REC_WORD']
            n_correct   = len(rec_words[rec_words['itemno'] > 0])
            n_intrusion = len(rec_words[rec_words['itemno'] < 0])

            recall_rows.append({
                'subject':     subject,
                'session':     session,
                'trial':       trial,
                'experiment':  exp,
                'n_words':     n_words,
                'n_correct':   n_correct,
                'n_intrusion': n_intrusion,
                'recall_rate': n_correct / n_words,
                'samplerate':  sr,
            })

            for i, row in trial_evs.iterrows():
                if row['type'] == 'pointing finished':
                    subsequent = trial_evs[trial_evs.index > i]
                    next_word  = subsequent[subsequent['type'] == 'WORD']

                    if len(next_word) > 0:
                        next_word_row    = next_word.iloc[0]
                        nav_time_samples = next_word_row['eegoffset'] - row['eegoffset']
                        nav_time_seconds = nav_time_samples / sr

                        nav_rows.append({
                            'subject':                 subject,
                            'session':                 session,
                            'trial':                   trial,
                            'experiment':              exp,
                            'samplerate':              sr,
                            'navigation_time_samples': nav_time_samples,
                            'navigation_time_seconds': nav_time_seconds,
                        })

    except Exception as e:
        print(f"  ✗ {subject} sess {session} ({exp}): {e}")

    return recall_rows, nav_rows
# ============================================================================
# BUILD TASK LIST
# ============================================================================

whole_df       = cml.CMLReader.get_data_index()
EXPERIMENTS    = ['DBOY1', 'EFRCourierReadOnly']
irasa_subjects = set(long['subject'].unique())

tasks = []
for exp in EXPERIMENTS:
    all_exp_subs = set(whole_df.query('experiment == @exp')['subject'].unique())
    subjects     = sorted(irasa_subjects & all_exp_subs)
    for subject in subjects:
        sub_df   = whole_df.query('experiment == @exp and subject == @subject')
        sessions = sub_df['session'].unique()
        for session in sessions:
            tasks.append((subject, session, exp))

print(f"Total subject-session tasks: {len(tasks)}")

# ============================================================================
# RUN IN PARALLEL
# ============================================================================

N_JOBS = 12  # conservative to avoid OOM on rhino2

print(f"Running on {N_JOBS} cores...")

results = Parallel(n_jobs=N_JOBS, verbose=5)(
    delayed(process_session)(subject, session, exp)
    for subject, session, exp in tasks
)

# ============================================================================
# COLLECT RESULTS
# ============================================================================

all_recall_rows = []
all_nav_rows    = []

for recall_rows, nav_rows in results:
    all_recall_rows.extend(recall_rows)
    all_nav_rows.extend(nav_rows)

recall_df = pd.DataFrame(all_recall_rows)
nav_df    = pd.DataFrame(all_nav_rows)

print(f"\nrecall_df shape: {recall_df.shape}")
print(f"nav_df shape:    {nav_df.shape}")
print(f"Experiments in recall_df: {recall_df['experiment'].unique().tolist()}")
print(f"Subjects in recall_df:    {sorted(recall_df['subject'].unique())}")

# QC — flag suspiciously long navigation times
nav_df['nav_too_long'] = nav_df['navigation_time_seconds'] > 180
print(f"\nNav events > 3 min: {nav_df['nav_too_long'].sum()}")
if nav_df['nav_too_long'].sum() > 0:
    print(nav_df[nav_df['nav_too_long']][
        ['subject', 'session', 'trial', 'navigation_time_seconds', 'samplerate']
    ].to_string(index=False))

# Sampling rate summary per subject
print("\nSampling rate per subject:")
print(nav_df.groupby('subject')['samplerate'].unique().to_string())

# ============================================================================
# TRIAL-LEVEL AGGREGATION
# ============================================================================

# Nav time — mean across pointing events per trial
nav_trial = (
    nav_df.groupby(['subject', 'session', 'trial'])
    .agg(
        nav_time_seconds = ('navigation_time_seconds', 'mean'),
        samplerate       = ('samplerate',              'first'),
        n_nav_events     = ('navigation_time_seconds', 'count'),
    )
    .reset_index()
)

# Recall rate per trial
trial_recall = (
    recall_df[['subject', 'session', 'trial', 'recall_rate',
               'n_correct', 'n_words', 'n_intrusion']]
    .drop_duplicates(subset=['subject', 'session', 'trial'])
)

# Gamma — mean across frequencies and words within trial, per region
gamma_trial = (
    gamma_avg
    .groupby(['subject', 'session', 'trial', 'region'])
    .agg(
        encoding_frac_ssl  = ('encoding_frac_ssl',  'mean'),
        encoding_osc_ssl   = ('encoding_osc_ssl',   'mean'),
        retrieval_frac_ssl = ('retrieval_frac_ssl',  'mean'),
        retrieval_osc_ssl  = ('retrieval_osc_ssl',   'mean'),
    )
    .reset_index()
)

gamma_trial_wide = gamma_trial.pivot_table(
    index   = ['subject', 'session', 'trial'],
    columns = 'region',
    values  = ['encoding_frac_ssl', 'encoding_osc_ssl',
               'retrieval_frac_ssl', 'retrieval_osc_ssl'],
).reset_index()

gamma_trial_wide.columns = [
    '_'.join(filter(None, map(str, col))).strip()
    if col[1] != '' else col[0]
    for col in gamma_trial_wide.columns
]

# SP and TC clustering — mean across recalled words per trial per region
clust_trial = (
    long[long['clust_T'] == 0]
    .groupby(['subject', 'session', 'trial', 'region'])
    .agg(sp_clustering = ('clustering_score', 'mean'))
    .reset_index()
    .merge(
        long[long['clust_T'] == 1]
        .groupby(['subject', 'session', 'trial', 'region'])
        .agg(tc_clustering = ('clustering_score', 'mean'))
        .reset_index(),
        on  = ['subject', 'session', 'trial', 'region'],
        how = 'outer'
    )
)

clust_trial_wide = clust_trial.pivot_table(
    index   = ['subject', 'session', 'trial'],
    columns = 'region',
    values  = ['sp_clustering', 'tc_clustering'],
).reset_index()

clust_trial_wide.columns = [
    '_'.join(filter(None, map(str, col))).strip()
    if col[1] != '' else col[0]
    for col in clust_trial_wide.columns
]

# ============================================================================
# MERGE ALL INTO ONE TRIAL-LEVEL CSV
# ============================================================================

trial_df = (
    gamma_trial_wide
    .merge(clust_trial_wide, on=['subject', 'session', 'trial'], how='outer')
    .merge(trial_recall,     on=['subject', 'session', 'trial'], how='left')
    .merge(nav_trial,        on=['subject', 'session', 'trial'], how='left')
    .sort_values(['subject', 'session', 'trial'])
    .reset_index(drop=True)
)

# ============================================================================
# SANITY CHECK
# ============================================================================

print(f"\ntrial_df shape: {trial_df.shape}")
print(f"Subjects: {sorted(trial_df['subject'].unique())}")
print(f"\nColumn list:")
for c in trial_df.columns:
    n_nan = trial_df[c].isna().sum()
    print(f"  {c:<45} NaNs: {n_nan}")

print(f"\nSample rows:")
print(trial_df.head(10).to_string(index=False))

print(f"\nNav time (seconds) summary:")
print(trial_df['nav_time_seconds'].describe())

print(f"\nRecall rate summary:")
print(trial_df['recall_rate'].describe())

# ============================================================================
# SAVE
# ============================================================================

trial_df.to_csv('trial_level_summary.csv', index=False)
print("\nSaved: trial_level_summary.csv")
print(f"Shape: {trial_df.shape}")
print(f"Columns: {trial_df.columns.tolist()}")

Total subject-session tasks: 84
Running on 12 cores...


[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  48 tasks      | elapsed:   47.9s
[Parallel(n_jobs=12)]: Done  78 out of  84 | elapsed:  1.3min remaining:    5.8s



recall_df shape: (278, 9)
nav_df shape:    (3334, 7)
Experiments in recall_df: ['DBOY1', 'EFRCourierReadOnly']
Subjects in recall_df:    ['R1494D', 'R1501J', 'R1502D', 'R1503E', 'R1504E', 'R1509E', 'R1520T', 'R1521E', 'R1523E', 'R1529T', 'R1530J', 'R1532T', 'R1536J', 'R1537T', 'R1538E', 'R1539E', 'R1542J', 'R1543E', 'R1546E', 'R1551T', 'R1554T', 'R1560T', 'R1561E', 'R1564J', 'R1567T', 'R1569T', 'R1570T', 'R1571T', 'R1572T', 'R1573T', 'R1575E', 'R1620J', 'R1637T', 'R1642J', 'R1651J', 'R1653J']

Nav events > 3 min: 2
subject  session  trial  navigation_time_seconds  samplerate
 R1509E        0      1               260.813477      1024.0
 R1538E        1      0               363.142090      2048.0

Sampling rate per subject:
subject
R1494D            [1000.0]
R1501J            [2000.0]
R1502D            [1000.0]
R1503E            [2048.0]
R1504E            [2048.0]
R1509E            [1024.0]
R1520T            [1000.0]
R1521E            [2048.0]
R1523E            [2048.0]
R1529T          

[Parallel(n_jobs=12)]: Done  84 out of  84 | elapsed:  1.3min finished


In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

# ============================================================================
# SUBJECT-LEVEL BEHAVIORAL AGGREGATES
# ============================================================================

recall_subj = (
    recall_df.groupby('subject')
    .agg(mean_recall_rate = ('recall_rate', 'mean'))
    .reset_index()
)

nav_clean = nav_df[nav_df['navigation_time_seconds'] <= 180].copy()
nav_subj = (
    nav_clean.groupby('subject')
    .agg(
        mean_nav_time_seconds = ('navigation_time_seconds', 'mean'),  # seconds only
        n_nav_events          = ('navigation_time_seconds', 'count'),
    )
    .reset_index()
)

# Fixed: aggregate from full long, not clust_T==0 filtered
clust_subj = (
    long.groupby('subject')
    .agg(
        mean_sp_clustering = ('SP_clustering_from_prev', 'mean'),
        mean_tc_clustering = ('T_clustering_from_prev',  'mean'),
    )
    .reset_index()
)

subj_behav = (
    clust_subj
    .merge(recall_subj, on='subject', how='inner')
    .merge(nav_subj,    on='subject', how='inner')
)

# ============================================================================
# SUBJECT-LEVEL NEURAL AGGREGATES — split by region
# ============================================================================

neural_cols = [
    'encoding_frac_ssl', 'encoding_osc_ssl',
    'retrieval_frac_ssl', 'retrieval_osc_ssl',
]

subj_neural_rows = []

for subj_id, grp in gamma_avg.groupby('subject'):
    row = {'subject': subj_id}
    for col in neural_cols:
        for region, rgrp in grp.groupby('region'):
            tag = 'LHP' if region == 'LHP' else 'RHP'
            row[f'{col}_{tag}'] = rgrp[col].mean()
    subj_neural_rows.append(row)

subj_neural = pd.DataFrame(subj_neural_rows)

# ============================================================================
# MERGE BEHAVIORAL + NEURAL
# ============================================================================

subj = subj_behav.merge(subj_neural, on='subject', how='inner')
subj = subj.loc[:, ~subj.columns.duplicated()]

# Fixed: dropna on mean_nav_time_seconds, not mean_nav_time_samples
subj = subj.dropna(subset=['mean_sp_clustering', 'mean_tc_clustering',
                            'mean_recall_rate',   'mean_nav_time_seconds'])

print(f"Subjects with all variables: {len(subj)}")
print(f"Columns: {subj.columns.tolist()}")

# ============================================================================
# VARIABLE DEFINITIONS FOR MATRIX
# ============================================================================

var_defs = {
    'Nav time (s)':  'mean_nav_time_seconds',
    'SP clustering': 'mean_sp_clustering',
    'TC clustering': 'mean_tc_clustering',
    'Recall rate':   'mean_recall_rate',
    'Enc frac LHP':  'encoding_frac_ssl_LHP',
    'Enc frac RHP':  'encoding_frac_ssl_RHP',
    'Enc osc LHP':   'encoding_osc_ssl_LHP',
    'Enc osc RHP':   'encoding_osc_ssl_RHP',
    'Ret frac LHP':  'retrieval_frac_ssl_LHP',
    'Ret frac RHP':  'retrieval_frac_ssl_RHP',
    'Ret osc LHP':   'retrieval_osc_ssl_LHP',
    'Ret osc RHP':   'retrieval_osc_ssl_RHP',
}

# Keep only variables whose columns actually exist in subj
var_defs = {k: v for k, v in var_defs.items() if v in subj.columns}
missing  = [v for v in var_defs.values() if v not in subj.columns]
if missing:
    print(f"WARNING — missing columns dropped: {missing}")

labels = list(var_defs.keys())
cols   = list(var_defs.values())
n_vars = len(labels)

# ============================================================================
# CORRELATION MATRIX + p-VALUES
# ============================================================================

corr_mat = np.full((n_vars, n_vars), np.nan)
p_mat    = np.full((n_vars, n_vars), np.nan)

for i, ci in enumerate(cols):
    for j, cj in enumerate(cols):
        xi = subj[ci].squeeze()
        xj = subj[cj].squeeze()
        valid_mask = xi.notna() & xj.notna()
        xi_vals = xi[valid_mask].values.astype(float)
        xj_vals = xj[valid_mask].values.astype(float)
        if len(xi_vals) < 4:
            continue
        if xi_vals.std() < 1e-10 or xj_vals.std() < 1e-10:
            continue
        r, p = stats.pearsonr(xi_vals, xj_vals)
        corr_mat[i, j] = r
        p_mat[i, j]    = p

# ============================================================================
# PRINT CORRELATION TABLE
# ============================================================================

print("\nPearson correlations (r / p):")
print(f"{'':20s}" + "".join(f"{l:>14s}" for l in labels))
for i, li in enumerate(labels):
    row_str = f"{li:20s}"
    for j in range(n_vars):
        if j <= i:
            row_str += f"{'—':>14s}"
        else:
            r = corr_mat[i, j]
            p = p_mat[i, j]
            if np.isnan(r):
                row_str += f"{'NaN':>14s}"
            else:
                stars = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
                row_str += f"{r:+.2f}{stars:3s}{'':5s}"
    print(row_str)

# ============================================================================
# FIGURE — full correlation matrix heatmap
# ============================================================================

group_colors = {
    'Nav time (s)':  '#555555',
    'SP clustering': '#2166AC',
    'TC clustering': '#D6604D',
    'Recall rate':   '#1A9850',
    'Enc frac LHP':  '#7B2D8B',
    'Enc frac RHP':  '#B15CC8',
    'Enc osc LHP':   '#C07020',
    'Enc osc RHP':   '#E8A040',
    'Ret frac LHP':  '#7B2D8B',
    'Ret frac RHP':  '#B15CC8',
    'Ret osc LHP':   '#C07020',
    'Ret osc RHP':   '#E8A040',
}

fig, ax = plt.subplots(figsize=(13, 11))
fig.patch.set_facecolor('white')

im = ax.imshow(corr_mat, vmin=-1, vmax=1, cmap='RdBu_r', aspect='auto')

# Annotate cells
for i in range(n_vars):
    for j in range(n_vars):
        r = corr_mat[i, j]
        p = p_mat[i, j]
        if np.isnan(r):
            continue
        stars = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
        text_color = 'white' if abs(r) > 0.55 else 'black'
        ax.text(j, i, f'{r:+.2f}\n{stars}',
                ha='center', va='center', fontsize=7.5,
                color=text_color, fontweight='500')

# Axis tick labels colored by group
ax.set_xticks(range(n_vars))
ax.set_yticks(range(n_vars))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(labels, fontsize=9)
for tick, label in zip(ax.get_xticklabels(), labels):
    tick.set_color(group_colors.get(label, 'black'))
for tick, label in zip(ax.get_yticklabels(), labels):
    tick.set_color(group_colors.get(label, 'black'))

# White divider lines separating variable groups
n_behav = sum(1 for l in labels if l in ['Nav time (s)','SP clustering','TC clustering','Recall rate'])
n_enc   = sum(1 for l in labels if l.startswith('Enc'))
b1 = n_behav - 0.5
b2 = n_behav + n_enc - 0.5
for boundary in [b1, b2]:
    ax.axhline(boundary, color='white', linewidth=2.5)
    ax.axvline(boundary, color='white', linewidth=2.5)

# Group labels on top via twin axis
group_spans = [
    ('Behavioral',  0,               n_behav - 1),
    ('Encoding γ',  n_behav,         n_behav + n_enc - 1),
    ('Retrieval γ', n_behav + n_enc, n_vars - 1),
]
ax2 = ax.twiny()
ax2.set_xlim(ax.get_xlim())
ax2.set_xticks([(s + e) / 2 for _, s, e in group_spans])
ax2.set_xticklabels([g for g, _, _ in group_spans], fontsize=10, fontweight='500')
ax2.tick_params(length=0)

plt.colorbar(im, ax=ax, fraction=0.025, pad=0.01, label='Pearson r')
ax.set_title(
    f'Correlation matrix — behavioral & gamma-band iEEG  (n={len(subj)})\n'
    f'* p<.05   ** p<.01   *** p<.001  (uncorrected)',
    fontsize=11, fontweight='500', pad=28
)
plt.tight_layout()
plt.savefig('full_correlation_matrix.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print("Saved: full_correlation_matrix.png")

Subjects with all variables: 36
Columns: ['subject', 'mean_sp_clustering', 'mean_tc_clustering', 'mean_recall_rate', 'mean_nav_time_seconds', 'n_nav_events', 'encoding_frac_ssl_RHP', 'encoding_osc_ssl_RHP', 'retrieval_frac_ssl_RHP', 'retrieval_osc_ssl_RHP', 'encoding_frac_ssl_LHP', 'encoding_osc_ssl_LHP', 'retrieval_frac_ssl_LHP', 'retrieval_osc_ssl_LHP']

Pearson correlations (r / p):
                      Nav time (s) SP clustering TC clustering   Recall rate  Enc frac LHP  Enc frac RHP   Enc osc LHP   Enc osc RHP  Ret frac LHP  Ret frac RHP   Ret osc LHP   Ret osc RHP
Nav time (s)                     —+0.07        -0.30        -0.26        +0.05        -0.09        +0.05        -0.04        -0.10        -0.12        +0.10        +0.06        
SP clustering                    —             —-0.01        -0.08        +0.30        +0.28        +0.45*       +0.47*       +0.19        +0.24        +0.35        +0.44*       
TC clustering                    —             —             —-0.

In [16]:
"""
Subject-level analysis: average everything per subject first,
then run simple OLS (one data point per participant).

4 models:
  A: LHP gamma ~ SP_clustering + nav_speed + n_correct
  B: RHP gamma ~ SP_clustering + nav_speed + n_correct
  C: LHP gamma ~ TC_clustering + nav_speed + n_correct
  D: RHP gamma ~ TC_clustering + nav_speed + n_correct

FDR correction across all 12 predictor tests.
"""

import pandas as pd
import numpy as np
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. LOAD AND AVERAGE TO SUBJECT LEVEL
# ============================================================================

df = pd.read_csv('trial_level_summary.csv')
print(f"Trial-level shape: {df.shape}")
print(f"Subjects: {sorted(df['subject'].unique())}")

cols = [
    'encoding_osc_ssl_LHP', 'encoding_osc_ssl_RHP',
    'sp_clustering_LHP',    'sp_clustering_RHP',
    'tc_clustering_LHP',    'tc_clustering_RHP',
    'n_correct',            'nav_time_seconds',
]
cols = [c for c in cols if c in df.columns]

# Average all trials within each subject (one row per subject)
subj = df.groupby('subject')[cols].mean().reset_index()

# Navigation speed = 1 / nav_time
subj['nav_speed'] = 1.0 / subj['nav_time_seconds']

print(f"\nSubject-level shape: {subj.shape}")
print(f"\nDescriptives:")
print(subj[cols + ['nav_speed']].describe().round(3).to_string())

# ============================================================================
# 2. Z-SCORE EVERYTHING
# ============================================================================

to_z = [
    'encoding_osc_ssl_LHP', 'encoding_osc_ssl_RHP',
    'sp_clustering_LHP',    'sp_clustering_RHP',
    'tc_clustering_LHP',    'tc_clustering_RHP',
    'n_correct',            'nav_speed',
]
to_z = [c for c in to_z if c in subj.columns]

for col in to_z:
    subj[col + '_z'] = (subj[col] - subj[col].mean()) / subj[col].std()

# ============================================================================
# 3. OLS FUNCTION
# ============================================================================

def run_ols(outcome, predictors, label, data):
    sub = data[[outcome] + predictors].dropna()
    n   = len(sub)
    X   = add_constant(sub[predictors])
    y   = sub[outcome]
    fit = OLS(y, X).fit()

    print(f"\n{'='*60}")
    print(f"{label}  (N={n})")
    print(f"{'='*60}")
    print(fit.summary2())

    rows = []
    for pred in predictors:
        rows.append({
            'label':     label,
            'outcome':   outcome,
            'predictor': pred,
            'beta':      fit.params[pred],
            'SE':        fit.bse[pred],
            't':         fit.tvalues[pred],
            'p':         fit.pvalues[pred],
            'R2':        fit.rsquared,
            'R2_adj':    fit.rsquared_adj,
            'N':         n,
        })
    return pd.DataFrame(rows)

# ============================================================================
# 4. RUN MODELS
# ============================================================================

res_A = run_ols(
    outcome    = 'encoding_osc_ssl_LHP_z',
    predictors = ['sp_clustering_LHP_z', 'nav_speed_z', 'n_correct_z'],
    label      = 'A: LHP gamma ~ SP + nav_speed + n_correct',
    data       = subj,
)

res_B = run_ols(
    outcome    = 'encoding_osc_ssl_RHP_z',
    predictors = ['sp_clustering_RHP_z', 'nav_speed_z', 'n_correct_z'],
    label      = 'B: RHP gamma ~ SP + nav_speed + n_correct',
    data       = subj,
)

res_C = run_ols(
    outcome    = 'encoding_osc_ssl_LHP_z',
    predictors = ['tc_clustering_LHP_z', 'nav_speed_z', 'n_correct_z'],
    label      = 'C: LHP gamma ~ TC + nav_speed + n_correct',
    data       = subj,
)

res_D = run_ols(
    outcome    = 'encoding_osc_ssl_RHP_z',
    predictors = ['tc_clustering_RHP_z', 'nav_speed_z', 'n_correct_z'],
    label      = 'D: RHP gamma ~ TC + nav_speed + n_correct',
    data       = subj,
)

# ============================================================================
# 5. COMBINED TABLE + FDR
# ============================================================================

results = pd.concat([res_A, res_B, res_C, res_D], ignore_index=True)

# FDR across all 12 tests (3 predictors x 4 models)
_, results['p_fdr'], _, _ = multipletests(results['p'].values, method='fdr_bh')
results['sig'] = results['p_fdr'].apply(
    lambda p: '***' if p < .001 else ('**' if p < .01 else ('*' if p < .05 else ''))
)

print("\n\n" + "="*70)
print("ALL RESULTS — BH-FDR corrected across 12 tests")
print("="*70)
print(results[['label', 'predictor', 'beta', 'SE', 't', 'p', 'p_fdr', 'sig', 'N', 'R2_adj']
              ].to_string(index=False))

# ============================================================================
# 6. SAVE
# ============================================================================

results.to_csv('gamma_osc_subject_level_results.csv', index=False)
subj.to_csv('subject_level_averages.csv', index=False)
print("\nSaved: gamma_osc_subject_level_results.csv")
print("Saved: subject_level_averages.csv")

Trial-level shape: (217, 22)
Subjects: ['R1494D', 'R1501J', 'R1502D', 'R1503E', 'R1504E', 'R1509E', 'R1520T', 'R1521E', 'R1523E', 'R1529T', 'R1530J', 'R1532T', 'R1536J', 'R1537T', 'R1538E', 'R1539E', 'R1542J', 'R1543E', 'R1546E', 'R1551T', 'R1554T', 'R1560T', 'R1561E', 'R1564J', 'R1567T', 'R1569T', 'R1570T', 'R1571T', 'R1572T', 'R1573T', 'R1575E', 'R1620J', 'R1637T', 'R1642J', 'R1651J', 'R1653J']

Subject-level shape: (36, 10)

Descriptives:
       encoding_osc_ssl_LHP  encoding_osc_ssl_RHP  sp_clustering_LHP  sp_clustering_RHP  tc_clustering_LHP  tc_clustering_RHP  n_correct  nav_time_seconds  nav_speed
count                27.000                27.000             27.000             27.000             27.000             27.000     36.000            36.000     36.000
mean                  0.322                 0.289              0.514              0.529              0.595              0.604      5.134            31.509      0.034
std                   0.299                 0.330       

In [19]:
"""
Visualize subject-level OLS results:
  - Beta coefficients with error bars (±1 SE) for all 4 models
  - Scatter plots: gamma osc vs each significant predictor
  - One figure saved as gamma_osc_results_figure.png
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

# ============================================================================
# 1. LOAD DATA
# ============================================================================

df       = pd.read_csv('trial_level_summary.csv')
results  = pd.read_csv('gamma_osc_subject_level_results.csv')

# ── Rebuild subject-level averages ──────────────────────────────────────────
cols = [
    'encoding_osc_ssl_LHP', 'encoding_osc_ssl_RHP',
    'sp_clustering_LHP',    'sp_clustering_RHP',
    'tc_clustering_LHP',    'tc_clustering_RHP',
    'n_correct',            'nav_time_seconds',
]
cols  = [c for c in cols if c in df.columns]
subj  = df.groupby('subject')[cols].mean().reset_index()
subj['nav_speed'] = 1.0 / subj['nav_time_seconds']

for col in [c for c in cols if c != 'nav_time_seconds'] + ['nav_speed']:
    subj[col + '_z'] = (subj[col] - subj[col].mean()) / subj[col].std()

# ============================================================================
# 2. PLOTTING SETUP
# ============================================================================

BLUE   = '#185FA5'
GREEN  = '#0F6E56'
GRAY   = '#888780'
RED    = '#D85A30'

plt.rcParams.update({
    'font.family':     'sans-serif',
    'font.size':       11,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.linewidth':  0.8,
    'xtick.major.size': 3,
    'ytick.major.size': 3,
})

fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

# ============================================================================
# 3. PANEL A & B — BETA COEFFICIENT PLOTS (SP and TC models)
# ============================================================================

predictor_labels = {
    'sp_clustering_LHP_z': 'SP clust',
    'sp_clustering_RHP_z': 'SP clust',
    'tc_clustering_LHP_z': 'TC clust',
    'tc_clustering_RHP_z': 'TC clust',
    'nav_speed_z':         'Nav speed',
    'n_correct_z':         'N correct',
}

PRED_ORDER = {
    'SP': ['sp_clustering_LHP_z', 'sp_clustering_RHP_z', 'nav_speed_z', 'n_correct_z'],
    'TC': ['tc_clustering_LHP_z', 'tc_clustering_RHP_z', 'nav_speed_z', 'n_correct_z'],
}
TICK_LABELS = {
    'SP': ['SP clust', 'Nav speed', 'N correct'],
    'TC': ['TC clust', 'Nav speed', 'N correct'],
}

def get_model_stats(label_exact, clust_pred):
    """Extract beta/SE/p for [clust_pred, nav_speed_z, n_correct_z] from results df."""
    sub = results[results['label'] == label_exact]
    rows = []
    for pred in [clust_pred, 'nav_speed_z', 'n_correct_z']:
        row = sub[sub['predictor'] == pred]
        if len(row) == 0:
            rows.append({'beta': 0.0, 'SE': 0.0, 'p_fdr': 1.0})
        else:
            rows.append(row.iloc[0][['beta', 'SE', 'p_fdr']].to_dict())
    return rows

def plot_betas(ax, model_specs, title):
    """
    model_specs: list of dicts with keys:
        label     — exact label string in results CSV
        clust_pred — the clustering predictor column name
        hem_label  — display name
        color
    """
    n_pred  = 3
    x       = np.arange(n_pred)
    width   = 0.35
    offsets = [-width/2, width/2]

    for idx, spec in enumerate(model_specs):
        stats_rows = get_model_stats(spec['label'], spec['clust_pred'])
        betas = np.array([r['beta']  for r in stats_rows])
        ses   = np.array([r['SE']    for r in stats_rows])
        ps    = np.array([r['p_fdr'] for r in stats_rows])

        ax.bar(
            x + offsets[idx], betas, width,
            color=spec['color'], alpha=0.85, label=spec['hem_label'],
            zorder=3, linewidth=0
        )
        ax.errorbar(
            x + offsets[idx], betas,
            yerr=ses, fmt='none',
            color='#444', linewidth=1.2, capsize=3, zorder=4
        )

        for i, (b, se, p) in enumerate(zip(betas, ses, ps)):
            if p < 0.05:
                marker = '***' if p < .001 else ('**' if p < .01 else '*')
                ypos = b + se + 0.05 if b >= 0 else b - se - 0.12
                ax.text(x[i] + offsets[idx], ypos, marker,
                        ha='center', va='bottom', fontsize=10,
                        color=RED, fontweight='bold')
            elif p < 0.1:
                ypos = b + se + 0.05 if b >= 0 else b - se - 0.12
                ax.text(x[i] + offsets[idx], ypos, '†',
                        ha='center', va='bottom', fontsize=11, color=GRAY)

    ax.axhline(0, color='#aaa', linewidth=0.8, zorder=2)
    ax.set_xticks(x)
    # tick labels derived from first spec's clust_pred name
    clust_name = predictor_labels.get(model_specs[0]['clust_pred'], 'Clustering')
    ax.set_xticklabels([clust_name, 'Nav speed', 'N correct'], fontsize=10)
    ax.set_ylabel('Standardized β', fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='normal', pad=8)
    ax.legend(fontsize=9, frameon=False)
    ax.set_ylim(
        min(results['beta'].min() - results['SE'].max() - 0.25, -0.3),
        max(results['beta'].max() + results['SE'].max() + 0.25,  0.7)
    )
    ax.yaxis.grid(True, linewidth=0.4, color='#ddd', zorder=0)
    ax.set_axisbelow(True)

ax_spbeta = fig.add_subplot(gs[0, 0])
plot_betas(
    ax_spbeta,
    model_specs=[
        {'label': 'A: LHP gamma ~ SP + nav_speed + n_correct',
         'clust_pred': 'sp_clustering_LHP_z', 'hem_label': 'LHP', 'color': BLUE},
        {'label': 'B: RHP gamma ~ SP + nav_speed + n_correct',
         'clust_pred': 'sp_clustering_RHP_z', 'hem_label': 'RHP', 'color': GREEN},
    ],
    title='A   SP clustering models — beta coefficients',
)

ax_tcbeta = fig.add_subplot(gs[0, 1])
plot_betas(
    ax_tcbeta,
    model_specs=[
        {'label': 'C: LHP gamma ~ TC + nav_speed + n_correct',
         'clust_pred': 'tc_clustering_LHP_z', 'hem_label': 'LHP', 'color': BLUE},
        {'label': 'D: RHP gamma ~ TC + nav_speed + n_correct',
         'clust_pred': 'tc_clustering_RHP_z', 'hem_label': 'RHP', 'color': GREEN},
    ],
    title='B   TC clustering models — beta coefficients',
)

# ============================================================================
# 4. PANEL C — SCATTER: SP clustering RHP vs RHP gamma (the key finding)
# ============================================================================

ax_sc1 = fig.add_subplot(gs[1, 0])

x_col = 'sp_clustering_RHP_z'
y_col = 'encoding_osc_ssl_RHP_z'
sub   = subj[[x_col, y_col]].dropna()
x_    = sub[x_col].values
y_    = sub[y_col].values

slope, intercept, r, p, _ = stats.linregress(x_, y_)
xline = np.linspace(x_.min(), x_.max(), 100)

ax_sc1.scatter(x_, y_, color=GREEN, alpha=0.75, s=45, zorder=3,
               edgecolors='white', linewidths=0.5)
ax_sc1.plot(xline, slope * xline + intercept,
            color=GREEN, linewidth=1.8, zorder=4)

# shaded CI band
n    = len(x_)
se_  = np.std(y_ - (slope * x_ + intercept), ddof=2)
ci   = se_ * stats.t.ppf(0.975, df=n-2) * np.sqrt(
           1/n + (xline - x_.mean())**2 / np.sum((x_ - x_.mean())**2))
ax_sc1.fill_between(xline,
                     slope * xline + intercept - ci,
                     slope * xline + intercept + ci,
                     color=GREEN, alpha=0.12, zorder=2)

p_str = f'p = {p:.3f}' if p >= 0.001 else 'p < 0.001'
ax_sc1.text(0.05, 0.92,
            f'r = {r:.2f},  {p_str}',
            transform=ax_sc1.transAxes,
            fontsize=9, color=GREEN, va='top')
ax_sc1.set_xlabel('SP clustering RHP (z)', fontsize=10)
ax_sc1.set_ylabel('Gamma osc RHP (z)', fontsize=10)
ax_sc1.set_title('C   SP clustering vs RHP gamma', fontsize=11, fontweight='normal', pad=8)

# ============================================================================
# 5. PANEL D — SCATTER: SP clustering LHP vs LHP gamma (trending)
# ============================================================================

ax_sc2 = fig.add_subplot(gs[1, 1])

x_col2 = 'sp_clustering_LHP_z'
y_col2 = 'encoding_osc_ssl_LHP_z'
sub2   = subj[[x_col2, y_col2]].dropna()
x2_    = sub2[x_col2].values
y2_    = sub2[y_col2].values

slope2, intercept2, r2, p2, _ = stats.linregress(x2_, y2_)
xline2 = np.linspace(x2_.min(), x2_.max(), 100)

ax_sc2.scatter(x2_, y2_, color=BLUE, alpha=0.75, s=45, zorder=3,
               edgecolors='white', linewidths=0.5)
ax_sc2.plot(xline2, slope2 * xline2 + intercept2,
            color=BLUE, linewidth=1.8, zorder=4, linestyle='--')

se2_ = np.std(y2_ - (slope2 * x2_ + intercept2), ddof=2)
ci2  = se2_ * stats.t.ppf(0.975, df=len(x2_)-2) * np.sqrt(
            1/len(x2_) + (xline2 - x2_.mean())**2 / np.sum((x2_ - x2_.mean())**2))
ax_sc2.fill_between(xline2,
                     slope2 * xline2 + intercept2 - ci2,
                     slope2 * xline2 + intercept2 + ci2,
                     color=BLUE, alpha=0.10, zorder=2)

p_str2 = f'p = {p2:.3f}' if p2 >= 0.001 else 'p < 0.001'
ax_sc2.text(0.05, 0.92,
            f'r = {r2:.2f},  {p_str2}',
            transform=ax_sc2.transAxes,
            fontsize=9, color=BLUE, va='top')
ax_sc2.set_xlabel('SP clustering LHP (z)', fontsize=10)
ax_sc2.set_ylabel('Gamma osc LHP (z)', fontsize=10)
ax_sc2.set_title('D   SP clustering vs LHP gamma (trend)', fontsize=11, fontweight='normal', pad=8)

# ============================================================================
# 6. FIGURE ANNOTATIONS & SAVE
# ============================================================================

fig.text(
    0.5, 0.01,
    '* p < .05   ** p < .01   *** p < .001 (FDR-corrected)   † p < .10 trend   '
    'Error bars = ±1 SE   Dashed line = non-significant trend',
    ha='center', fontsize=8, color='#888'
)
plt.show()
plt.savefig('gamma_osc_results_figure.png', dpi=150, bbox_inches='tight',
            facecolor='white')
print("Saved: gamma_osc_results_figure.png")

Saved: gamma_osc_results_figure.png


In [20]:
"""
Subject-level OLS: gamma oscillatory power ~ clustering + n_correct
(navigation speed excluded)

4 models:
  A: LHP gamma ~ SP_clustering + n_correct
  B: RHP gamma ~ SP_clustering + n_correct
  C: LHP gamma ~ TC_clustering + n_correct
  D: RHP gamma ~ TC_clustering + n_correct

FDR correction across all 8 predictor tests (2 predictors x 4 models).
"""

import pandas as pd
import numpy as np
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. LOAD AND AVERAGE TO SUBJECT LEVEL
# ============================================================================

df = pd.read_csv('trial_level_summary.csv')

cols = [
    'encoding_osc_ssl_LHP', 'encoding_osc_ssl_RHP',
    'sp_clustering_LHP',    'sp_clustering_RHP',
    'tc_clustering_LHP',    'tc_clustering_RHP',
    'n_correct',
]
cols = [c for c in cols if c in df.columns]

subj = df.groupby('subject')[cols].mean().reset_index()

print(f"Subject-level shape: {subj.shape}")
print(f"\nDescriptives:")
print(subj[cols].describe().round(3).to_string())

# ============================================================================
# 2. Z-SCORE
# ============================================================================

for col in cols:
    subj[col + '_z'] = (subj[col] - subj[col].mean()) / subj[col].std()

# ============================================================================
# 3. OLS FUNCTION
# ============================================================================

def run_ols(outcome, predictors, label, data):
    sub = data[[outcome] + predictors].dropna()
    n   = len(sub)
    X   = add_constant(sub[predictors])
    y   = sub[outcome]
    fit = OLS(y, X).fit()

    print(f"\n{'='*60}")
    print(f"{label}  (N={n})")
    print(f"{'='*60}")
    print(fit.summary2())

    rows = []
    for pred in predictors:
        rows.append({
            'label':   label,
            'outcome': outcome,
            'predictor': pred,
            'beta':    fit.params[pred],
            'SE':      fit.bse[pred],
            't':       fit.tvalues[pred],
            'p':       fit.pvalues[pred],
            'R2':      fit.rsquared,
            'R2_adj':  fit.rsquared_adj,
            'N':       n,
        })
    return pd.DataFrame(rows)

# ============================================================================
# 4. RUN MODELS
# ============================================================================

res_A = run_ols(
    outcome    = 'encoding_osc_ssl_LHP_z',
    predictors = ['sp_clustering_LHP_z', 'n_correct_z'],
    label      = 'A: LHP gamma ~ SP + n_correct',
    data       = subj,
)

res_B = run_ols(
    outcome    = 'encoding_osc_ssl_RHP_z',
    predictors = ['sp_clustering_RHP_z', 'n_correct_z'],
    label      = 'B: RHP gamma ~ SP + n_correct',
    data       = subj,
)

res_C = run_ols(
    outcome    = 'encoding_osc_ssl_LHP_z',
    predictors = ['tc_clustering_LHP_z', 'n_correct_z'],
    label      = 'C: LHP gamma ~ TC + n_correct',
    data       = subj,
)

res_D = run_ols(
    outcome    = 'encoding_osc_ssl_RHP_z',
    predictors = ['tc_clustering_RHP_z', 'n_correct_z'],
    label      = 'D: RHP gamma ~ TC + n_correct',
    data       = subj,
)

# ============================================================================
# 5. COMBINED TABLE + FDR
# ============================================================================

results = pd.concat([res_A, res_B, res_C, res_D], ignore_index=True)

# FDR across all 8 tests (2 predictors x 4 models)
_, results['p_fdr'], _, _ = multipletests(results['p'].values, method='fdr_bh')
results['sig'] = results['p_fdr'].apply(
    lambda p: '***' if p < .001 else ('**' if p < .01 else ('*' if p < .05 else ''))
)

print("\n\n" + "="*70)
print("ALL RESULTS — BH-FDR corrected across 8 tests")
print("="*70)
print(results[['label', 'predictor', 'beta', 'SE', 't', 'p', 'p_fdr', 'sig', 'N', 'R2_adj']
              ].to_string(index=False))

# ============================================================================
# 6. SAVE
# ============================================================================

results.to_csv('gamma_osc_no_navspeed_results.csv', index=False)
print("\nSaved: gamma_osc_no_navspeed_results.csv")

Subject-level shape: (36, 8)

Descriptives:
       encoding_osc_ssl_LHP  encoding_osc_ssl_RHP  sp_clustering_LHP  sp_clustering_RHP  tc_clustering_LHP  tc_clustering_RHP  n_correct
count                27.000                27.000             27.000             27.000             27.000             27.000     36.000
mean                  0.322                 0.289              0.514              0.529              0.595              0.604      5.134
std                   0.299                 0.330              0.139              0.116              0.109              0.103      1.775
min                   0.018                -0.541              0.186              0.261              0.356              0.389      3.000
25%                   0.088                 0.056              0.461              0.461              0.519              0.519      3.938
50%                   0.207                 0.224              0.511              0.511              0.597              0.601      4.6

In [21]:
"""
Subject-level OLS: gamma oscillatory power ~ clustering only
(navigation speed and recall number both excluded)

4 models:
  A: LHP gamma ~ SP_clustering
  B: RHP gamma ~ SP_clustering
  C: LHP gamma ~ TC_clustering
  D: RHP gamma ~ TC_clustering

FDR correction across all 4 predictor tests.
"""

import pandas as pd
import numpy as np
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. LOAD AND AVERAGE TO SUBJECT LEVEL
# ============================================================================

df = pd.read_csv('trial_level_summary.csv')

cols = [
    'encoding_osc_ssl_LHP', 'encoding_osc_ssl_RHP',
    'sp_clustering_LHP',    'sp_clustering_RHP',
    'tc_clustering_LHP',    'tc_clustering_RHP',
]
cols = [c for c in cols if c in df.columns]

subj = df.groupby('subject')[cols].mean().reset_index()

print(f"Subject-level shape: {subj.shape}")
print(f"\nDescriptives:")
print(subj[cols].describe().round(3).to_string())

# ============================================================================
# 2. Z-SCORE
# ============================================================================

for col in cols:
    subj[col + '_z'] = (subj[col] - subj[col].mean()) / subj[col].std()

# ============================================================================
# 3. OLS FUNCTION
# ============================================================================

def run_ols(outcome, predictors, label, data):
    sub = data[[outcome] + predictors].dropna()
    n   = len(sub)
    X   = add_constant(sub[predictors])
    y   = sub[outcome]
    fit = OLS(y, X).fit()

    print(f"\n{'='*60}")
    print(f"{label}  (N={n})")
    print(f"{'='*60}")
    print(fit.summary2())

    rows = []
    for pred in predictors:
        rows.append({
            'label':     label,
            'outcome':   outcome,
            'predictor': pred,
            'beta':      fit.params[pred],
            'SE':        fit.bse[pred],
            't':         fit.tvalues[pred],
            'p':         fit.pvalues[pred],
            'R2':        fit.rsquared,
            'R2_adj':    fit.rsquared_adj,
            'N':         n,
        })
    return pd.DataFrame(rows)

# ============================================================================
# 4. RUN MODELS
# ============================================================================

res_A = run_ols(
    outcome    = 'encoding_osc_ssl_LHP_z',
    predictors = ['sp_clustering_LHP_z'],
    label      = 'A: LHP gamma ~ SP',
    data       = subj,
)

res_B = run_ols(
    outcome    = 'encoding_osc_ssl_RHP_z',
    predictors = ['sp_clustering_RHP_z'],
    label      = 'B: RHP gamma ~ SP',
    data       = subj,
)

res_C = run_ols(
    outcome    = 'encoding_osc_ssl_LHP_z',
    predictors = ['tc_clustering_LHP_z'],
    label      = 'C: LHP gamma ~ TC',
    data       = subj,
)

res_D = run_ols(
    outcome    = 'encoding_osc_ssl_RHP_z',
    predictors = ['tc_clustering_RHP_z'],
    label      = 'D: RHP gamma ~ TC',
    data       = subj,
)

# ============================================================================
# 5. COMBINED TABLE + FDR
# ============================================================================

results = pd.concat([res_A, res_B, res_C, res_D], ignore_index=True)

# FDR across all 4 tests (1 predictor x 4 models)
_, results['p_fdr'], _, _ = multipletests(results['p'].values, method='fdr_bh')
results['sig'] = results['p_fdr'].apply(
    lambda p: '***' if p < .001 else ('**' if p < .01 else ('*' if p < .05 else ''))
)

print("\n\n" + "="*70)
print("ALL RESULTS — BH-FDR corrected across 4 tests")
print("="*70)
print(results[['label', 'predictor', 'beta', 'SE', 't', 'p', 'p_fdr', 'sig', 'N', 'R2_adj']
              ].to_string(index=False))

# ============================================================================
# 6. SAVE
# ============================================================================

results.to_csv('gamma_osc_clustering_only_results.csv', index=False)
print("\nSaved: gamma_osc_clustering_only_results.csv")

Subject-level shape: (36, 7)

Descriptives:
       encoding_osc_ssl_LHP  encoding_osc_ssl_RHP  sp_clustering_LHP  sp_clustering_RHP  tc_clustering_LHP  tc_clustering_RHP
count                27.000                27.000             27.000             27.000             27.000             27.000
mean                  0.322                 0.289              0.514              0.529              0.595              0.604
std                   0.299                 0.330              0.139              0.116              0.109              0.103
min                   0.018                -0.541              0.186              0.261              0.356              0.389
25%                   0.088                 0.056              0.461              0.461              0.519              0.519
50%                   0.207                 0.224              0.511              0.511              0.597              0.601
75%                   0.528                 0.531              0.587      

In [22]:
"""
Subject-level OLS: gamma oscillatory power ~ clustering only
No z-scoring — raw values used throughout.

4 models:
  A: LHP gamma ~ SP_clustering
  B: RHP gamma ~ SP_clustering
  C: LHP gamma ~ TC_clustering
  D: RHP gamma ~ TC_clustering

FDR correction across all 4 tests.
"""

import pandas as pd
import numpy as np
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. LOAD AND AVERAGE TO SUBJECT LEVEL
# ============================================================================

df = pd.read_csv('trial_level_summary.csv')

cols = [
    'encoding_osc_ssl_LHP', 'encoding_osc_ssl_RHP',
    'sp_clustering_LHP',    'sp_clustering_RHP',
    'tc_clustering_LHP',    'tc_clustering_RHP',
]
cols = [c for c in cols if c in df.columns]

subj = df.groupby('subject')[cols].mean().reset_index()

print(f"Subject-level shape: {subj.shape}")
print(f"\nDescriptives:")
print(subj[cols].describe().round(3).to_string())

# ============================================================================
# 2. OLS FUNCTION
# ============================================================================

def run_ols(outcome, predictors, label, data):
    sub = data[[outcome] + predictors].dropna()
    n   = len(sub)
    X   = add_constant(sub[predictors])
    y   = sub[outcome]
    fit = OLS(y, X).fit()

    print(f"\n{'='*60}")
    print(f"{label}  (N={n})")
    print(f"{'='*60}")
    print(fit.summary2())

    rows = []
    for pred in predictors:
        rows.append({
            'label':     label,
            'outcome':   outcome,
            'predictor': pred,
            'beta':      fit.params[pred],
            'SE':        fit.bse[pred],
            't':         fit.tvalues[pred],
            'p':         fit.pvalues[pred],
            'R2':        fit.rsquared,
            'R2_adj':    fit.rsquared_adj,
            'N':         n,
        })
    return pd.DataFrame(rows)

# ============================================================================
# 3. RUN MODELS
# ============================================================================

res_A = run_ols(
    outcome    = 'encoding_osc_ssl_LHP',
    predictors = ['sp_clustering_LHP'],
    label      = 'A: LHP gamma ~ SP',
    data       = subj,
)

res_B = run_ols(
    outcome    = 'encoding_osc_ssl_RHP',
    predictors = ['sp_clustering_RHP'],
    label      = 'B: RHP gamma ~ SP',
    data       = subj,
)

res_C = run_ols(
    outcome    = 'encoding_osc_ssl_LHP',
    predictors = ['tc_clustering_LHP'],
    label      = 'C: LHP gamma ~ TC',
    data       = subj,
)

res_D = run_ols(
    outcome    = 'encoding_osc_ssl_RHP',
    predictors = ['tc_clustering_RHP'],
    label      = 'D: RHP gamma ~ TC',
    data       = subj,
)

# ============================================================================
# 4. COMBINED TABLE + FDR
# ============================================================================

results = pd.concat([res_A, res_B, res_C, res_D], ignore_index=True)

_, results['p_fdr'], _, _ = multipletests(results['p'].values, method='fdr_bh')
results['sig'] = results['p_fdr'].apply(
    lambda p: '***' if p < .001 else ('**' if p < .01 else ('*' if p < .05 else ''))
)

print("\n\n" + "="*70)
print("ALL RESULTS — BH-FDR corrected across 4 tests")
print("="*70)
print(results[['label', 'predictor', 'beta', 'SE', 't', 'p', 'p_fdr', 'sig', 'N', 'R2_adj']
              ].to_string(index=False))

# ============================================================================
# 5. SAVE
# ============================================================================

results.to_csv('gamma_osc_clustering_only_raw_results.csv', index=False)
print("\nSaved: gamma_osc_clustering_only_raw_results.csv")

Subject-level shape: (36, 7)

Descriptives:
       encoding_osc_ssl_LHP  encoding_osc_ssl_RHP  sp_clustering_LHP  sp_clustering_RHP  tc_clustering_LHP  tc_clustering_RHP
count                27.000                27.000             27.000             27.000             27.000             27.000
mean                  0.322                 0.289              0.514              0.529              0.595              0.604
std                   0.299                 0.330              0.139              0.116              0.109              0.103
min                   0.018                -0.541              0.186              0.261              0.356              0.389
25%                   0.088                 0.056              0.461              0.461              0.519              0.519
50%                   0.207                 0.224              0.511              0.511              0.597              0.601
75%                   0.528                 0.531              0.587      

In [24]:
"""
Subject-level OLS: retrieval gamma oscillatory power ~ clustering only
No z-scoring — raw values used throughout.

4 models:
  A: LHP retrieval gamma ~ SP_clustering
  B: RHP retrieval gamma ~ SP_clustering
  C: LHP retrieval gamma ~ TC_clustering
  D: RHP retrieval gamma ~ TC_clustering

FDR correction across all 4 tests.
"""

import pandas as pd
import numpy as np
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. LOAD AND AVERAGE TO SUBJECT LEVEL
# ============================================================================

df = pd.read_csv('trial_level_summary.csv')

cols = [
    'retrieval_osc_ssl_LHP', 'retrieval_osc_ssl_RHP',
    'sp_clustering_LHP',     'sp_clustering_RHP',
    'tc_clustering_LHP',     'tc_clustering_RHP',
]
cols = [c for c in cols if c in df.columns]

subj = df.groupby('subject')[cols].mean().reset_index()

print(f"Subject-level shape: {subj.shape}")
print(f"\nDescriptives:")
print(subj[cols].describe().round(3).to_string())

# ============================================================================
# 2. OLS FUNCTION
# ============================================================================

def run_ols(outcome, predictors, label, data):
    sub = data[[outcome] + predictors].dropna()
    n   = len(sub)
    X   = add_constant(sub[predictors])
    y   = sub[outcome]
    fit = OLS(y, X).fit()

    print(f"\n{'='*60}")
    print(f"{label}  (N={n})")
    print(f"{'='*60}")
    print(fit.summary2())

    rows = []
    for pred in predictors:
        rows.append({
            'label':     label,
            'outcome':   outcome,
            'predictor': pred,
            'beta':      fit.params[pred],
            'SE':        fit.bse[pred],
            't':         fit.tvalues[pred],
            'p':         fit.pvalues[pred],
            'R2':        fit.rsquared,
            'R2_adj':    fit.rsquared_adj,
            'N':         n,
        })
    return pd.DataFrame(rows)

# ============================================================================
# 3. RUN MODELS
# ============================================================================

res_A = run_ols(
    outcome    = 'retrieval_osc_ssl_LHP',
    predictors = ['sp_clustering_LHP'],
    label      = 'A: LHP retrieval gamma ~ SP',
    data       = subj,
)

res_B = run_ols(
    outcome    = 'retrieval_osc_ssl_RHP',
    predictors = ['sp_clustering_RHP'],
    label      = 'B: RHP retrieval gamma ~ SP',
    data       = subj,
)

res_C = run_ols(
    outcome    = 'retrieval_osc_ssl_LHP',
    predictors = ['tc_clustering_LHP'],
    label      = 'C: LHP retrieval gamma ~ TC',
    data       = subj,
)

res_D = run_ols(
    outcome    = 'retrieval_osc_ssl_RHP',
    predictors = ['tc_clustering_RHP'],
    label      = 'D: RHP retrieval gamma ~ TC',
    data       = subj,
)

# ============================================================================
# 4. COMBINED TABLE + FDR
# ============================================================================

results = pd.concat([res_A, res_B, res_C, res_D], ignore_index=True)

_, results['p_fdr'], _, _ = multipletests(results['p'].values, method='fdr_bh')
results['sig'] = results['p_fdr'].apply(
    lambda p: '***' if p < .001 else ('**' if p < .01 else ('*' if p < .05 else ''))
)

print("\n\n" + "="*70)
print("ALL RESULTS — BH-FDR corrected across 4 tests")
print("="*70)
print(results[['label', 'predictor', 'beta', 'SE', 't', 'p', 'p_fdr', 'sig', 'N', 'R2_adj']
              ].to_string(index=False))

# ============================================================================
# 5. SAVE
# ============================================================================

results.to_csv('gamma_retrieval_osc_clustering_only_raw_results.csv', index=False)
print("\nSaved: gamma_retrieval_osc_clustering_only_raw_results.csv")

Subject-level shape: (36, 7)

Descriptives:
       retrieval_osc_ssl_LHP  retrieval_osc_ssl_RHP  sp_clustering_LHP  sp_clustering_RHP  tc_clustering_LHP  tc_clustering_RHP
count                 27.000                 27.000             27.000             27.000             27.000             27.000
mean                   0.331                  0.316              0.514              0.529              0.595              0.604
std                    0.295                  0.285              0.139              0.116              0.109              0.103
min                    0.011                 -0.296              0.186              0.261              0.356              0.389
25%                    0.076                  0.157              0.461              0.461              0.519              0.519
50%                    0.248                  0.241              0.511              0.511              0.597              0.601
75%                    0.513                  0.515         

In [25]:
"""
Subject-level OLS replicating the between-subject analysis from the
ALL_SUBJECTS_irasa_clean.csv script, but using trial_level_summary.csv.

Direction: clustering ~ retrieval_gamma  (flipped from previous scripts)

Single omnibus model per gamma component (osc / frac):
  clustering_mean ~ gamma_mean * clustering_type * region

SP and TC stacked into long format.
clustering_type (SP vs TC) and region (LHP vs RHP) as categorical factors.
Separate bivariate correlations per region × clustering type.
"""

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. LOAD AND AVERAGE TO SUBJECT LEVEL
# ============================================================================

df = pd.read_csv('trial_level_summary.csv')

cols = [
    'retrieval_osc_ssl_LHP',  'retrieval_osc_ssl_RHP',
    'retrieval_frac_ssl_LHP', 'retrieval_frac_ssl_RHP',
    'sp_clustering_LHP',      'sp_clustering_RHP',
    'tc_clustering_LHP',      'tc_clustering_RHP',
]
cols = [c for c in cols if c in df.columns]

# One row per subject per region — mirror the other script's structure
# Average across trials within subject
subj = df.groupby('subject')[cols].mean().reset_index()

print(f"Subject-level shape: {subj.shape}")
print(f"Subjects: {subj['subject'].nunique()}")
print(f"\nDescriptives:")
print(subj[cols].describe().round(3).to_string())

# ============================================================================
# 2. RESHAPE TO LONG FORMAT  (subject × region × clustering_type)
# Mirrors: stacking SP & TC with clustering_type factor
# ============================================================================

rows = []
for _, row in subj.iterrows():
    for region in ['LHP', 'RHP']:
        osc_col  = f'retrieval_osc_ssl_{region}'
        frac_col = f'retrieval_frac_ssl_{region}'
        sp_col   = f'sp_clustering_{region}'
        tc_col   = f'tc_clustering_{region}'

        if osc_col not in subj.columns:
            continue

        # SP row
        rows.append({
            'subject':          row['subject'],
            'region':           region,
            'clustering_type':  'spatial',
            'clustering_mean':  row[sp_col]   if sp_col  in subj.columns else np.nan,
            'gamma_osc_mean':   row[osc_col]  if osc_col in subj.columns else np.nan,
            'gamma_frac_mean':  row[frac_col] if frac_col in subj.columns else np.nan,
        })
        # TC row
        rows.append({
            'subject':          row['subject'],
            'region':           region,
            'clustering_type':  'temporal',
            'clustering_mean':  row[tc_col]   if tc_col  in subj.columns else np.nan,
            'gamma_osc_mean':   row[osc_col]  if osc_col in subj.columns else np.nan,
            'gamma_frac_mean':  row[frac_col] if frac_col in subj.columns else np.nan,
        })

long_df = pd.DataFrame(rows).dropna(subset=['clustering_mean', 'gamma_osc_mean'])

# Set reference levels: RHP and spatial (matches the other script)
long_df['region']          = pd.Categorical(long_df['region'],
                                            categories=['RHP', 'LHP'])
long_df['clustering_type'] = pd.Categorical(long_df['clustering_type'],
                                            categories=['spatial', 'temporal'])

print(f"\nLong-format shape: {long_df.shape}")
print(f"  spatial : {(long_df['clustering_type']=='spatial').sum()}")
print(f"  temporal: {(long_df['clustering_type']=='temporal').sum()}")

# ============================================================================
# 3. OMNIBUS OLS MODELS
# clustering_mean ~ gamma * clustering_type * region
# ============================================================================

def fit_omnibus(data, gamma_col, label):
    formula = f'clustering_mean ~ {gamma_col} * clustering_type * region'
    sub     = data.dropna(subset=['clustering_mean', gamma_col])
    n       = len(sub)

    fit = smf.ols(formula=formula, data=sub).fit(cov_type='HC3')

    print(f"\n{'='*70}")
    print(f"MODEL: {label}  (N={n} obs, {sub['subject'].nunique()} subjects)")
    print(f"Formula: {formula}")
    print(f"{'='*70}")
    print(fit.summary2())

    rows = []
    for pname in fit.params.index:
        if pname == 'Intercept':
            continue
        rows.append({
            'label':     label,
            'gamma_col': gamma_col,
            'parameter': pname,
            'beta':      fit.params[pname],
            'SE':        fit.bse[pname],
            't':         fit.tvalues[pname],
            'p':         fit.pvalues[pname],
            'R2':        fit.rsquared,
            'R2_adj':    fit.rsquared_adj,
            'N':         n,
        })
    return pd.DataFrame(rows)

res_osc  = fit_omnibus(long_df, 'gamma_osc_mean',  'OSC  — clustering ~ retrieval gamma osc  * type * region')
res_frac = fit_omnibus(long_df, 'gamma_frac_mean', 'FRAC — clustering ~ retrieval gamma frac * type * region')

# ============================================================================
# 4. BIVARIATE CORRELATIONS  (per region × clustering type)
# Mirrors the correlation table in the other script
# ============================================================================

print("\n" + "="*70)
print("BIVARIATE CORRELATIONS (subject-level means)")
print("="*70)
print(f"\n  {'Gamma':<8} {'Region':<6} {'Clust':<10} {'r':>7} {'p':>8} {'n':>4}  sig")
print("  " + "-"*48)

corr_rows = []
for gamma_col, comp in [('gamma_osc_mean', 'osc'), ('gamma_frac_mean', 'frac')]:
    if gamma_col not in long_df.columns:
        continue
    for region in ['RHP', 'LHP']:
        for ctype in ['spatial', 'temporal']:
            sub = long_df[
                (long_df['region'] == region) &
                (long_df['clustering_type'] == ctype)
            ].dropna(subset=[gamma_col, 'clustering_mean'])

            if len(sub) < 3:
                continue

            r, p = stats.pearsonr(sub[gamma_col], sub['clustering_mean'])
            sig  = '***' if p < .001 else ('**' if p < .01 else ('*' if p < .05 else ''))
            print(f"  {comp:<8} {region:<6} {ctype:<10} {r:>7.3f} {p:>8.4f} {len(sub):>4}  {sig}")
            corr_rows.append(dict(component=comp, region=region,
                                  clust_type=ctype, r=r, p=p, n=len(sub)))

# ============================================================================
# 5. FDR ON OMNIBUS RESULTS
# ============================================================================

for res, label in [(res_osc, 'OSC'), (res_frac, 'FRAC')]:
    _, res['p_fdr'], _, _ = multipletests(res['p'].values, method='fdr_bh')
    res['sig'] = res['p_fdr'].apply(
        lambda p: '***' if p < .001 else ('**' if p < .01 else ('*' if p < .05 else ''))
    )

cols_show = ['label', 'parameter', 'beta', 'SE', 't', 'p', 'p_fdr', 'sig', 'N', 'R2_adj']

print("\n\n" + "="*70)
print("OSC MODEL — FDR corrected")
print("="*70)
print(res_osc[cols_show].to_string(index=False))

print("\n\n" + "="*70)
print("FRAC MODEL — FDR corrected")
print("="*70)
print(res_frac[cols_show].to_string(index=False))

# ============================================================================
# 6. SAVE
# ============================================================================

pd.concat([res_osc, res_frac], ignore_index=True).to_csv(
    'gamma_retrieval_predict_clustering_omnibus_results.csv', index=False)
pd.DataFrame(corr_rows).to_csv(
    'gamma_retrieval_clustering_correlations.csv', index=False)

print("\nSaved: gamma_retrieval_predict_clustering_omnibus_results.csv")
print("Saved: gamma_retrieval_clustering_correlations.csv")

Subject-level shape: (36, 9)
Subjects: 36

Descriptives:
       retrieval_osc_ssl_LHP  retrieval_osc_ssl_RHP  retrieval_frac_ssl_LHP  retrieval_frac_ssl_RHP  sp_clustering_LHP  sp_clustering_RHP  tc_clustering_LHP  tc_clustering_RHP
count                 27.000                 27.000                  27.000                  27.000             27.000             27.000             27.000             27.000
mean                   0.331                  0.316                   0.979                   1.187              0.514              0.529              0.595              0.604
std                    0.295                  0.285                   0.654                   0.659              0.139              0.116              0.109              0.103
min                    0.011                 -0.296                   0.053                   0.052              0.186              0.261              0.356              0.389
25%                    0.076                  0.157            

In [23]:
"""
Subject-level OLS: gamma power ~ clustering only
No z-scoring — raw values.
Runs both oscillatory (osc) and fractal (frac) as outcomes,
to test specificity of the clustering effect.

8 models total:
  A: LHP osc  ~ SP      E: LHP frac ~ SP
  B: RHP osc  ~ SP      F: RHP frac ~ SP
  C: LHP osc  ~ TC      G: LHP frac ~ TC
  D: RHP osc  ~ TC      H: RHP frac ~ TC

FDR correction separately within osc models and within frac models.
"""

import pandas as pd
import numpy as np
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. LOAD AND AVERAGE TO SUBJECT LEVEL
# ============================================================================

df = pd.read_csv('trial_level_summary.csv')

cols = [
    'encoding_osc_ssl_LHP',  'encoding_osc_ssl_RHP',
    'encoding_frac_ssl_LHP', 'encoding_frac_ssl_RHP',
    'sp_clustering_LHP',     'sp_clustering_RHP',
    'tc_clustering_LHP',     'tc_clustering_RHP',
]
cols = [c for c in cols if c in df.columns]

subj = df.groupby('subject')[cols].mean().reset_index()

print(f"Subject-level shape: {subj.shape}")
print(f"\nDescriptives:")
print(subj[cols].describe().round(3).to_string())

# ============================================================================
# 2. OLS FUNCTION
# ============================================================================

def run_ols(outcome, predictors, label, data):
    sub = data[[outcome] + predictors].dropna()
    n   = len(sub)
    X   = add_constant(sub[predictors])
    y   = sub[outcome]
    fit = OLS(y, X).fit()

    print(f"\n{'='*60}")
    print(f"{label}  (N={n})")
    print(f"{'='*60}")
    print(fit.summary2())

    rows = []
    for pred in predictors:
        rows.append({
            'label':     label,
            'outcome':   outcome,
            'predictor': pred,
            'beta':      fit.params[pred],
            'SE':        fit.bse[pred],
            't':         fit.tvalues[pred],
            'p':         fit.pvalues[pred],
            'R2':        fit.rsquared,
            'R2_adj':    fit.rsquared_adj,
            'N':         n,
        })
    return pd.DataFrame(rows)

# ============================================================================
# 3. OSCILLATORY MODELS
# ============================================================================

print("\n" + "#"*60)
print("# OSCILLATORY COMPONENT")
print("#"*60)

res_A = run_ols('encoding_osc_ssl_LHP', ['sp_clustering_LHP'],
                'A: LHP osc ~ SP',  subj)
res_B = run_ols('encoding_osc_ssl_RHP', ['sp_clustering_RHP'],
                'B: RHP osc ~ SP',  subj)
res_C = run_ols('encoding_osc_ssl_LHP', ['tc_clustering_LHP'],
                'C: LHP osc ~ TC',  subj)
res_D = run_ols('encoding_osc_ssl_RHP', ['tc_clustering_RHP'],
                'D: RHP osc ~ TC',  subj)

# ============================================================================
# 4. FRACTAL MODELS
# ============================================================================

print("\n" + "#"*60)
print("# FRACTAL COMPONENT")
print("#"*60)

res_E = run_ols('encoding_frac_ssl_LHP', ['sp_clustering_LHP'],
                'E: LHP frac ~ SP', subj)
res_F = run_ols('encoding_frac_ssl_RHP', ['sp_clustering_RHP'],
                'F: RHP frac ~ SP', subj)
res_G = run_ols('encoding_frac_ssl_LHP', ['tc_clustering_LHP'],
                'G: LHP frac ~ TC', subj)
res_H = run_ols('encoding_frac_ssl_RHP', ['tc_clustering_RHP'],
                'H: RHP frac ~ TC', subj)

# ============================================================================
# 5. RESULTS TABLES WITH SEPARATE FDR
# ============================================================================

osc_results  = pd.concat([res_A, res_B, res_C, res_D], ignore_index=True)
frac_results = pd.concat([res_E, res_F, res_G, res_H], ignore_index=True)

# FDR separately within osc and frac (4 tests each)
_, osc_results['p_fdr'],  _, _ = multipletests(osc_results['p'].values,  method='fdr_bh')
_, frac_results['p_fdr'], _, _ = multipletests(frac_results['p'].values, method='fdr_bh')

for res in [osc_results, frac_results]:
    res['sig'] = res['p_fdr'].apply(
        lambda p: '***' if p < .001 else ('**' if p < .01 else ('*' if p < .05 else ''))
    )

cols_show = ['label', 'predictor', 'beta', 'SE', 't', 'p', 'p_fdr', 'sig', 'N', 'R2_adj']

print("\n\n" + "="*70)
print("OSCILLATORY RESULTS — BH-FDR corrected across 4 tests")
print("="*70)
print(osc_results[cols_show].to_string(index=False))

print("\n\n" + "="*70)
print("FRACTAL RESULTS — BH-FDR corrected across 4 tests")
print("="*70)
print(frac_results[cols_show].to_string(index=False))

# ============================================================================
# 6. SAVE
# ============================================================================

all_results = pd.concat([osc_results, frac_results], ignore_index=True)
all_results['component'] = all_results['outcome'].apply(
    lambda x: 'oscillatory' if 'osc' in x else 'fractal'
)

all_results.to_csv('gamma_osc_frac_clustering_results.csv', index=False)
print("\nSaved: gamma_osc_frac_clustering_results.csv")

Subject-level shape: (36, 9)

Descriptives:
       encoding_osc_ssl_LHP  encoding_osc_ssl_RHP  encoding_frac_ssl_LHP  encoding_frac_ssl_RHP  sp_clustering_LHP  sp_clustering_RHP  tc_clustering_LHP  tc_clustering_RHP
count                27.000                27.000                 27.000                 27.000             27.000             27.000             27.000             27.000
mean                  0.322                 0.289                  1.046                  1.159              0.514              0.529              0.595              0.604
std                   0.299                 0.330                  0.808                  0.674              0.139              0.116              0.109              0.103
min                   0.018                -0.541                  0.045                  0.054              0.186              0.261              0.356              0.389
25%                   0.088                 0.056                  0.335                  0.601 

In [17]:
"""
Visualize subject-level OLS results:
  - Beta coefficients with error bars (±1 SE) for all 4 models
  - Scatter plots: gamma osc vs each significant predictor
  - One figure saved as gamma_osc_results_figure.png
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

# ============================================================================
# 1. LOAD DATA
# ============================================================================

df       = pd.read_csv('trial_level_summary.csv')
results  = pd.read_csv('gamma_osc_subject_level_results.csv')

# ── Rebuild subject-level averages ──────────────────────────────────────────
cols = [
    'encoding_osc_ssl_LHP', 'encoding_osc_ssl_RHP',
    'sp_clustering_LHP',    'sp_clustering_RHP',
    'tc_clustering_LHP',    'tc_clustering_RHP',
    'n_correct',            'nav_time_seconds',
]
cols  = [c for c in cols if c in df.columns]
subj  = df.groupby('subject')[cols].mean().reset_index()
subj['nav_speed'] = 1.0 / subj['nav_time_seconds']

for col in [c for c in cols if c != 'nav_time_seconds'] + ['nav_speed']:
    subj[col + '_z'] = (subj[col] - subj[col].mean()) / subj[col].std()

# ============================================================================
# 2. PLOTTING SETUP
# ============================================================================

BLUE   = '#185FA5'
GREEN  = '#0F6E56'
GRAY   = '#888780'
RED    = '#D85A30'

plt.rcParams.update({
    'font.family':     'sans-serif',
    'font.size':       11,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.linewidth':  0.8,
    'xtick.major.size': 3,
    'ytick.major.size': 3,
})

fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

# ============================================================================
# 3. PANEL A & B — BETA COEFFICIENT PLOTS (SP and TC models)
# ============================================================================

predictor_labels = {
    'sp_clustering_LHP_z': 'SP clust',
    'sp_clustering_RHP_z': 'SP clust',
    'tc_clustering_LHP_z': 'TC clust',
    'tc_clustering_RHP_z': 'TC clust',
    'nav_speed_z':         'Nav speed',
    'n_correct_z':         'N correct',
}

def plot_betas(ax, models, title, hemi_colors):
    """
    Bar chart of beta coefficients ± SE for one set of models.
    models: list of (label_substr, hemisphere_label, color)
    """
    n_pred   = 3
    x        = np.arange(n_pred)
    width    = 0.35
    offsets  = [-width/2, width/2]

    for idx, (substr, hem_label, color) in enumerate(models):
        sub = results[results['label'].str.contains(substr)].copy()
        # keep only the 3 main predictors (exclude const)
        sub = sub[~sub['predictor'].str.contains('const')]

        betas = sub['beta'].values
        ses   = sub['SE'].values
        ps    = sub['p_fdr'].values

        bars = ax.bar(
            x + offsets[idx], betas, width,
            color=color, alpha=0.85, label=hem_label,
            zorder=3, linewidth=0
        )
        ax.errorbar(
            x + offsets[idx], betas,
            yerr=ses, fmt='none',
            color='#444', linewidth=1.2, capsize=3, zorder=4
        )

        # significance markers
        for i, (b, p) in enumerate(zip(betas, ps)):
            if p < 0.05:
                marker = '***' if p < .001 else ('**' if p < .01 else '*')
                ypos   = b + ses[i] + 0.05 if b >= 0 else b - ses[i] - 0.12
                ax.text(x[i] + offsets[idx], ypos, marker,
                        ha='center', va='bottom', fontsize=10,
                        color=RED, fontweight='bold')
            elif p < 0.1:
                ypos = b + ses[i] + 0.05 if b >= 0 else b - ses[i] - 0.12
                ax.text(x[i] + offsets[idx], ypos, '†',
                        ha='center', va='bottom', fontsize=11, color=GRAY)

    ax.axhline(0, color='#aaa', linewidth=0.8, zorder=2)
    ax.set_xticks(x)

    # derive tick labels from the first model's predictors
    first_sub = results[results['label'].str.contains(models[0][0])]
    first_sub = first_sub[~first_sub['predictor'].str.contains('const')]
    ax.set_xticklabels(
        [predictor_labels.get(p, p) for p in first_sub['predictor'].values],
        fontsize=10
    )
    ax.set_ylabel('Standardized β', fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='normal', pad=8)
    ax.legend(fontsize=9, frameon=False)
    ax.set_ylim(
        min(results['beta'].min() - results['SE'].max() - 0.25, -0.3),
        max(results['beta'].max() + results['SE'].max() + 0.25,  0.7)
    )
    ax.yaxis.grid(True, linewidth=0.4, color='#ddd', zorder=0)
    ax.set_axisbelow(True)

ax_spbeta = fig.add_subplot(gs[0, 0])
plot_betas(
    ax_spbeta,
    models=[('SP.*LHP', 'LHP', BLUE), ('SP.*RHP', 'RHP', GREEN)],
    title='A   SP clustering models — beta coefficients',
    hemi_colors=[BLUE, GREEN]
)

ax_tcbeta = fig.add_subplot(gs[0, 1])
plot_betas(
    ax_tcbeta,
    models=[('TC.*LHP', 'LHP', BLUE), ('TC.*RHP', 'RHP', GREEN)],
    title='B   TC clustering models — beta coefficients',
    hemi_colors=[BLUE, GREEN]
)

# ============================================================================
# 4. PANEL C — SCATTER: SP clustering RHP vs RHP gamma (the key finding)
# ============================================================================

ax_sc1 = fig.add_subplot(gs[1, 0])

x_col = 'sp_clustering_RHP_z'
y_col = 'encoding_osc_ssl_RHP_z'
sub   = subj[[x_col, y_col]].dropna()
x_    = sub[x_col].values
y_    = sub[y_col].values

slope, intercept, r, p, _ = stats.linregress(x_, y_)
xline = np.linspace(x_.min(), x_.max(), 100)

ax_sc1.scatter(x_, y_, color=GREEN, alpha=0.75, s=45, zorder=3,
               edgecolors='white', linewidths=0.5)
ax_sc1.plot(xline, slope * xline + intercept,
            color=GREEN, linewidth=1.8, zorder=4)

# shaded CI band
n    = len(x_)
se_  = np.std(y_ - (slope * x_ + intercept), ddof=2)
ci   = se_ * stats.t.ppf(0.975, df=n-2) * np.sqrt(
           1/n + (xline - x_.mean())**2 / np.sum((x_ - x_.mean())**2))
ax_sc1.fill_between(xline,
                     slope * xline + intercept - ci,
                     slope * xline + intercept + ci,
                     color=GREEN, alpha=0.12, zorder=2)

p_str = f'p = {p:.3f}' if p >= 0.001 else 'p < 0.001'
ax_sc1.text(0.05, 0.92,
            f'r = {r:.2f},  {p_str}',
            transform=ax_sc1.transAxes,
            fontsize=9, color=GREEN, va='top')
ax_sc1.set_xlabel('SP clustering RHP (z)', fontsize=10)
ax_sc1.set_ylabel('Gamma osc RHP (z)', fontsize=10)
ax_sc1.set_title('C   SP clustering vs RHP gamma', fontsize=11, fontweight='normal', pad=8)

# ============================================================================
# 5. PANEL D — SCATTER: SP clustering LHP vs LHP gamma (trending)
# ============================================================================

ax_sc2 = fig.add_subplot(gs[1, 1])

x_col2 = 'sp_clustering_LHP_z'
y_col2 = 'encoding_osc_ssl_LHP_z'
sub2   = subj[[x_col2, y_col2]].dropna()
x2_    = sub2[x_col2].values
y2_    = sub2[y_col2].values

slope2, intercept2, r2, p2, _ = stats.linregress(x2_, y2_)
xline2 = np.linspace(x2_.min(), x2_.max(), 100)

ax_sc2.scatter(x2_, y2_, color=BLUE, alpha=0.75, s=45, zorder=3,
               edgecolors='white', linewidths=0.5)
ax_sc2.plot(xline2, slope2 * xline2 + intercept2,
            color=BLUE, linewidth=1.8, zorder=4, linestyle='--')

se2_ = np.std(y2_ - (slope2 * x2_ + intercept2), ddof=2)
ci2  = se2_ * stats.t.ppf(0.975, df=len(x2_)-2) * np.sqrt(
            1/len(x2_) + (xline2 - x2_.mean())**2 / np.sum((x2_ - x2_.mean())**2))
ax_sc2.fill_between(xline2,
                     slope2 * xline2 + intercept2 - ci2,
                     slope2 * xline2 + intercept2 + ci2,
                     color=BLUE, alpha=0.10, zorder=2)

p_str2 = f'p = {p2:.3f}' if p2 >= 0.001 else 'p < 0.001'
ax_sc2.text(0.05, 0.92,
            f'r = {r2:.2f},  {p_str2}',
            transform=ax_sc2.transAxes,
            fontsize=9, color=BLUE, va='top')
ax_sc2.set_xlabel('SP clustering LHP (z)', fontsize=10)
ax_sc2.set_ylabel('Gamma osc LHP (z)', fontsize=10)
ax_sc2.set_title('D   SP clustering vs LHP gamma (trend)', fontsize=11, fontweight='normal', pad=8)

# ============================================================================
# 6. FIGURE ANNOTATIONS & SAVE
# ============================================================================

fig.text(
    0.5, 0.01,
    '* p < .05   ** p < .01   *** p < .001 (FDR-corrected)   † p < .10 trend   '
    'Error bars = ±1 SE   Dashed line = non-significant trend',
    ha='center', fontsize=8, color='#888'
)

plt.savefig('gamma_osc_results_figure.png', dpi=150, bbox_inches='tight',
            facecolor='white')
print("Saved: gamma_osc_results_figure.png")

ValueError: shape mismatch: objects cannot be broadcast to a single shape

In [15]:
"""
Subject-level LMM analysis predicting gamma oscillatory power (LHP / RHP)
from:
  1. Spatial clustering (SP)
  2. Navigation speed (1 / nav_time_seconds)
  3. Number of correct recalls (n_correct)

Two separate models are run:
  - Model A: outcome = encoding_osc_ssl_LHP
  - Model B: outcome = encoding_osc_ssl_RHP

Each model has three predictors entered together (simultaneous), with
nested random effects (session within subject).

Dependencies: pandas, numpy, statsmodels, scipy
"""

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests  # BH-FDR
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. LOAD DATA
# ============================================================================

df = pd.read_csv('trial_level_summary.csv')
print(f"Raw shape: {df.shape}")
print(f"Subjects:  {sorted(df['subject'].unique())}")

# ============================================================================
# 2. AGGREGATE TO SUBJECT LEVEL
# ============================================================================
# Average all numeric columns across trials within each subject × session,
# then average sessions within each subject.

numeric_cols = [
    'encoding_osc_ssl_LHP', 'encoding_osc_ssl_RHP',
    'encoding_frac_ssl_LHP', 'encoding_frac_ssl_RHP',
    'sp_clustering_LHP',     'sp_clustering_RHP',
    'tc_clustering_LHP',     'tc_clustering_RHP',
    'recall_rate',           'n_correct',
    'nav_time_seconds',
]

# Keep only columns that actually exist
numeric_cols = [c for c in numeric_cols if c in df.columns]
missing = set(['encoding_osc_ssl_LHP', 'encoding_osc_ssl_RHP',
               'sp_clustering_LHP',    'sp_clustering_RHP',
               'n_correct',            'nav_time_seconds']) - set(df.columns)
if missing:
    raise ValueError(f"Required columns missing from CSV: {missing}")

# Session-level means
sess_df = (
    df.groupby(['subject', 'session'])[numeric_cols]
    .mean()
    .reset_index()
)

# Subject-level means (average across sessions)
subj_df = (
    sess_df.groupby('subject')[numeric_cols]
    .mean()
    .reset_index()
)

# ── Derived predictor: navigation SPEED (1 / time) ──────────────────────────
subj_df['nav_speed'] = 1.0 / subj_df['nav_time_seconds']

print(f"\nSubject-level df shape: {subj_df.shape}")
print(f"Subjects with nav data:  {subj_df['nav_speed'].notna().sum()}")

# ── Z-score all predictors and outcomes for interpretable betas ─────────────
cols_to_z = [
    'encoding_osc_ssl_LHP', 'encoding_osc_ssl_RHP',
    'sp_clustering_LHP',     'sp_clustering_RHP',
    'tc_clustering_LHP',     'tc_clustering_RHP',
    'n_correct',             'nav_speed',
]
cols_to_z = [c for c in cols_to_z if c in subj_df.columns]

for col in cols_to_z:
    subj_df[col + '_z'] = (
        (subj_df[col] - subj_df[col].mean()) / subj_df[col].std()
    )

print(f"\nSubject-level descriptives (raw):")
print(subj_df[numeric_cols + ['nav_speed']].describe().round(3).to_string())

# ============================================================================
# 3. OLS AT SUBJECT LEVEL
# (N ~ 20-40 subjects — LMM not needed; simple OLS with robust SE)
# ============================================================================

from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

def run_ols(outcome_z, predictor_zs, label, data):
    """
    OLS predicting outcome_z from predictor_zs.
    Returns a tidy summary DataFrame and the fitted model.
    """
    sub = data[[outcome_z] + predictor_zs].dropna()
    n   = len(sub)
    print(f"\n{'='*60}")
    print(f"Model: {label}  (N = {n} subjects)")
    print(f"{'='*60}")

    X = add_constant(sub[predictor_zs])
    y = sub[outcome_z]

    model  = OLS(y, X).fit(cov_type='HC3')   # heteroscedasticity-robust SE
    print(model.summary2())

    rows = []
    for pred in predictor_zs:
        rows.append({
            'outcome':   outcome_z,
            'predictor': pred,
            'beta':      model.params[pred],
            'SE':        model.bse[pred],
            't':         model.tvalues[pred],
            'p':         model.pvalues[pred],
            'N':         n,
            'R2':        model.rsquared,
            'R2_adj':    model.rsquared_adj,
            'label':     label,
        })
    return pd.DataFrame(rows), model

# ── Model A: LHP ~ SP ────────────────────────────────────────────────────────
res_LHP_SP, _ = run_ols(
    outcome_z    = 'encoding_osc_ssl_LHP_z',
    predictor_zs = ['sp_clustering_LHP_z', 'nav_speed_z', 'n_correct_z'],
    label        = 'LHP gamma ~ SP_clust + nav_speed + n_correct',
    data         = subj_df,
)

# ── Model B: RHP ~ SP ────────────────────────────────────────────────────────
res_RHP_SP, _ = run_ols(
    outcome_z    = 'encoding_osc_ssl_RHP_z',
    predictor_zs = ['sp_clustering_RHP_z', 'nav_speed_z', 'n_correct_z'],
    label        = 'RHP gamma ~ SP_clust + nav_speed + n_correct',
    data         = subj_df,
)

# ── Model C: LHP ~ TC ────────────────────────────────────────────────────────
res_LHP_TC, _ = run_ols(
    outcome_z    = 'encoding_osc_ssl_LHP_z',
    predictor_zs = ['tc_clustering_LHP_z', 'nav_speed_z', 'n_correct_z'],
    label        = 'LHP gamma ~ TC_clust + nav_speed + n_correct',
    data         = subj_df,
)

# ── Model D: RHP ~ TC ────────────────────────────────────────────────────────
res_RHP_TC, _ = run_ols(
    outcome_z    = 'encoding_osc_ssl_RHP_z',
    predictor_zs = ['tc_clustering_RHP_z', 'nav_speed_z', 'n_correct_z'],
    label        = 'RHP gamma ~ TC_clust + nav_speed + n_correct',
    data         = subj_df,
)

# ============================================================================
# 4. COMBINED RESULTS TABLE WITH FDR CORRECTION
# ============================================================================

results = pd.concat([res_LHP_SP, res_RHP_SP, res_LHP_TC, res_RHP_TC], ignore_index=True)

# BH-FDR across all 12 tests (3 predictors × 2 hemispheres × 2 clustering types)
_, results['p_fdr'], _, _ = multipletests(results['p'].values, method='fdr_bh')
results['sig'] = results['p_fdr'].apply(
    lambda p: '***' if p < .001 else ('**' if p < .01 else ('*' if p < .05 else ''))
)

print("\n\n" + "="*70)
print("SP MODELS — FDR-corrected across 12 tests")
print("="*70)
print(results[results['label'].str.contains('SP')][
    ['label', 'predictor', 'beta', 'SE', 't', 'p', 'p_fdr', 'sig', 'N', 'R2_adj']
].to_string(index=False))

print("\n\n" + "="*70)
print("TC MODELS — FDR-corrected across 12 tests")
print("="*70)
print(results[results['label'].str.contains('TC')][
    ['label', 'predictor', 'beta', 'SE', 't', 'p', 'p_fdr', 'sig', 'N', 'R2_adj']
].to_string(index=False))

# ============================================================================
# 5. SAVE
# ============================================================================

results.to_csv('gamma_osc_subject_level_results.csv', index=False)
print("\nSaved: gamma_osc_subject_level_results.csv")

# ── Quick interpretation guide ───────────────────────────────────────────────
print("""
──────────────────────────────────────────────────────────────────
INTERPRETATION NOTES
──────────────────────────────────────────────────────────────────
All betas are standardized (z-scored predictors and outcomes),
so beta = SD change in gamma osc per 1 SD change in predictor.

Predictors:
  sp_clustering_[L/R]HP_z  — spatial clustering score (higher = more spatial)
  tc_clustering_[L/R]HP_z  — temporal clustering score (higher = more temporal)
  nav_speed_z              — 1/nav_time (higher = faster navigation)
  n_correct_z              — number of correctly recalled words

Outcomes:
  encoding_osc_ssl_LHP_z   — left hippocampal gamma oscillatory power at encoding
  encoding_osc_ssl_RHP_z   — right hippocampal gamma oscillatory power at encoding

FDR correction: Benjamini-Hochberg across all 12 predictor tests
  (3 predictors × 2 hemispheres × 2 clustering types).
Robust SEs (HC3) used to handle heteroscedasticity.
──────────────────────────────────────────────────────────────────
""")

Raw shape: (217, 22)
Subjects:  ['R1494D', 'R1501J', 'R1502D', 'R1503E', 'R1504E', 'R1509E', 'R1520T', 'R1521E', 'R1523E', 'R1529T', 'R1530J', 'R1532T', 'R1536J', 'R1537T', 'R1538E', 'R1539E', 'R1542J', 'R1543E', 'R1546E', 'R1551T', 'R1554T', 'R1560T', 'R1561E', 'R1564J', 'R1567T', 'R1569T', 'R1570T', 'R1571T', 'R1572T', 'R1573T', 'R1575E', 'R1620J', 'R1637T', 'R1642J', 'R1651J', 'R1653J']

Subject-level df shape: (36, 13)
Subjects with nav data:  36

Subject-level descriptives (raw):
       encoding_osc_ssl_LHP  encoding_osc_ssl_RHP  encoding_frac_ssl_LHP  encoding_frac_ssl_RHP  sp_clustering_LHP  sp_clustering_RHP  tc_clustering_LHP  tc_clustering_RHP  recall_rate  n_correct  nav_time_seconds  nav_speed
count                27.000                27.000                 27.000                 27.000             27.000             27.000             27.000             27.000       36.000     36.000            36.000     36.000
mean                  0.323                 0.293           

In [11]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from scipy import stats
import matplotlib.pyplot as plt
import warnings
from joblib import Parallel, delayed
warnings.filterwarnings('ignore')

# ============================================================================
# SETTINGS
# ============================================================================

PHASE  = 'retrieval'
COMP   = 'frac'
REGION = 'RHP'

col_between = f'{PHASE}_{COMP}_ssl_between'

N_BOOT = 2000
N_JOBS = -1

# ============================================================================
# BUILD NAV TIME (seconds) PER SUBJECT
# ============================================================================

nav_clean = nav_df[nav_df['navigation_time_seconds'] <= 180].copy()
nav_subj  = (
    nav_clean.groupby('subject')
    .agg(nav_between = ('navigation_time_seconds', 'mean'))
    .reset_index()
)

# ============================================================================
# SUBJECT-LEVEL AGGREGATION
# ============================================================================

subj_med = (
    long[
        (long['clust_T'] == 0) &
        (long['region']  == REGION)
    ]
    .groupby('subject')
    .agg(
        gamma_between = (col_between,      'first'),
        sp_clustering = ('clustering_score', 'mean'),
    )
    .reset_index()
    .merge(nav_subj, on='subject', how='inner')
    .dropna()
)

print(f"N subjects: {len(subj_med)}")
print(subj_med[['subject', 'gamma_between', 'nav_between', 'sp_clustering']].to_string(index=False))

# ============================================================================
# OBSERVED PATHS — all OLS at subject level
# ============================================================================

# c-path: total effect gamma → SP clustering
m_c    = smf.ols('sp_clustering ~ gamma_between', data=subj_med).fit()
c_coef = m_c.params['gamma_between']
c_p    = m_c.pvalues['gamma_between']
c_t    = m_c.tvalues['gamma_between']
print(f"\nc-path  total  (γ → SP):          β={c_coef:+.4f}, t={c_t:+.3f}, p={c_p:.4f}")

# a-path: gamma → nav time
m_a    = smf.ols('nav_between ~ gamma_between', data=subj_med).fit()
a_coef = m_a.params['gamma_between']
a_p    = m_a.pvalues['gamma_between']
a_t    = m_a.tvalues['gamma_between']
print(f"a-path  (γ → nav):                β={a_coef:+.4f}, t={a_t:+.3f}, p={a_p:.4f}")

# b and c' paths: gamma + nav → SP clustering
m_full  = smf.ols('sp_clustering ~ gamma_between + nav_between', data=subj_med).fit()
b_coef  = m_full.params['nav_between']
b_p     = m_full.pvalues['nav_between']
b_t     = m_full.tvalues['nav_between']
c_prime = m_full.params['gamma_between']
cp_p    = m_full.pvalues['gamma_between']
cp_t    = m_full.tvalues['gamma_between']
print(f"b-path  (nav → SP | γ):           β={b_coef:+.6f}, t={b_t:+.3f}, p={b_p:.4f}")
print(f"c'-path direct (γ → SP | nav):    β={c_prime:+.4f}, t={cp_t:+.3f}, p={cp_p:.4f}")

indirect_obs = a_coef * b_coef
print(f"\nIndirect effect (a×b):            {indirect_obs:+.8f}")

# ============================================================================
# BOOTSTRAP — parallel, resample subjects
# ============================================================================

n_subj = len(subj_med)

def one_bootstrap(seed):
    rng     = np.random.default_rng(seed)
    idx     = rng.choice(n_subj, size=n_subj, replace=True)
    boot_df = subj_med.iloc[idx].copy().reset_index(drop=True)
    try:
        ma = smf.ols('nav_between ~ gamma_between',                 data=boot_df).fit()
        mb = smf.ols('sp_clustering ~ gamma_between + nav_between', data=boot_df).fit()
        return ma.params['gamma_between'] * mb.params['nav_between']
    except Exception:
        return np.nan

print(f"\nRunning {N_BOOT} bootstrap iterations on {N_JOBS} cores...")

boot_results  = Parallel(n_jobs=N_JOBS, verbose=5)(
    delayed(one_bootstrap)(seed) for seed in range(N_BOOT)
)

boot_indirect = np.array(boot_results, dtype=float)
boot_indirect = boot_indirect[~np.isnan(boot_indirect)]
n_valid       = len(boot_indirect)
print(f"Valid bootstrap samples: {n_valid} / {N_BOOT}")

ci_lo, ci_hi = np.percentile(boot_indirect, [2.5, 97.5])
p_boot       = np.mean(boot_indirect <= 0) if indirect_obs > 0 else np.mean(boot_indirect >= 0)
p_boot       = 2 * min(p_boot, 1 - p_boot)

prop_mediated = indirect_obs / c_coef if abs(c_coef) > 1e-10 else np.nan

# ============================================================================
# SUMMARY
# ============================================================================

print(f"""
{'='*65}
BETWEEN-SUBJECT MEDIATION  [{PHASE.upper()} | {COMP.upper()} | {REGION}]
N subjects = {len(subj_med)}
Nav time units = seconds (outliers >180s excluded)
{'='*65}
Path                              β              t        p
{'─'*65}
c  total  (γ → SP clust)        {c_coef:+.4f}         {c_t:+.3f}    {c_p:.4f}
a  (γ → nav time)               {a_coef:+.4f}         {a_t:+.3f}    {a_p:.4f}
b  (nav → SP clust | γ)         {b_coef:+.6f}         {b_t:+.3f}    {b_p:.4f}
c' direct (γ → SP | nav)        {c_prime:+.4f}         {cp_t:+.3f}    {cp_p:.4f}
{'─'*65}
Indirect (a×b)                  {indirect_obs:+.8f}
95% bootstrap CI                [{ci_lo:+.8f}, {ci_hi:+.8f}]
Bootstrap p                     {p_boot:.4f}
Proportion mediated             {prop_mediated:.3f} ({prop_mediated*100:.1f}%)
Mediation                       {'SIGNIFICANT' if ci_lo * ci_hi > 0 else 'not significant'}
{'='*65}
""")

# ============================================================================
# FIGURE
# ============================================================================

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.patch.set_facecolor('white')

# Panel A — path diagram
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis('off')

for (x, y, lbl) in [
    (1, 3, 'Gamma\npower'),
    (5, 5, 'Nav time\n(s)'),          # updated label
    (9, 3, 'SP\nclustering')
]:
    ax.add_patch(plt.Rectangle(
        (x-1, y-0.7), 2, 1.4,
        facecolor='#EAF3DE', edgecolor='#639922', linewidth=1.5
    ))
    ax.text(x, y, lbl, ha='center', va='center', fontsize=9, fontweight='500')

def arrow(ax, x1, y1, x2, y2, label, color='#444'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5))
    ax.text((x1+x2)/2, (y1+y2)/2 + 0.25, label,
            ha='center', va='bottom', fontsize=8, color=color)

arrow(ax, 2, 3.4, 4, 4.7,
      f'a={a_coef:+.3f}\np={a_p:.3f}', color='#2166AC')
arrow(ax, 6, 4.7, 8, 3.4,
      f'b={b_coef:+.5f}\np={b_p:.3f}', color='#2166AC')
arrow(ax, 2, 3,   8, 3,
      f"c'={c_prime:+.3f}\np={cp_p:.3f}", color='#D6604D')

sig_str = 'SIGNIFICANT' if ci_lo * ci_hi > 0 else 'n.s.'
ax.set_title(
    f'Path diagram\n'
    f'Indirect={indirect_obs:+.6f}\n'
    f'95% CI [{ci_lo:+.5f}, {ci_hi:+.5f}]  {sig_str}',
    fontsize=8
)

# Panel B — bootstrap distribution
ax = axes[1]
ax.hist(boot_indirect, bins=50, color='#B5D4F4', edgecolor='white', linewidth=0.3)
ax.axvline(indirect_obs, color='#D6604D', linewidth=2,
           label=f'Observed={indirect_obs:+.5f}')
ax.axvline(ci_lo, color='#444', linewidth=1, linestyle='--', label='95% CI')
ax.axvline(ci_hi, color='#444', linewidth=1, linestyle='--')
ax.axvline(0,     color='black', linewidth=0.8, linestyle=':')
ax.set_xlabel('Indirect effect (a×b)', fontsize=9)
ax.set_ylabel('Bootstrap count', fontsize=9)
ax.set_title(
    f'Bootstrap distribution\n'
    f'p={p_boot:.4f}, n_valid={n_valid}',
    fontsize=9
)
ax.legend(fontsize=7)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Panel C — partial regression plot (direct effect c')
ax = axes[2]
resid_gamma = smf.ols('gamma_between ~ nav_between', data=subj_med).fit().resid
resid_clust = smf.ols('sp_clustering ~ nav_between', data=subj_med).fit().resid
ax.scatter(resid_gamma, resid_clust,
           color='#D6604D', edgecolors='white', linewidths=0.5, s=80, zorder=3)
for i, row in subj_med.reset_index().iterrows():
    ax.annotate(row['subject'],
                (resid_gamma.iloc[i], resid_clust.iloc[i]),
                fontsize=6, xytext=(3, 3), textcoords='offset points')
m, b = np.polyfit(resid_gamma, resid_clust, 1)
xs   = np.linspace(resid_gamma.min(), resid_gamma.max(), 100)
ax.plot(xs, m*xs + b, 'k--', linewidth=1.2)
r_partial, p_partial = stats.pearsonr(resid_gamma, resid_clust)
ax.set_xlabel('Gamma (residualized for nav time)', fontsize=9)
ax.set_ylabel('SP clustering (residualized for nav time)', fontsize=9)
ax.set_title(
    f"Direct effect (c')\n"
    f"β={c_prime:+.4f}, p={cp_p:.4f}, r={r_partial:+.3f}",
    fontsize=9
)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.suptitle(
    f'Between-subject mediation: Nav time mediates γ → SP clustering\n'
    f'[{PHASE.upper()} | {COMP.upper()} | {REGION}]  N={len(subj_med)} subjects  '
    f'(nav time in seconds)',
    fontsize=11, fontweight='500'
)
plt.tight_layout()
plt.savefig(
    f'mediation_between_{PHASE}_{COMP}_{REGION}.png',
    dpi=200, bbox_inches='tight', facecolor='white'
)
plt.show()
print(f"Saved: mediation_between_{PHASE}_{COMP}_{REGION}.png")

N subjects: 27
subject  gamma_between  nav_between  sp_clustering
 R1494D       1.151131    28.673906       0.593954
 R1501J       1.612811    30.970684       0.519660
 R1502D       0.792238    29.217699       0.491887
 R1504E       0.172455    41.991903       0.526058
 R1520T       1.211198    34.606417       0.727778
 R1521E       0.148169    22.414681       0.458384
 R1523E       0.145108    24.396661       0.596561
 R1530J       1.555788    32.759958       0.800000
 R1532T       1.615807    25.890694       0.497206
 R1536J       1.957775    23.159956       0.510408
 R1538E       0.281835    44.138173       0.437626
 R1542J       1.690893    21.640333       0.538241
 R1546E       0.419732    23.648532       0.617956
 R1551T       1.508755    26.007375       0.507507
 R1554T       2.009218    36.737521       0.578571
 R1567T       1.655666    30.086229       0.614574
 R1569T       1.899221    44.201021       0.744444
 R1570T       1.913300    27.547000       0.505502
 R1571T       1.

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:    1.0s
[Parallel(n_jobs=-1)]: Done 392 tasks      | elapsed:    3.0s
[Parallel(n_jobs=-1)]: Done 1832 tasks      | elapsed:    9.5s
[Parallel(n_jobs=-1)]: Done 2000 out of 2000 | elapsed:   10.1s finished


Valid bootstrap samples: 2000 / 2000

BETWEEN-SUBJECT MEDIATION  [RETRIEVAL | FRAC | RHP]
N subjects = 27
Nav time units = seconds (outliers >180s excluded)
Path                              β              t        p
─────────────────────────────────────────────────────────────────
c  total  (γ → SP clust)        +0.0415         +1.205    0.2394
a  (γ → nav time)               -1.1178         -0.438    0.6649
b  (nav → SP clust | γ)         -0.001637         -0.599    0.5550
c' direct (γ → SP | nav)        +0.0397         +1.133    0.2685
─────────────────────────────────────────────────────────────────
Indirect (a×b)                  +0.00183023
95% bootstrap CI                [-0.02191170, +0.03289471]
Bootstrap p                     0.8420
Proportion mediated             0.044 (4.4%)
Mediation                       not significant

Saved: mediation_between_retrieval_frac_RHP.png


In [6]:
print([c for c in long.columns if 'nav' in c.lower()])

['mean_nav_time_samples', 'trial_nav_time_samples']


In [7]:
"""
Gamma-band iEEG LMM Analysis — Final Version + Covariates
==========================================================
Fixes applied:
  1. freq_hz removed from between-subject model
  2. FDR correction (Benjamini-Hochberg) applied within each model level
  3. Effect sizes reported: partial eta-squared (η²p) and Cohen's d
  4. Key interaction visualized: Neural × clustering_type (2×2 panel)
  5. Behavioral covariates added: recall rate + navigation time
     (re-merged from recall_df/nav_df since long is rebuilt from scratch)

Model structure
---------------
BETWEEN-SUBJECT (Level-2 OLS):
  clustering_score_j ~ neural_between_j + clust_T + region_RHP
                     + mean_recall_rate_b + mean_nav_time_b
                     + neural_between_j:clust_T
                     + neural_between_j:region_RHP

WITHIN-SUBJECT (LMM):
  clustering_score_ij ~ neural_within_ij + clust_T + region_RHP + freq_hz
                      + trial_recall_rate_within + trial_nav_time_samples_within
                      + neural_within_ij:clust_T
                      + neural_within_ij:region_RHP
                      + neural_within_ij:freq_hz
                      + (1 + neural_within | subject)
                      + (1 | subject:session)
"""

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# LOAD & FILTER TO GAMMA
# ============================================================================

gamma     = df[df['freq_hz'] >= 40].copy()
gamma     = gamma[gamma['SP_clustering_from_prev'].notna()].copy()
gamma_avg = gamma.copy()   # pre-stack reference for neural subject-mean computation

# ============================================================================
# RESHAPE: clustering_type as factor
# ============================================================================

sp = gamma.copy()
sp['clustering_score'] = sp['SP_clustering_from_prev']
sp['clustering_type']  = 'SP'

t = gamma.copy()
t['clustering_score'] = t['T_clustering_from_prev']
t['clustering_type']  = 'T'

long = pd.concat([sp, t], ignore_index=True)
long['clust_T']    = (long['clustering_type'] == 'T').astype(int)
long['region_RHP'] = (long['region'] == 'RHP').astype(int)
long['sess_id']    = long['subject'].astype(str) + '_' + long['session'].astype(str)

# ============================================================================
# RE-MERGE BEHAVIORAL COVARIATES
# long is rebuilt from scratch here, so the merges from the first script
# must be repeated. recall_df and nav_df must still be in memory.
# ============================================================================

# Trial-level recall rate
long = long.merge(
    recall_df[['subject', 'session', 'trial', 'recall_rate']]
              .rename(columns={'recall_rate': 'trial_recall_rate'}),
    on=['subject', 'session', 'trial'],
    how='left'
)

# Trial-level navigation time (averaged across pointing events within trial)
nav_trial = (
    nav_df.groupby(['subject', 'session', 'trial'])
    ['navigation_time_samples']
    .mean()
    .reset_index()
    .rename(columns={'navigation_time_samples': 'trial_nav_time_samples'})
)
long = long.merge(nav_trial, on=['subject', 'session', 'trial'], how='left')

print(f"trial_recall_rate NaNs:        {long['trial_recall_rate'].isna().sum()}")
print(f"trial_nav_time_samples NaNs:   {long['trial_nav_time_samples'].isna().sum()}")

# ============================================================================
# BETWEEN / WITHIN DECOMPOSITION — neural predictors
# Computed on gamma_avg (pre-stack) to avoid duplication from SP/T stacking
# ============================================================================

for col in ['encoding_osc_ssl', 'encoding_frac_ssl',
            'retrieval_osc_ssl', 'retrieval_frac_ssl']:
    subj_mean_map          = gamma_avg.groupby('subject')[col].mean()
    long[col + '_between'] = long['subject'].map(subj_mean_map)
    long[col + '_within']  = long[col] - long[col + '_between']

# ============================================================================
# BETWEEN / WITHIN DECOMPOSITION — behavioral covariates
# Use long[clust_T==0] as pre-stack equivalent (one row per trial×region)
# since behavioral cols are not in gamma_avg
# ============================================================================

gamma_long = long[long['clust_T'] == 0].copy()

for cov in ['trial_recall_rate', 'trial_nav_time_samples']:
    subj_mean_map          = gamma_long.groupby('subject')[cov].mean()
    long[cov + '_between'] = long['subject'].map(subj_mean_map)
    long[cov + '_within']  = long[cov] - long[cov + '_between']

# Aliases for between-subject model
long['mean_recall_rate_b'] = long['trial_recall_rate_between']
long['mean_nav_time_b']    = long['trial_nav_time_samples_between']

# ============================================================================
# EFFECT SIZE HELPERS
# ============================================================================

def partial_eta_sq(t_val, df_resid):
    if pd.isna(t_val) or pd.isna(df_resid):
        return np.nan
    return t_val**2 / (t_val**2 + df_resid)

def cohens_d_from_t(t_val, n):
    if pd.isna(t_val) or pd.isna(n) or n == 0:
        return np.nan
    return t_val / np.sqrt(n)

# ============================================================================
# MODELS
# ============================================================================

PHASES = ['encoding', 'retrieval']
COMPS  = ['osc', 'frac']

results_rows = []

for phase in PHASES:
    for comp in COMPS:

        col_base    = f'{phase}_{comp}_ssl'
        col_between = col_base + '_between'
        col_within  = col_base + '_within'

        # ── BETWEEN-SUBJECT MODEL (OLS) ───────────────────────────────────
        between_data = (
            long.groupby(['subject', 'clust_T', 'region_RHP'])
            [[col_between, 'clustering_score',
              'mean_recall_rate_b', 'mean_nav_time_b']]
            .mean()
            .reset_index()
            .dropna()
        )

        try:
            formula_b = (
                f'clustering_score ~ {col_between} + clust_T + region_RHP '
                f'+ mean_recall_rate_b + mean_nav_time_b '
                f'+ {col_between}:clust_T '
                f'+ {col_between}:region_RHP'
            )
            res_b      = smf.ols(formula_b, data=between_data).fit()
            df_resid_b = res_b.df_resid
            n_subj     = between_data['subject'].nunique()

            def gb(key):
                t = res_b.tvalues.get(key, np.nan)
                p = res_b.pvalues.get(key, np.nan)
                e = partial_eta_sq(t, df_resid_b)
                return t, p, e

            t_b,         p_b,         e_b         = gb(col_between)
            t_b_xclust,  p_b_xclust,  e_b_xclust  = gb(f'{col_between}:clust_T')
            t_b_xregion, p_b_xregion, e_b_xregion = gb(f'{col_between}:region_RHP')
            t_clust_b,   p_clust_b,   e_clust_b   = gb('clust_T')
            t_region_b,  p_region_b,  e_region_b  = gb('region_RHP')
            t_recall_b,  p_recall_b,  e_recall_b  = gb('mean_recall_rate_b')
            t_nav_b,     p_nav_b,     e_nav_b     = gb('mean_nav_time_b')
            d_b    = cohens_d_from_t(t_b, n_subj)
            n_b    = int(res_b.nobs)
            note_b = 'ok'

        except Exception as e:
            (t_b, p_b, e_b, t_b_xclust, p_b_xclust, e_b_xclust,
             t_b_xregion, p_b_xregion, e_b_xregion,
             t_clust_b, p_clust_b, e_clust_b,
             t_region_b, p_region_b, e_region_b,
             t_recall_b, p_recall_b, e_recall_b,
             t_nav_b, p_nav_b, e_nav_b,
             d_b, n_b, n_subj) = [np.nan] * 23
            note_b = f'failed: {e}'

        # ── WITHIN-SUBJECT MODEL (LMM) ────────────────────────────────────
        within_data = long[[
            'subject', 'sess_id', 'clustering_score',
            col_within, 'clust_T', 'region_RHP', 'freq_hz',
            'trial_recall_rate_within', 'trial_nav_time_samples_within'
        ]].dropna().copy()
        within_data = within_data.rename(columns={col_within: 'neural_within'})

        try:
            formula_w = (
                'clustering_score ~ neural_within + clust_T + region_RHP + freq_hz '
                '+ trial_recall_rate_within + trial_nav_time_samples_within '
                '+ neural_within:clust_T '
                '+ neural_within:region_RHP '
                '+ neural_within:freq_hz'
            )
            md = smf.mixedlm(
                formula_w,
                data=within_data,
                groups=within_data['subject'],
                re_formula='~neural_within',
                vc_formula={'sess_id': '0 + C(sess_id)'}
            )
            res_w      = md.fit(reml=True, method='powell')
            df_resid_w = res_w.df_resid

            def gw(key):
                t = res_w.tvalues.get(key, np.nan)
                p = res_w.pvalues.get(key, np.nan)
                e = partial_eta_sq(t, df_resid_w)
                return t, p, e

            t_w,         p_w,         e_w         = gw('neural_within')
            t_w_xclust,  p_w_xclust,  e_w_xclust  = gw('neural_within:clust_T')
            t_w_xregion, p_w_xregion, e_w_xregion = gw('neural_within:region_RHP')
            t_w_xfreq,   p_w_xfreq,   e_w_xfreq   = gw('neural_within:freq_hz')
            t_clust_w,   p_clust_w,   e_clust_w   = gw('clust_T')
            t_region_w,  p_region_w,  e_region_w  = gw('region_RHP')
            t_freq_w,    p_freq_w,    e_freq_w    = gw('freq_hz')
            t_recall_w,  p_recall_w,  e_recall_w  = gw('trial_recall_rate_within')
            t_nav_w,     p_nav_w,     e_nav_w     = gw('trial_nav_time_samples_within')
            d_w    = cohens_d_from_t(t_w, within_data['subject'].nunique())
            n_w    = int(res_w.nobs)
            conv_w = res_w.converged
            note_w = 'ok' if conv_w else 'ok (convergence warning)'

        except Exception as e:
            (t_w, p_w, e_w, t_w_xclust, p_w_xclust, e_w_xclust,
             t_w_xregion, p_w_xregion, e_w_xregion,
             t_w_xfreq, p_w_xfreq, e_w_xfreq,
             t_clust_w, p_clust_w, e_clust_w,
             t_region_w, p_region_w, e_region_w,
             t_freq_w, p_freq_w, e_freq_w,
             t_recall_w, p_recall_w, e_recall_w,
             t_nav_w, p_nav_w, e_nav_w,
             d_w, n_w) = [np.nan] * 27
            note_w = f'failed: {e}'

        results_rows.append({
            'phase': phase, 'component': comp,
            # Between — neural
            't_between':           t_b,          'p_between':           p_b,          'eta2p_between':           e_b,
            't_between_x_clustT':  t_b_xclust,   'p_between_x_clustT':  p_b_xclust,   'eta2p_between_x_clustT':  e_b_xclust,
            't_between_x_RHP':     t_b_xregion,  'p_between_x_RHP':     p_b_xregion,  'eta2p_between_x_RHP':     e_b_xregion,
            't_clustT_between':    t_clust_b,    'p_clustT_between':    p_clust_b,    'eta2p_clustT_between':    e_clust_b,
            't_region_between':    t_region_b,   'p_region_between':    p_region_b,   'eta2p_region_between':    e_region_b,
            # Between — covariates
            't_recall_between':    t_recall_b,   'p_recall_between':    p_recall_b,   'eta2p_recall_between':    e_recall_b,
            't_nav_between':       t_nav_b,      'p_nav_between':       p_nav_b,      'eta2p_nav_between':       e_nav_b,
            'd_between': d_b, 'n_between': n_b, 'n_subjects': n_subj, 'note_between': note_b,
            # Within — neural
            't_within':            t_w,          'p_within':            p_w,          'eta2p_within':            e_w,
            't_within_x_clustT':   t_w_xclust,   'p_within_x_clustT':   p_w_xclust,   'eta2p_within_x_clustT':   e_w_xclust,
            't_within_x_RHP':      t_w_xregion,  'p_within_x_RHP':      p_w_xregion,  'eta2p_within_x_RHP':      e_w_xregion,
            't_within_x_freq':     t_w_xfreq,    'p_within_x_freq':     p_w_xfreq,    'eta2p_within_x_freq':     e_w_xfreq,
            't_clustT_within':     t_clust_w,    'p_clustT_within':     p_clust_w,    'eta2p_clustT_within':     e_clust_w,
            't_region_within':     t_region_w,   'p_region_within':     p_region_w,   'eta2p_region_within':     e_region_w,
            't_freq_within':       t_freq_w,     'p_freq_within':       p_freq_w,     'eta2p_freq_within':       e_freq_w,
            # Within — covariates
            't_recall_within':     t_recall_w,   'p_recall_within':     p_recall_w,   'eta2p_recall_within':     e_recall_w,
            't_nav_within':        t_nav_w,      'p_nav_within':        p_nav_w,      'eta2p_nav_within':        e_nav_w,
            'd_within': d_w, 'n_within': n_w, 'note_within': note_w,
        })

results = pd.DataFrame(results_rows)

# ============================================================================
# FDR CORRECTION (Benjamini-Hochberg)
# ============================================================================

def apply_fdr(results, p_cols, suffix):
    all_p = []
    for pc in p_cols:
        all_p.extend(results[pc].tolist())
    all_p = np.array(all_p, dtype=float)
    valid_mask = ~np.isnan(all_p)
    fdr_p = np.full_like(all_p, np.nan)
    if valid_mask.sum() > 0:
        _, p_corrected, _, _ = multipletests(all_p[valid_mask], method='fdr_bh')
        fdr_p[valid_mask] = p_corrected
    idx = 0
    for pc in p_cols:
        n = len(results)
        col_fdr = pc.replace('p_', 'pfdr_')
        col_sig = pc.replace('p_', 'sig_fdr_')
        results[col_fdr] = fdr_p[idx:idx+n]
        results[col_sig] = results[col_fdr].apply(
            lambda x: x < 0.05 if pd.notna(x) else False)
        idx += n
    return results

between_p_cols = ['p_between', 'p_between_x_clustT', 'p_between_x_RHP',
                  'p_clustT_between', 'p_region_between',
                  'p_recall_between', 'p_nav_between']

within_p_cols  = ['p_within', 'p_within_x_clustT', 'p_within_x_RHP',
                  'p_within_x_freq', 'p_clustT_within', 'p_region_within',
                  'p_freq_within', 'p_recall_within', 'p_nav_within']

results = apply_fdr(results, between_p_cols, 'between')
results = apply_fdr(results, within_p_cols,  'within')

for pc in between_p_cols + within_p_cols:
    results['sig_unc_' + pc[2:]] = results[pc].apply(
        lambda x: x < 0.05 if pd.notna(x) else False)

# ============================================================================
# SAVE
# ============================================================================

results.to_csv('gamma_lmm_results.csv', index=False)

# ============================================================================
# VISUALIZATION
# ============================================================================

def get_between_plot_data(long, col_between, phase, comp):
    data = (
        long.groupby(['subject', 'clust_T'])
        [[col_between, 'clustering_score']]
        .mean()
        .reset_index()
        .dropna()
    )
    return data[data['clust_T'] == 0].copy(), data[data['clust_T'] == 1].copy()

COL_SP = '#2166AC'; COL_T = '#D6604D'; COL_GRID = '#E8E8E8'

fig = plt.figure(figsize=(12, 10))
fig.patch.set_facecolor('white')
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.38)
panel_labels = [['A', 'B'], ['C', 'D']]

for pi, phase in enumerate(PHASES):
    for ci, comp in enumerate(COMPS):
        ax = fig.add_subplot(gs[pi, ci])
        col_base    = f'{phase}_{comp}_ssl'
        col_between = col_base + '_between'
        sp_data, t_data = get_between_plot_data(long, col_between, phase, comp)

        ax.scatter(sp_data[col_between], sp_data['clustering_score'],
                   color=COL_SP, s=60, zorder=3, alpha=0.85,
                   label='SP clustering', edgecolors='white', linewidths=0.5)
        ax.scatter(t_data[col_between],  t_data['clustering_score'],
                   color=COL_T,  s=60, zorder=3, alpha=0.85,
                   label='T clustering',  edgecolors='white', linewidths=0.5)

        for dat, col in [(sp_data, COL_SP), (t_data, COL_T)]:
            x = dat[col_between].values; y = dat['clustering_score'].values
            if len(x) > 1:
                m, b = np.polyfit(x, y, 1)
                xr = np.linspace(x.min(), x.max(), 100)
                ax.plot(xr, m*xr + b, color=col, linewidth=2, alpha=0.9)

        row = results[(results['phase']==phase) & (results['component']==comp)].iloc[0]

        def pstr(p):
            if pd.isna(p): return 'n.s.'
            if p < 0.001:  return 'p<.001'
            if p < 0.05:   return f'p={p:.3f}'
            return f'p={p:.2f} n.s.'

        ann = (f"Neural main: t={row['t_between']:+.2f}, {pstr(row['pfdr_between'])}, d={row['d_between']:.2f}\n"
               f"× Clust type: t={row['t_between_x_clustT']:+.2f}, {pstr(row['pfdr_between_x_clustT'])}")
        ax.text(0.03, 0.97, ann, transform=ax.transAxes, fontsize=7.5, va='top', ha='left',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='#F7F7F7', edgecolor='#CCCCCC', alpha=0.9))
        ax.text(-0.12, 1.05, panel_labels[pi][ci], transform=ax.transAxes,
                fontsize=14, fontweight='bold', va='top')

        comp_label = 'Oscillatory' if comp == 'osc' else 'Aperiodic (Fractal)'
        ax.set_title(f'{phase.capitalize()} — {comp_label} γ', fontsize=11, fontweight='bold', pad=8)
        ax.set_xlabel('Gamma power (subject mean, SSL)', fontsize=9)
        ax.set_ylabel('Clustering score', fontsize=9)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        ax.grid(True, color=COL_GRID, linewidth=0.7, zorder=0)
        ax.tick_params(labelsize=8)
        if pi == 0 and ci == 0:
            ax.legend(fontsize=8.5, framealpha=0.9, loc='lower right')

fig.suptitle('Between-subject gamma power predicts semantic clustering\n'
             'across encoding and retrieval phases (iEEG, n=26)',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig('gamma_neural_x_clustering_type.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print("Figure saved: gamma_neural_x_clustering_type.png")

# ============================================================================
# PRINT REPORT
# ============================================================================

lines = []; L = lines.append

L("=" * 110)
L("GAMMA-BAND iEEG LMM — Between- and Within-Subject Effects on Clustering  [+ COVARIATES]")
L("  Covariates: recall rate (mean/trial-within) and navigation time (mean/trial-within)")
L("  Factors   : clustering_type (SP=0/T=1), region (LHP=0/RHP=1)")
L("  Sig       : * uncorrected p<.05  | † FDR-corrected p<.05")
L("=" * 110)

def fmt(label, t, p_unc, p_fdr, eta2, d=None):
    sig_unc = '*' if (pd.notna(p_unc) and p_unc < 0.05) else ''
    sig_fdr = '†' if (pd.notna(p_fdr) and p_fdr < 0.05) else ''
    ts  = f'{t:+.3f}'    if pd.notna(t)     else '    NaN'
    pu  = f'{p_unc:.3f}' if pd.notna(p_unc) else '    NaN'
    pf  = f'{p_fdr:.3f}' if pd.notna(p_fdr) else '    NaN'
    e2  = f'{eta2:.3f}'  if pd.notna(eta2)  else '    NaN'
    d_s = f'{d:+.3f}'    if (d is not None and pd.notna(d)) else '      —'
    return f"  {label:<45} {ts:>9} {pu:>9} {pf:>9} {e2:>8} {d_s:>9}  {sig_unc}{sig_fdr}"

HDR = f"  {'Effect':<45} {'t':>9} {'p_unc':>9} {'p_fdr':>9} {'η²p':>8} {'d':>9}  sig"
SEP = f"  {'─' * 100}"

for _, row in results.iterrows():
    phase = row['phase']; comp = row['component']
    L(f"\n{'━' * 110}")
    L(f"  PHASE: {phase.upper()}   COMPONENT: {'OSCILLATORY (OSC)' if comp=='osc' else 'APERIODIC/FRACTAL (FRAC)'}")
    L(f"{'━' * 110}")

    if row['note_between'] == 'ok':
        L(f"\n  BETWEEN-SUBJECT (Level-2 OLS)  n_subjects={int(row['n_subjects'])}  n_obs={int(row['n_between'])}")
        L(HDR); L(SEP)
        L(fmt('Neural main effect',          row['t_between'],         row['p_between'],         row['pfdr_between'],         row['eta2p_between'],         row['d_between']))
        L(fmt('Clustering type (T vs SP)',    row['t_clustT_between'],  row['p_clustT_between'],  row['pfdr_clustT_between'],  row['eta2p_clustT_between']))
        L(fmt('Region (RHP vs LHP)',          row['t_region_between'],  row['p_region_between'],  row['pfdr_region_between'],  row['eta2p_region_between']))
        L(fmt('Neural × clustering_type',     row['t_between_x_clustT'],row['p_between_x_clustT'],row['pfdr_between_x_clustT'],row['eta2p_between_x_clustT']))
        L(fmt('Neural × region',              row['t_between_x_RHP'],   row['p_between_x_RHP'],   row['pfdr_between_x_RHP'],   row['eta2p_between_x_RHP']))
        L(SEP)
        L(fmt('COV: Mean recall rate',        row['t_recall_between'],  row['p_recall_between'],  row['pfdr_recall_between'],  row['eta2p_recall_between']))
        L(fmt('COV: Mean nav time (samples)', row['t_nav_between'],     row['p_nav_between'],     row['pfdr_nav_between'],     row['eta2p_nav_between']))
    else:
        L(f"\n  BETWEEN-SUBJECT  *** {row['note_between']} ***")

    status = row['note_within']
    if status.startswith('ok'):
        L(f"\n  WITHIN-SUBJECT (LMM)  n_obs={int(row['n_within'])}  [{status}]")
        L(HDR); L(SEP)
        L(fmt('Neural main effect',          row['t_within'],           row['p_within'],           row['pfdr_within'],           row['eta2p_within'],           row['d_within']))
        L(fmt('Clustering type (T vs SP)',    row['t_clustT_within'],    row['p_clustT_within'],    row['pfdr_clustT_within'],    row['eta2p_clustT_within']))
        L(fmt('Region (RHP vs LHP)',          row['t_region_within'],    row['p_region_within'],    row['pfdr_region_within'],    row['eta2p_region_within']))
        L(fmt('Frequency (freq_hz)',          row['t_freq_within'],      row['p_freq_within'],      row['pfdr_freq_within'],      row['eta2p_freq_within']))
        L(fmt('Neural × clustering_type',     row['t_within_x_clustT'],  row['p_within_x_clustT'],  row['pfdr_within_x_clustT'],  row['eta2p_within_x_clustT']))
        L(fmt('Neural × region',              row['t_within_x_RHP'],     row['p_within_x_RHP'],     row['pfdr_within_x_RHP'],     row['eta2p_within_x_RHP']))
        L(fmt('Neural × freq_hz',             row['t_within_x_freq'],    row['p_within_x_freq'],    row['pfdr_within_x_freq'],    row['eta2p_within_x_freq']))
        L(SEP)
        L(fmt('COV: Trial recall rate',       row['t_recall_within'],    row['p_recall_within'],    row['pfdr_recall_within'],    row['eta2p_recall_within']))
        L(fmt('COV: Trial nav time (within)', row['t_nav_within'],       row['p_nav_within'],       row['pfdr_nav_within'],       row['eta2p_nav_within']))
    else:
        L(f"\n  WITHIN-SUBJECT  *** {status} ***")

# FDR summary
L(f"\n\n{'=' * 110}")
L("FDR CORRECTION SUMMARY")
L(f"{'=' * 110}")

between_effects = [
    ('pfdr_between',          'Neural main'),
    ('pfdr_between_x_clustT', 'Neural × clust_T'),
    ('pfdr_between_x_RHP',    'Neural × region'),
    ('pfdr_clustT_between',   'Clust type main'),
    ('pfdr_region_between',   'Region main'),
    ('pfdr_recall_between',   'COV: recall rate'),
    ('pfdr_nav_between',      'COV: nav time'),
]
within_effects = [
    ('pfdr_within',           'Neural main'),
    ('pfdr_within_x_clustT',  'Neural × clust_T'),
    ('pfdr_within_x_RHP',     'Neural × region'),
    ('pfdr_within_x_freq',    'Neural × freq_hz'),
    ('pfdr_clustT_within',    'Clust type main'),
    ('pfdr_region_within',    'Region main'),
    ('pfdr_freq_within',      'Freq main'),
    ('pfdr_recall_within',    'COV: recall rate'),
    ('pfdr_nav_within',       'COV: nav time'),
]

for level, effects in [('Between-subject', between_effects),
                        ('Within-subject',  within_effects)]:
    L(f"\n{level} — effects surviving FDR correction:")
    found = False
    for _, row in results.iterrows():
        for pc, label in effects:
            p = row.get(pc, np.nan)
            if pd.notna(p) and p < 0.05:
                found = True
                tag = 'OSC' if row['component'] == 'osc' else 'FRAC'
                L(f"  [{row['phase'].upper()} | {tag}]  {label:<35} p_fdr={p:.3f}")
    if not found:
        L("  None.")

L(f"\n{'=' * 110}")
L("FILES SAVED:")
L("  gamma_lmm_results.csv              — full results with FDR p-values and effect sizes")
L("  gamma_lmm_summary.csv              — FDR-significant effects only")
L("  gamma_neural_x_clustering_type.png — Figure: Neural × clustering_type (2×2 panel)")
L("=" * 110)

report = '\n'.join(lines)
print(report)
with open('gamma_lmm_report.txt', 'w') as f:
    f.write(report)

sig_fdr_cols = [c for c in results.columns if c.startswith('sig_fdr_')]
results[results[sig_fdr_cols].any(axis=1)].to_csv('gamma_lmm_summary.csv', index=False)
print("\nDone.")

NameError: name 'recall_df' is not defined

In [36]:
"""
Gamma-band iEEG LMM Analysis — Final Version + Covariates
==========================================================
Fixes applied:
  1. freq_hz removed from between-subject model
  2. FDR correction (Benjamini-Hochberg) applied within each model level
  3. Effect sizes reported: partial eta-squared (η²p) and Cohen's d
  4. Key interaction visualized: Neural × clustering_type (2×2 panel)
  5. Behavioral covariates added: recall rate + navigation time
     (re-merged from recall_df/nav_df since long is rebuilt from scratch)
  6. Navigation time in SECONDS (outliers >180s excluded)

Model structure
---------------
BETWEEN-SUBJECT (Level-2 OLS):
  clustering_score_j ~ neural_between_j + clust_T + region_RHP
                     + mean_recall_rate_b + mean_nav_time_b
                     + neural_between_j:clust_T
                     + neural_between_j:region_RHP

WITHIN-SUBJECT (LMM):
  clustering_score_ij ~ neural_within_ij + clust_T + region_RHP + freq_hz
                      + trial_recall_rate_within + trial_nav_time_seconds_within
                      + neural_within_ij:clust_T
                      + neural_within_ij:region_RHP
                      + neural_within_ij:freq_hz
                      + (1 + neural_within | subject)
                      + (1 | subject:session)
"""

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# LOAD & FILTER TO GAMMA
# ============================================================================

gamma     = df[df['freq_hz'] >= 40].copy()
gamma     = gamma[gamma['SP_clustering_from_prev'].notna()].copy()
gamma_avg = gamma.copy()

# ============================================================================
# RESHAPE: clustering_type as factor
# ============================================================================

sp = gamma.copy()
sp['clustering_score'] = sp['SP_clustering_from_prev']
sp['clustering_type']  = 'SP'

t = gamma.copy()
t['clustering_score'] = t['T_clustering_from_prev']
t['clustering_type']  = 'T'

long = pd.concat([sp, t], ignore_index=True)
long['clust_T']    = (long['clustering_type'] == 'T').astype(int)
long['region_RHP'] = (long['region'] == 'RHP').astype(int)
long['sess_id']    = long['subject'].astype(str) + '_' + long['session'].astype(str)

# ============================================================================
# RE-MERGE BEHAVIORAL COVARIATES
# ============================================================================

# Trial-level recall rate
long = long.merge(
    recall_df[['subject', 'session', 'trial', 'recall_rate']]
              .rename(columns={'recall_rate': 'trial_recall_rate'}),
    on=['subject', 'session', 'trial'],
    how='left'
)

# Trial-level navigation time in SECONDS (outliers >180s excluded)
nav_trial = (
    nav_df[nav_df['navigation_time_seconds'] <= 180]
    .groupby(['subject', 'session', 'trial'])
    ['navigation_time_seconds']
    .mean()
    .reset_index()
    .rename(columns={'navigation_time_seconds': 'trial_nav_time_seconds'})
)
long = long.merge(nav_trial, on=['subject', 'session', 'trial'], how='left')

print(f"trial_recall_rate NaNs:       {long['trial_recall_rate'].isna().sum()}")
print(f"trial_nav_time_seconds NaNs:  {long['trial_nav_time_seconds'].isna().sum()}")

# ============================================================================
# BETWEEN / WITHIN DECOMPOSITION — neural predictors
# ============================================================================

for col in ['encoding_osc_ssl', 'encoding_frac_ssl',
            'retrieval_osc_ssl', 'retrieval_frac_ssl']:
    subj_mean_map          = gamma_avg.groupby('subject')[col].mean()
    long[col + '_between'] = long['subject'].map(subj_mean_map)
    long[col + '_within']  = long[col] - long[col + '_between']

# ============================================================================
# BETWEEN / WITHIN DECOMPOSITION — behavioral covariates
# ============================================================================

gamma_long = long[long['clust_T'] == 0].copy()

for cov in ['trial_recall_rate', 'trial_nav_time_seconds']:   # <-- seconds
    subj_mean_map          = gamma_long.groupby('subject')[cov].mean()
    long[cov + '_between'] = long['subject'].map(subj_mean_map)
    long[cov + '_within']  = long[cov] - long[cov + '_between']

# Aliases for between-subject model
long['mean_recall_rate_b'] = long['trial_recall_rate_between']
long['mean_nav_time_b']    = long['trial_nav_time_seconds_between']  # <-- seconds

# ============================================================================
# EFFECT SIZE HELPERS
# ============================================================================

def partial_eta_sq(t_val, df_resid):
    if pd.isna(t_val) or pd.isna(df_resid):
        return np.nan
    return t_val**2 / (t_val**2 + df_resid)

def cohens_d_from_t(t_val, n):
    if pd.isna(t_val) or pd.isna(n) or n == 0:
        return np.nan
    return t_val / np.sqrt(n)

# ============================================================================
# MODELS
# ============================================================================

PHASES = ['encoding', 'retrieval']
COMPS  = ['osc', 'frac']

results_rows = []

for phase in PHASES:
    for comp in COMPS:

        col_base    = f'{phase}_{comp}_ssl'
        col_between = col_base + '_between'
        col_within  = col_base + '_within'

        # ── BETWEEN-SUBJECT MODEL (OLS) ───────────────────────────────────
        between_data = (
            long.groupby(['subject', 'clust_T', 'region_RHP'])
            [[col_between, 'clustering_score',
              'mean_recall_rate_b', 'mean_nav_time_b']]
            .mean()
            .reset_index()
            .dropna()
        )

        try:
            formula_b = (
                f'clustering_score ~ {col_between} + clust_T + region_RHP '
                f'+ mean_recall_rate_b + mean_nav_time_b '
                f'+ {col_between}:clust_T '
                f'+ {col_between}:region_RHP'
            )
            res_b      = smf.ols(formula_b, data=between_data).fit()
            df_resid_b = res_b.df_resid
            n_subj     = between_data['subject'].nunique()

            def gb(key):
                t = res_b.tvalues.get(key, np.nan)
                p = res_b.pvalues.get(key, np.nan)
                e = partial_eta_sq(t, df_resid_b)
                return t, p, e

            t_b,         p_b,         e_b         = gb(col_between)
            t_b_xclust,  p_b_xclust,  e_b_xclust  = gb(f'{col_between}:clust_T')
            t_b_xregion, p_b_xregion, e_b_xregion = gb(f'{col_between}:region_RHP')
            t_clust_b,   p_clust_b,   e_clust_b   = gb('clust_T')
            t_region_b,  p_region_b,  e_region_b  = gb('region_RHP')
            t_recall_b,  p_recall_b,  e_recall_b  = gb('mean_recall_rate_b')
            t_nav_b,     p_nav_b,     e_nav_b     = gb('mean_nav_time_b')
            d_b    = cohens_d_from_t(t_b, n_subj)
            n_b    = int(res_b.nobs)
            note_b = 'ok'

        except Exception as e:
            (t_b, p_b, e_b, t_b_xclust, p_b_xclust, e_b_xclust,
             t_b_xregion, p_b_xregion, e_b_xregion,
             t_clust_b, p_clust_b, e_clust_b,
             t_region_b, p_region_b, e_region_b,
             t_recall_b, p_recall_b, e_recall_b,
             t_nav_b, p_nav_b, e_nav_b,
             d_b, n_b, n_subj) = [np.nan] * 23
            note_b = f'failed: {e}'

        # ── WITHIN-SUBJECT MODEL (LMM) ────────────────────────────────────
        within_data = long[[
            'subject', 'sess_id', 'clustering_score',
            col_within, 'clust_T', 'region_RHP', 'freq_hz',
            'trial_recall_rate_within', 'trial_nav_time_seconds_within'  # <-- seconds
        ]].dropna().copy()
        within_data = within_data.rename(columns={col_within: 'neural_within'})

        try:
            formula_w = (
                'clustering_score ~ neural_within + clust_T + region_RHP + freq_hz '
                '+ trial_recall_rate_within + trial_nav_time_seconds_within '  # <-- seconds
                '+ neural_within:clust_T '
                '+ neural_within:region_RHP '
                '+ neural_within:freq_hz'
            )
            md = smf.mixedlm(
                formula_w,
                data=within_data,
                groups=within_data['subject'],
                re_formula='~neural_within',
                vc_formula={'sess_id': '0 + C(sess_id)'}
            )
            res_w      = md.fit(reml=True, method='powell')
            df_resid_w = res_w.df_resid

            def gw(key):
                t = res_w.tvalues.get(key, np.nan)
                p = res_w.pvalues.get(key, np.nan)
                e = partial_eta_sq(t, df_resid_w)
                return t, p, e

            t_w,         p_w,         e_w         = gw('neural_within')
            t_w_xclust,  p_w_xclust,  e_w_xclust  = gw('neural_within:clust_T')
            t_w_xregion, p_w_xregion, e_w_xregion = gw('neural_within:region_RHP')
            t_w_xfreq,   p_w_xfreq,   e_w_xfreq   = gw('neural_within:freq_hz')
            t_clust_w,   p_clust_w,   e_clust_w   = gw('clust_T')
            t_region_w,  p_region_w,  e_region_w  = gw('region_RHP')
            t_freq_w,    p_freq_w,    e_freq_w    = gw('freq_hz')
            t_recall_w,  p_recall_w,  e_recall_w  = gw('trial_recall_rate_within')
            t_nav_w,     p_nav_w,     e_nav_w     = gw('trial_nav_time_seconds_within')  # <-- seconds
            d_w    = cohens_d_from_t(t_w, within_data['subject'].nunique())
            n_w    = int(res_w.nobs)
            conv_w = res_w.converged
            note_w = 'ok' if conv_w else 'ok (convergence warning)'

        except Exception as e:
            (t_w, p_w, e_w, t_w_xclust, p_w_xclust, e_w_xclust,
             t_w_xregion, p_w_xregion, e_w_xregion,
             t_w_xfreq, p_w_xfreq, e_w_xfreq,
             t_clust_w, p_clust_w, e_clust_w,
             t_region_w, p_region_w, e_region_w,
             t_freq_w, p_freq_w, e_freq_w,
             t_recall_w, p_recall_w, e_recall_w,
             t_nav_w, p_nav_w, e_nav_w,
             d_w, n_w) = [np.nan] * 27
            note_w = f'failed: {e}'

        results_rows.append({
            'phase': phase, 'component': comp,
            't_between':           t_b,          'p_between':           p_b,          'eta2p_between':           e_b,
            't_between_x_clustT':  t_b_xclust,   'p_between_x_clustT':  p_b_xclust,   'eta2p_between_x_clustT':  e_b_xclust,
            't_between_x_RHP':     t_b_xregion,  'p_between_x_RHP':     p_b_xregion,  'eta2p_between_x_RHP':     e_b_xregion,
            't_clustT_between':    t_clust_b,    'p_clustT_between':    p_clust_b,    'eta2p_clustT_between':    e_clust_b,
            't_region_between':    t_region_b,   'p_region_between':    p_region_b,   'eta2p_region_between':    e_region_b,
            't_recall_between':    t_recall_b,   'p_recall_between':    p_recall_b,   'eta2p_recall_between':    e_recall_b,
            't_nav_between':       t_nav_b,      'p_nav_between':       p_nav_b,      'eta2p_nav_between':       e_nav_b,
            'd_between': d_b, 'n_between': n_b, 'n_subjects': n_subj, 'note_between': note_b,
            't_within':            t_w,          'p_within':            p_w,          'eta2p_within':            e_w,
            't_within_x_clustT':   t_w_xclust,   'p_within_x_clustT':   p_w_xclust,   'eta2p_within_x_clustT':   e_w_xclust,
            't_within_x_RHP':      t_w_xregion,  'p_within_x_RHP':      p_w_xregion,  'eta2p_within_x_RHP':      e_w_xregion,
            't_within_x_freq':     t_w_xfreq,    'p_within_x_freq':     p_w_xfreq,    'eta2p_within_x_freq':     e_w_xfreq,
            't_clustT_within':     t_clust_w,    'p_clustT_within':     p_clust_w,    'eta2p_clustT_within':     e_clust_w,
            't_region_within':     t_region_w,   'p_region_within':     p_region_w,   'eta2p_region_within':     e_region_w,
            't_freq_within':       t_freq_w,     'p_freq_within':       p_freq_w,     'eta2p_freq_within':       e_freq_w,
            't_recall_within':     t_recall_w,   'p_recall_within':     p_recall_w,   'eta2p_recall_within':     e_recall_w,
            't_nav_within':        t_nav_w,      'p_nav_within':        p_nav_w,      'eta2p_nav_within':        e_nav_w,
            'd_within': d_w, 'n_within': n_w, 'note_within': note_w,
        })

results = pd.DataFrame(results_rows)

# ============================================================================
# FDR CORRECTION (Benjamini-Hochberg)
# ============================================================================

def apply_fdr(results, p_cols, suffix):
    all_p = []
    for pc in p_cols:
        all_p.extend(results[pc].tolist())
    all_p = np.array(all_p, dtype=float)
    valid_mask = ~np.isnan(all_p)
    fdr_p = np.full_like(all_p, np.nan)
    if valid_mask.sum() > 0:
        _, p_corrected, _, _ = multipletests(all_p[valid_mask], method='fdr_bh')
        fdr_p[valid_mask] = p_corrected
    idx = 0
    for pc in p_cols:
        n = len(results)
        col_fdr = pc.replace('p_', 'pfdr_')
        col_sig = pc.replace('p_', 'sig_fdr_')
        results[col_fdr] = fdr_p[idx:idx+n]
        results[col_sig] = results[col_fdr].apply(
            lambda x: x < 0.05 if pd.notna(x) else False)
        idx += n
    return results

between_p_cols = ['p_between', 'p_between_x_clustT', 'p_between_x_RHP',
                  'p_clustT_between', 'p_region_between',
                  'p_recall_between', 'p_nav_between']

within_p_cols  = ['p_within', 'p_within_x_clustT', 'p_within_x_RHP',
                  'p_within_x_freq', 'p_clustT_within', 'p_region_within',
                  'p_freq_within', 'p_recall_within', 'p_nav_within']

results = apply_fdr(results, between_p_cols, 'between')
results = apply_fdr(results, within_p_cols,  'within')

for pc in between_p_cols + within_p_cols:
    results['sig_unc_' + pc[2:]] = results[pc].apply(
        lambda x: x < 0.05 if pd.notna(x) else False)

# ============================================================================
# SAVE
# ============================================================================

results.to_csv('gamma_lmm_results.csv', index=False)

# ============================================================================
# VISUALIZATION
# ============================================================================

def get_between_plot_data(long, col_between, phase, comp):
    data = (
        long.groupby(['subject', 'clust_T'])
        [[col_between, 'clustering_score']]
        .mean()
        .reset_index()
        .dropna()
    )
    return data[data['clust_T'] == 0].copy(), data[data['clust_T'] == 1].copy()

COL_SP = '#2166AC'; COL_T = '#D6604D'; COL_GRID = '#E8E8E8'

fig = plt.figure(figsize=(12, 10))
fig.patch.set_facecolor('white')
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.38)
panel_labels = [['A', 'B'], ['C', 'D']]

for pi, phase in enumerate(PHASES):
    for ci, comp in enumerate(COMPS):
        ax = fig.add_subplot(gs[pi, ci])
        col_base    = f'{phase}_{comp}_ssl'
        col_between = col_base + '_between'
        sp_data, t_data = get_between_plot_data(long, col_between, phase, comp)

        ax.scatter(sp_data[col_between], sp_data['clustering_score'],
                   color=COL_SP, s=60, zorder=3, alpha=0.85,
                   label='SP clustering', edgecolors='white', linewidths=0.5)
        ax.scatter(t_data[col_between],  t_data['clustering_score'],
                   color=COL_T,  s=60, zorder=3, alpha=0.85,
                   label='T clustering',  edgecolors='white', linewidths=0.5)

        for dat, col in [(sp_data, COL_SP), (t_data, COL_T)]:
            x = dat[col_between].values; y = dat['clustering_score'].values
            if len(x) > 1:
                m, b = np.polyfit(x, y, 1)
                xr = np.linspace(x.min(), x.max(), 100)
                ax.plot(xr, m*xr + b, color=col, linewidth=2, alpha=0.9)

        row = results[(results['phase']==phase) & (results['component']==comp)].iloc[0]

        def pstr(p):
            if pd.isna(p): return 'n.s.'
            if p < 0.001:  return 'p<.001'
            if p < 0.05:   return f'p={p:.3f}'
            return f'p={p:.2f} n.s.'

        ann = (f"Neural main: t={row['t_between']:+.2f}, {pstr(row['pfdr_between'])}, d={row['d_between']:.2f}\n"
               f"× Clust type: t={row['t_between_x_clustT']:+.2f}, {pstr(row['pfdr_between_x_clustT'])}")
        ax.text(0.03, 0.97, ann, transform=ax.transAxes, fontsize=7.5, va='top', ha='left',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='#F7F7F7', edgecolor='#CCCCCC', alpha=0.9))
        ax.text(-0.12, 1.05, panel_labels[pi][ci], transform=ax.transAxes,
                fontsize=14, fontweight='bold', va='top')

        comp_label = 'Oscillatory' if comp == 'osc' else 'Aperiodic (Fractal)'
        ax.set_title(f'{phase.capitalize()} — {comp_label} γ', fontsize=11, fontweight='bold', pad=8)
        ax.set_xlabel('Gamma power (subject mean, SSL)', fontsize=9)
        ax.set_ylabel('Clustering score', fontsize=9)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        ax.grid(True, color=COL_GRID, linewidth=0.7, zorder=0)
        ax.tick_params(labelsize=8)
        if pi == 0 and ci == 0:
            ax.legend(fontsize=8.5, framealpha=0.9, loc='lower right')

fig.suptitle('Between-subject gamma power predicts semantic clustering\n'
             'across encoding and retrieval phases (iEEG, n=26)',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig('gamma_neural_x_clustering_type.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print("Figure saved: gamma_neural_x_clustering_type.png")

# ============================================================================
# PRINT REPORT
# ============================================================================

lines = []; L = lines.append

L("=" * 110)
L("GAMMA-BAND iEEG LMM — Between- and Within-Subject Effects on Clustering  [+ COVARIATES]")
L("  Covariates: recall rate (mean/trial-within) and navigation time in SECONDS (mean/trial-within)")
L("  Factors   : clustering_type (SP=0/T=1), region (LHP=0/RHP=1)")
L("  Sig       : * uncorrected p<.05  | † FDR-corrected p<.05")
L("=" * 110)

def fmt(label, t, p_unc, p_fdr, eta2, d=None):
    sig_unc = '*' if (pd.notna(p_unc) and p_unc < 0.05) else ''
    sig_fdr = '†' if (pd.notna(p_fdr) and p_fdr < 0.05) else ''
    ts  = f'{t:+.3f}'    if pd.notna(t)     else '    NaN'
    pu  = f'{p_unc:.3f}' if pd.notna(p_unc) else '    NaN'
    pf  = f'{p_fdr:.3f}' if pd.notna(p_fdr) else '    NaN'
    e2  = f'{eta2:.3f}'  if pd.notna(eta2)  else '    NaN'
    d_s = f'{d:+.3f}'    if (d is not None and pd.notna(d)) else '      —'
    return f"  {label:<45} {ts:>9} {pu:>9} {pf:>9} {e2:>8} {d_s:>9}  {sig_unc}{sig_fdr}"

HDR = f"  {'Effect':<45} {'t':>9} {'p_unc':>9} {'p_fdr':>9} {'η²p':>8} {'d':>9}  sig"
SEP = f"  {'─' * 100}"

for _, row in results.iterrows():
    phase = row['phase']; comp = row['component']
    L(f"\n{'━' * 110}")
    L(f"  PHASE: {phase.upper()}   COMPONENT: {'OSCILLATORY (OSC)' if comp=='osc' else 'APERIODIC/FRACTAL (FRAC)'}")
    L(f"{'━' * 110}")

    if row['note_between'] == 'ok':
        L(f"\n  BETWEEN-SUBJECT (Level-2 OLS)  n_subjects={int(row['n_subjects'])}  n_obs={int(row['n_between'])}")
        L(HDR); L(SEP)
        L(fmt('Neural main effect',          row['t_between'],         row['p_between'],         row['pfdr_between'],         row['eta2p_between'],         row['d_between']))
        L(fmt('Clustering type (T vs SP)',    row['t_clustT_between'],  row['p_clustT_between'],  row['pfdr_clustT_between'],  row['eta2p_clustT_between']))
        L(fmt('Region (RHP vs LHP)',          row['t_region_between'],  row['p_region_between'],  row['pfdr_region_between'],  row['eta2p_region_between']))
        L(fmt('Neural × clustering_type',     row['t_between_x_clustT'],row['p_between_x_clustT'],row['pfdr_between_x_clustT'],row['eta2p_between_x_clustT']))
        L(fmt('Neural × region',              row['t_between_x_RHP'],   row['p_between_x_RHP'],   row['pfdr_between_x_RHP'],   row['eta2p_between_x_RHP']))
        L(SEP)
        L(fmt('COV: Mean recall rate',        row['t_recall_between'],  row['p_recall_between'],  row['pfdr_recall_between'],  row['eta2p_recall_between']))
        L(fmt('COV: Mean nav time (seconds)', row['t_nav_between'],     row['p_nav_between'],     row['pfdr_nav_between'],     row['eta2p_nav_between']))  # <-- label updated
    else:
        L(f"\n  BETWEEN-SUBJECT  *** {row['note_between']} ***")

    status = row['note_within']
    if status.startswith('ok'):
        L(f"\n  WITHIN-SUBJECT (LMM)  n_obs={int(row['n_within'])}  [{status}]")
        L(HDR); L(SEP)
        L(fmt('Neural main effect',           row['t_within'],           row['p_within'],           row['pfdr_within'],           row['eta2p_within'],           row['d_within']))
        L(fmt('Clustering type (T vs SP)',     row['t_clustT_within'],    row['p_clustT_within'],    row['pfdr_clustT_within'],    row['eta2p_clustT_within']))
        L(fmt('Region (RHP vs LHP)',           row['t_region_within'],    row['p_region_within'],    row['pfdr_region_within'],    row['eta2p_region_within']))
        L(fmt('Frequency (freq_hz)',           row['t_freq_within'],      row['p_freq_within'],      row['pfdr_freq_within'],      row['eta2p_freq_within']))
        L(fmt('Neural × clustering_type',      row['t_within_x_clustT'],  row['p_within_x_clustT'],  row['pfdr_within_x_clustT'],  row['eta2p_within_x_clustT']))
        L(fmt('Neural × region',               row['t_within_x_RHP'],     row['p_within_x_RHP'],     row['pfdr_within_x_RHP'],     row['eta2p_within_x_RHP']))
        L(fmt('Neural × freq_hz',              row['t_within_x_freq'],    row['p_within_x_freq'],    row['pfdr_within_x_freq'],    row['eta2p_within_x_freq']))
        L(SEP)
        L(fmt('COV: Trial recall rate',        row['t_recall_within'],    row['p_recall_within'],    row['pfdr_recall_within'],    row['eta2p_recall_within']))
        L(fmt('COV: Trial nav time (seconds)', row['t_nav_within'],       row['p_nav_within'],       row['pfdr_nav_within'],       row['eta2p_nav_within']))  # <-- label updated
    else:
        L(f"\n  WITHIN-SUBJECT  *** {status} ***")

# FDR summary
L(f"\n\n{'=' * 110}")
L("FDR CORRECTION SUMMARY")
L(f"{'=' * 110}")

between_effects = [
    ('pfdr_between',          'Neural main'),
    ('pfdr_between_x_clustT', 'Neural × clust_T'),
    ('pfdr_between_x_RHP',    'Neural × region'),
    ('pfdr_clustT_between',   'Clust type main'),
    ('pfdr_region_between',   'Region main'),
    ('pfdr_recall_between',   'COV: recall rate'),
    ('pfdr_nav_between',      'COV: nav time (s)'),
]
within_effects = [
    ('pfdr_within',           'Neural main'),
    ('pfdr_within_x_clustT',  'Neural × clust_T'),
    ('pfdr_within_x_RHP',     'Neural × region'),
    ('pfdr_within_x_freq',    'Neural × freq_hz'),
    ('pfdr_clustT_within',    'Clust type main'),
    ('pfdr_region_within',    'Region main'),
    ('pfdr_freq_within',      'Freq main'),
    ('pfdr_recall_within',    'COV: recall rate'),
    ('pfdr_nav_within',       'COV: nav time (s)'),
]

for level, effects in [('Between-subject', between_effects),
                        ('Within-subject',  within_effects)]:
    L(f"\n{level} — effects surviving FDR correction:")
    found = False
    for _, row in results.iterrows():
        for pc, label in effects:
            p = row.get(pc, np.nan)
            if pd.notna(p) and p < 0.05:
                found = True
                tag = 'OSC' if row['component'] == 'osc' else 'FRAC'
                L(f"  [{row['phase'].upper()} | {tag}]  {label:<35} p_fdr={p:.3f}")
    if not found:
        L("  None.")

L(f"\n{'=' * 110}")
L("FILES SAVED:")
L("  gamma_lmm_results.csv              — full results with FDR p-values and effect sizes")
L("  gamma_lmm_summary.csv              — FDR-significant effects only")
L("  gamma_neural_x_clustering_type.png — Figure: Neural × clustering_type (2×2 panel)")
L("=" * 110)

report = '\n'.join(lines)
print(report)
with open('gamma_lmm_report.txt', 'w') as f:
    f.write(report)

sig_fdr_cols = [c for c in results.columns if c.startswith('sig_fdr_')]
results[results[sig_fdr_cols].any(axis=1)].to_csv('gamma_lmm_summary.csv', index=False)
print("\nDone.")

trial_recall_rate NaNs:       0
trial_nav_time_seconds NaNs:  0
Figure saved: gamma_neural_x_clustering_type.png
GAMMA-BAND iEEG LMM — Between- and Within-Subject Effects on Clustering  [+ COVARIATES]
  Covariates: recall rate (mean/trial-within) and navigation time in SECONDS (mean/trial-within)
  Factors   : clustering_type (SP=0/T=1), region (LHP=0/RHP=1)
  Sig       : * uncorrected p<.05  | † FDR-corrected p<.05

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  PHASE: ENCODING   COMPONENT: OSCILLATORY (OSC)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  BETWEEN-SUBJECT (Level-2 OLS)  n_subjects=36  n_obs=108
  Effect                                                t     p_unc     p_fdr      η²p         d  sig
  ────────────────────────────────────────────────────────────────────────────────────────────────────
  Neural main effect                        

In [37]:
"""
Gamma-band iEEG LMM Analysis — Averaged over Gamma Band
==========================================================
Changes from previous version:
  - Gamma frequencies (>40 Hz) averaged BEFORE analysis
  - freq_hz removed from within-subject model entirely
  - Navigation time in SECONDS (outliers >180s excluded)

Model structure
---------------
BETWEEN-SUBJECT (Level-2 OLS):
  clustering_score_j ~ neural_between_j + clust_T + region_RHP
                     + mean_recall_rate_b + mean_nav_time_b
                     + neural_between_j:clust_T
                     + neural_between_j:region_RHP

WITHIN-SUBJECT (LMM):
  clustering_score_ij ~ neural_within_ij + clust_T + region_RHP
                      + trial_recall_rate_within + trial_nav_time_seconds_within
                      + neural_within_ij:clust_T
                      + neural_within_ij:region_RHP
                      + (1 + neural_within | subject)
                      + (1 | subject:session)
"""

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# LOAD & FILTER TO GAMMA — AVERAGE OVER FREQUENCIES FIRST
# ============================================================================

gamma = df[df['freq_hz'] >= 40].copy()
gamma = gamma[gamma['SP_clustering_from_prev'].notna()].copy()

# Average over all gamma frequencies per word × region × subject × session × trial
# This collapses freq_hz out of the dataframe entirely
gamma_avg = (
    gamma.groupby(['subject', 'session', 'trial', 'region',
                   'SP_clustering_from_prev', 'T_clustering_from_prev'])
    [['encoding_osc_ssl', 'encoding_frac_ssl',
      'retrieval_osc_ssl', 'retrieval_frac_ssl']]
    .mean()
    .reset_index()
)

# ============================================================================
# RESHAPE: clustering_type as factor
# ============================================================================

sp = gamma_avg.copy()
sp['clustering_score'] = sp['SP_clustering_from_prev']
sp['clustering_type']  = 'SP'

t = gamma_avg.copy()
t['clustering_score'] = t['T_clustering_from_prev']
t['clustering_type']  = 'T'

long = pd.concat([sp, t], ignore_index=True)
long['clust_T']    = (long['clustering_type'] == 'T').astype(int)
long['region_RHP'] = (long['region'] == 'RHP').astype(int)
long['sess_id']    = long['subject'].astype(str) + '_' + long['session'].astype(str)

# ============================================================================
# RE-MERGE BEHAVIORAL COVARIATES
# ============================================================================

# Trial-level recall rate
long = long.merge(
    recall_df[['subject', 'session', 'trial', 'recall_rate']]
              .rename(columns={'recall_rate': 'trial_recall_rate'}),
    on=['subject', 'session', 'trial'],
    how='left'
)

# Trial-level navigation time in SECONDS (outliers >180s excluded)
nav_trial = (
    nav_df[nav_df['navigation_time_seconds'] <= 180]
    .groupby(['subject', 'session', 'trial'])
    ['navigation_time_seconds']
    .mean()
    .reset_index()
    .rename(columns={'navigation_time_seconds': 'trial_nav_time_seconds'})
)
long = long.merge(nav_trial, on=['subject', 'session', 'trial'], how='left')

print(f"trial_recall_rate NaNs:       {long['trial_recall_rate'].isna().sum()}")
print(f"trial_nav_time_seconds NaNs:  {long['trial_nav_time_seconds'].isna().sum()}")

# ============================================================================
# BETWEEN / WITHIN DECOMPOSITION — neural predictors
# Computed on gamma_avg (pre-stack) to avoid duplication from SP/T stacking
# ============================================================================

for col in ['encoding_osc_ssl', 'encoding_frac_ssl',
            'retrieval_osc_ssl', 'retrieval_frac_ssl']:
    subj_mean_map          = gamma_avg.groupby('subject')[col].mean()
    long[col + '_between'] = long['subject'].map(subj_mean_map)
    long[col + '_within']  = long[col] - long[col + '_between']

# ============================================================================
# BETWEEN / WITHIN DECOMPOSITION — behavioral covariates
# ============================================================================

gamma_long = long[long['clust_T'] == 0].copy()

for cov in ['trial_recall_rate', 'trial_nav_time_seconds']:
    subj_mean_map          = gamma_long.groupby('subject')[cov].mean()
    long[cov + '_between'] = long['subject'].map(subj_mean_map)
    long[cov + '_within']  = long[cov] - long[cov + '_between']

long['mean_recall_rate_b'] = long['trial_recall_rate_between']
long['mean_nav_time_b']    = long['trial_nav_time_seconds_between']

# ============================================================================
# EFFECT SIZE HELPERS
# ============================================================================

def partial_eta_sq(t_val, df_resid):
    if pd.isna(t_val) or pd.isna(df_resid):
        return np.nan
    return t_val**2 / (t_val**2 + df_resid)

def cohens_d_from_t(t_val, n):
    if pd.isna(t_val) or pd.isna(n) or n == 0:
        return np.nan
    return t_val / np.sqrt(n)

# ============================================================================
# MODELS
# ============================================================================

PHASES = ['encoding', 'retrieval']
COMPS  = ['osc', 'frac']

results_rows = []

for phase in PHASES:
    for comp in COMPS:

        col_base    = f'{phase}_{comp}_ssl'
        col_between = col_base + '_between'
        col_within  = col_base + '_within'

        # ── BETWEEN-SUBJECT MODEL (OLS) ───────────────────────────────────
        between_data = (
            long.groupby(['subject', 'clust_T', 'region_RHP'])
            [[col_between, 'clustering_score',
              'mean_recall_rate_b', 'mean_nav_time_b']]
            .mean()
            .reset_index()
            .dropna()
        )

        try:
            formula_b = (
                f'clustering_score ~ {col_between} + clust_T + region_RHP '
                f'+ mean_recall_rate_b + mean_nav_time_b '
                f'+ {col_between}:clust_T '
                f'+ {col_between}:region_RHP'
            )
            res_b      = smf.ols(formula_b, data=between_data).fit()
            df_resid_b = res_b.df_resid
            n_subj     = between_data['subject'].nunique()

            def gb(key):
                t = res_b.tvalues.get(key, np.nan)
                p = res_b.pvalues.get(key, np.nan)
                e = partial_eta_sq(t, df_resid_b)
                return t, p, e

            t_b,         p_b,         e_b         = gb(col_between)
            t_b_xclust,  p_b_xclust,  e_b_xclust  = gb(f'{col_between}:clust_T')
            t_b_xregion, p_b_xregion, e_b_xregion = gb(f'{col_between}:region_RHP')
            t_clust_b,   p_clust_b,   e_clust_b   = gb('clust_T')
            t_region_b,  p_region_b,  e_region_b  = gb('region_RHP')
            t_recall_b,  p_recall_b,  e_recall_b  = gb('mean_recall_rate_b')
            t_nav_b,     p_nav_b,     e_nav_b     = gb('mean_nav_time_b')
            d_b    = cohens_d_from_t(t_b, n_subj)
            n_b    = int(res_b.nobs)
            note_b = 'ok'

        except Exception as e:
            (t_b, p_b, e_b, t_b_xclust, p_b_xclust, e_b_xclust,
             t_b_xregion, p_b_xregion, e_b_xregion,
             t_clust_b, p_clust_b, e_clust_b,
             t_region_b, p_region_b, e_region_b,
             t_recall_b, p_recall_b, e_recall_b,
             t_nav_b, p_nav_b, e_nav_b,
             d_b, n_b, n_subj) = [np.nan] * 23
            note_b = f'failed: {e}'

        # ── WITHIN-SUBJECT MODEL (LMM) — no freq_hz ──────────────────────
        within_data = long[[
            'subject', 'sess_id', 'clustering_score',
            col_within, 'clust_T', 'region_RHP',
            'trial_recall_rate_within', 'trial_nav_time_seconds_within'
        ]].dropna().copy()
        within_data = within_data.rename(columns={col_within: 'neural_within'})

        try:
            formula_w = (
                'clustering_score ~ neural_within + clust_T + region_RHP '
                '+ trial_recall_rate_within + trial_nav_time_seconds_within '
                '+ neural_within:clust_T '
                '+ neural_within:region_RHP'
            )
            md = smf.mixedlm(
                formula_w,
                data=within_data,
                groups=within_data['subject'],
                re_formula='~neural_within',
                vc_formula={'sess_id': '0 + C(sess_id)'}
            )
            res_w      = md.fit(reml=True, method='powell')
            df_resid_w = res_w.df_resid

            def gw(key):
                t = res_w.tvalues.get(key, np.nan)
                p = res_w.pvalues.get(key, np.nan)
                e = partial_eta_sq(t, df_resid_w)
                return t, p, e

            t_w,         p_w,         e_w         = gw('neural_within')
            t_w_xclust,  p_w_xclust,  e_w_xclust  = gw('neural_within:clust_T')
            t_w_xregion, p_w_xregion, e_w_xregion = gw('neural_within:region_RHP')
            t_clust_w,   p_clust_w,   e_clust_w   = gw('clust_T')
            t_region_w,  p_region_w,  e_region_w  = gw('region_RHP')
            t_recall_w,  p_recall_w,  e_recall_w  = gw('trial_recall_rate_within')
            t_nav_w,     p_nav_w,     e_nav_w     = gw('trial_nav_time_seconds_within')
            d_w    = cohens_d_from_t(t_w, within_data['subject'].nunique())
            n_w    = int(res_w.nobs)
            conv_w = res_w.converged
            note_w = 'ok' if conv_w else 'ok (convergence warning)'

        except Exception as e:
            (t_w, p_w, e_w, t_w_xclust, p_w_xclust, e_w_xclust,
             t_w_xregion, p_w_xregion, e_w_xregion,
             t_clust_w, p_clust_w, e_clust_w,
             t_region_w, p_region_w, e_region_w,
             t_recall_w, p_recall_w, e_recall_w,
             t_nav_w, p_nav_w, e_nav_w,
             d_w, n_w) = [np.nan] * 21
            note_w = f'failed: {e}'

        results_rows.append({
            'phase': phase, 'component': comp,
            # Between — neural
            't_between':           t_b,          'p_between':           p_b,          'eta2p_between':           e_b,
            't_between_x_clustT':  t_b_xclust,   'p_between_x_clustT':  p_b_xclust,   'eta2p_between_x_clustT':  e_b_xclust,
            't_between_x_RHP':     t_b_xregion,  'p_between_x_RHP':     p_b_xregion,  'eta2p_between_x_RHP':     e_b_xregion,
            't_clustT_between':    t_clust_b,    'p_clustT_between':    p_clust_b,    'eta2p_clustT_between':    e_clust_b,
            't_region_between':    t_region_b,   'p_region_between':    p_region_b,   'eta2p_region_between':    e_region_b,
            # Between — covariates
            't_recall_between':    t_recall_b,   'p_recall_between':    p_recall_b,   'eta2p_recall_between':    e_recall_b,
            't_nav_between':       t_nav_b,      'p_nav_between':       p_nav_b,      'eta2p_nav_between':       e_nav_b,
            'd_between': d_b, 'n_between': n_b, 'n_subjects': n_subj, 'note_between': note_b,
            # Within — neural
            't_within':            t_w,          'p_within':            p_w,          'eta2p_within':            e_w,
            't_within_x_clustT':   t_w_xclust,   'p_within_x_clustT':   p_w_xclust,   'eta2p_within_x_clustT':   e_w_xclust,
            't_within_x_RHP':      t_w_xregion,  'p_within_x_RHP':      p_w_xregion,  'eta2p_within_x_RHP':      e_w_xregion,
            't_clustT_within':     t_clust_w,    'p_clustT_within':     p_clust_w,    'eta2p_clustT_within':     e_clust_w,
            't_region_within':     t_region_w,   'p_region_within':     p_region_w,   'eta2p_region_within':     e_region_w,
            # Within — covariates
            't_recall_within':     t_recall_w,   'p_recall_within':     p_recall_w,   'eta2p_recall_within':     e_recall_w,
            't_nav_within':        t_nav_w,      'p_nav_within':        p_nav_w,      'eta2p_nav_within':        e_nav_w,
            'd_within': d_w, 'n_within': n_w, 'note_within': note_w,
        })

results = pd.DataFrame(results_rows)

# ============================================================================
# FDR CORRECTION (Benjamini-Hochberg)
# ============================================================================

def apply_fdr(results, p_cols, suffix):
    all_p = []
    for pc in p_cols:
        all_p.extend(results[pc].tolist())
    all_p = np.array(all_p, dtype=float)
    valid_mask = ~np.isnan(all_p)
    fdr_p = np.full_like(all_p, np.nan)
    if valid_mask.sum() > 0:
        _, p_corrected, _, _ = multipletests(all_p[valid_mask], method='fdr_bh')
        fdr_p[valid_mask] = p_corrected
    idx = 0
    for pc in p_cols:
        n = len(results)
        col_fdr = pc.replace('p_', 'pfdr_')
        col_sig = pc.replace('p_', 'sig_fdr_')
        results[col_fdr] = fdr_p[idx:idx+n]
        results[col_sig] = results[col_fdr].apply(
            lambda x: x < 0.05 if pd.notna(x) else False)
        idx += n
    return results

between_p_cols = ['p_between', 'p_between_x_clustT', 'p_between_x_RHP',
                  'p_clustT_between', 'p_region_between',
                  'p_recall_between', 'p_nav_between']

within_p_cols  = ['p_within', 'p_within_x_clustT', 'p_within_x_RHP',
                  'p_clustT_within', 'p_region_within',
                  'p_recall_within', 'p_nav_within']

results = apply_fdr(results, between_p_cols, 'between')
results = apply_fdr(results, within_p_cols,  'within')

for pc in between_p_cols + within_p_cols:
    results['sig_unc_' + pc[2:]] = results[pc].apply(
        lambda x: x < 0.05 if pd.notna(x) else False)

# ============================================================================
# SAVE
# ============================================================================

results.to_csv('gamma_lmm_results.csv', index=False)

# ============================================================================
# VISUALIZATION
# ============================================================================

def get_between_plot_data(long, col_between, phase, comp):
    data = (
        long.groupby(['subject', 'clust_T'])
        [[col_between, 'clustering_score']]
        .mean()
        .reset_index()
        .dropna()
    )
    return data[data['clust_T'] == 0].copy(), data[data['clust_T'] == 1].copy()

COL_SP = '#2166AC'; COL_T = '#D6604D'; COL_GRID = '#E8E8E8'

fig = plt.figure(figsize=(12, 10))
fig.patch.set_facecolor('white')
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.38)
panel_labels = [['A', 'B'], ['C', 'D']]

for pi, phase in enumerate(PHASES):
    for ci, comp in enumerate(COMPS):
        ax = fig.add_subplot(gs[pi, ci])
        col_base    = f'{phase}_{comp}_ssl'
        col_between = col_base + '_between'
        sp_data, t_data = get_between_plot_data(long, col_between, phase, comp)

        ax.scatter(sp_data[col_between], sp_data['clustering_score'],
                   color=COL_SP, s=60, zorder=3, alpha=0.85,
                   label='SP clustering', edgecolors='white', linewidths=0.5)
        ax.scatter(t_data[col_between],  t_data['clustering_score'],
                   color=COL_T,  s=60, zorder=3, alpha=0.85,
                   label='T clustering',  edgecolors='white', linewidths=0.5)

        for dat, col in [(sp_data, COL_SP), (t_data, COL_T)]:
            x = dat[col_between].values; y = dat['clustering_score'].values
            if len(x) > 1:
                m, b = np.polyfit(x, y, 1)
                xr = np.linspace(x.min(), x.max(), 100)
                ax.plot(xr, m*xr + b, color=col, linewidth=2, alpha=0.9)

        row = results[(results['phase']==phase) & (results['component']==comp)].iloc[0]

        def pstr(p):
            if pd.isna(p): return 'n.s.'
            if p < 0.001:  return 'p<.001'
            if p < 0.05:   return f'p={p:.3f}'
            return f'p={p:.2f} n.s.'

        ann = (f"Neural main: t={row['t_between']:+.2f}, {pstr(row['pfdr_between'])}, d={row['d_between']:.2f}\n"
               f"× Clust type: t={row['t_between_x_clustT']:+.2f}, {pstr(row['pfdr_between_x_clustT'])}")
        ax.text(0.03, 0.97, ann, transform=ax.transAxes, fontsize=7.5, va='top', ha='left',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='#F7F7F7', edgecolor='#CCCCCC', alpha=0.9))
        ax.text(-0.12, 1.05, panel_labels[pi][ci], transform=ax.transAxes,
                fontsize=14, fontweight='bold', va='top')

        comp_label = 'Oscillatory' if comp == 'osc' else 'Aperiodic (Fractal)'
        ax.set_title(f'{phase.capitalize()} — {comp_label} γ', fontsize=11, fontweight='bold', pad=8)
        ax.set_xlabel('Gamma power (subject mean, SSL)', fontsize=9)
        ax.set_ylabel('Clustering score', fontsize=9)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        ax.grid(True, color=COL_GRID, linewidth=0.7, zorder=0)
        ax.tick_params(labelsize=8)
        if pi == 0 and ci == 0:
            ax.legend(fontsize=8.5, framealpha=0.9, loc='lower right')

fig.suptitle('Between-subject gamma power predicts clustering\n'
             'across encoding and retrieval phases (iEEG, gamma averaged >40Hz)',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig('gamma_lmm_avgband_neural_x_clustering_type.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print("Figure saved: gamma_lmm_avgband_neural_x_clustering_type.png")

# ============================================================================
# PRINT REPORT
# ============================================================================

lines = []; L = lines.append

L("=" * 110)
L("GAMMA-BAND iEEG LMM — Gamma Averaged (>40Hz) — Between- and Within-Subject Effects")
L("  Covariates: recall rate (mean/trial-within) and navigation time in SECONDS")
L("  Factors   : clustering_type (SP=0/T=1), region (LHP=0/RHP=1)")
L("  Note      : freq_hz removed — gamma averaged before analysis")
L("  Sig       : * uncorrected p<.05  | † FDR-corrected p<.05")
L("=" * 110)

def fmt(label, t, p_unc, p_fdr, eta2, d=None):
    sig_unc = '*' if (pd.notna(p_unc) and p_unc < 0.05) else ''
    sig_fdr = '†' if (pd.notna(p_fdr) and p_fdr < 0.05) else ''
    ts  = f'{t:+.3f}'    if pd.notna(t)     else '    NaN'
    pu  = f'{p_unc:.3f}' if pd.notna(p_unc) else '    NaN'
    pf  = f'{p_fdr:.3f}' if pd.notna(p_fdr) else '    NaN'
    e2  = f'{eta2:.3f}'  if pd.notna(eta2)  else '    NaN'
    d_s = f'{d:+.3f}'    if (d is not None and pd.notna(d)) else '      —'
    return f"  {label:<45} {ts:>9} {pu:>9} {pf:>9} {e2:>8} {d_s:>9}  {sig_unc}{sig_fdr}"

HDR = f"  {'Effect':<45} {'t':>9} {'p_unc':>9} {'p_fdr':>9} {'η²p':>8} {'d':>9}  sig"
SEP = f"  {'─' * 100}"

for _, row in results.iterrows():
    phase = row['phase']; comp = row['component']
    L(f"\n{'━' * 110}")
    L(f"  PHASE: {phase.upper()}   COMPONENT: {'OSCILLATORY (OSC)' if comp=='osc' else 'APERIODIC/FRACTAL (FRAC)'}")
    L(f"{'━' * 110}")

    if row['note_between'] == 'ok':
        L(f"\n  BETWEEN-SUBJECT (Level-2 OLS)  n_subjects={int(row['n_subjects'])}  n_obs={int(row['n_between'])}")
        L(HDR); L(SEP)
        L(fmt('Neural main effect',          row['t_between'],          row['p_between'],          row['pfdr_between'],          row['eta2p_between'],         row['d_between']))
        L(fmt('Clustering type (T vs SP)',    row['t_clustT_between'],   row['p_clustT_between'],   row['pfdr_clustT_between'],   row['eta2p_clustT_between']))
        L(fmt('Region (RHP vs LHP)',          row['t_region_between'],   row['p_region_between'],   row['pfdr_region_between'],   row['eta2p_region_between']))
        L(fmt('Neural × clustering_type',     row['t_between_x_clustT'], row['p_between_x_clustT'], row['pfdr_between_x_clustT'], row['eta2p_between_x_clustT']))
        L(fmt('Neural × region',              row['t_between_x_RHP'],    row['p_between_x_RHP'],    row['pfdr_between_x_RHP'],    row['eta2p_between_x_RHP']))
        L(SEP)
        L(fmt('COV: Mean recall rate',        row['t_recall_between'],   row['p_recall_between'],   row['pfdr_recall_between'],   row['eta2p_recall_between']))
        L(fmt('COV: Mean nav time (seconds)', row['t_nav_between'],      row['p_nav_between'],      row['pfdr_nav_between'],      row['eta2p_nav_between']))
    else:
        L(f"\n  BETWEEN-SUBJECT  *** {row['note_between']} ***")

    status = row['note_within']
    if status.startswith('ok'):
        L(f"\n  WITHIN-SUBJECT (LMM)  n_obs={int(row['n_within'])}  [{status}]")
        L(HDR); L(SEP)
        L(fmt('Neural main effect',           row['t_within'],           row['p_within'],           row['pfdr_within'],           row['eta2p_within'],          row['d_within']))
        L(fmt('Clustering type (T vs SP)',     row['t_clustT_within'],    row['p_clustT_within'],    row['pfdr_clustT_within'],    row['eta2p_clustT_within']))
        L(fmt('Region (RHP vs LHP)',           row['t_region_within'],    row['p_region_within'],    row['pfdr_region_within'],    row['eta2p_region_within']))
        L(fmt('Neural × clustering_type',      row['t_within_x_clustT'],  row['p_within_x_clustT'],  row['pfdr_within_x_clustT'],  row['eta2p_within_x_clustT']))
        L(fmt('Neural × region',               row['t_within_x_RHP'],     row['p_within_x_RHP'],     row['pfdr_within_x_RHP'],     row['eta2p_within_x_RHP']))
        L(SEP)
        L(fmt('COV: Trial recall rate',        row['t_recall_within'],    row['p_recall_within'],    row['pfdr_recall_within'],    row['eta2p_recall_within']))
        L(fmt('COV: Trial nav time (seconds)', row['t_nav_within'],       row['p_nav_within'],       row['pfdr_nav_within'],       row['eta2p_nav_within']))
    else:
        L(f"\n  WITHIN-SUBJECT  *** {status} ***")

# FDR summary
L(f"\n\n{'=' * 110}")
L("FDR CORRECTION SUMMARY")
L(f"{'=' * 110}")

between_effects = [
    ('pfdr_between',          'Neural main'),
    ('pfdr_between_x_clustT', 'Neural × clust_T'),
    ('pfdr_between_x_RHP',    'Neural × region'),
    ('pfdr_clustT_between',   'Clust type main'),
    ('pfdr_region_between',   'Region main'),
    ('pfdr_recall_between',   'COV: recall rate'),
    ('pfdr_nav_between',      'COV: nav time (s)'),
]
within_effects = [
    ('pfdr_within',           'Neural main'),
    ('pfdr_within_x_clustT',  'Neural × clust_T'),
    ('pfdr_within_x_RHP',     'Neural × region'),
    ('pfdr_clustT_within',    'Clust type main'),
    ('pfdr_region_within',    'Region main'),
    ('pfdr_recall_within',    'COV: recall rate'),
    ('pfdr_nav_within',       'COV: nav time (s)'),
]

for level, effects in [('Between-subject', between_effects),
                        ('Within-subject',  within_effects)]:
    L(f"\n{level} — effects surviving FDR correction:")
    found = False
    for _, row in results.iterrows():
        for pc, label in effects:
            p = row.get(pc, np.nan)
            if pd.notna(p) and p < 0.05:
                found = True
                tag = 'OSC' if row['component'] == 'osc' else 'FRAC'
                L(f"  [{row['phase'].upper()} | {tag}]  {label:<35} p_fdr={p:.3f}")
    if not found:
        L("  None.")

L(f"\n{'=' * 110}")
L("FILES SAVED:")
L("  gamma_lmm_results.csv                      — full results with FDR p-values and effect sizes")
L("  gamma_lmm_summary.csv                      — FDR-significant effects only")
L("  gamma_lmm_avgband_neural_x_clustering_type.png — Figure")
L("=" * 110)

report = '\n'.join(lines)
print(report)
with open('gamma_lmm_report.txt', 'w') as f:
    f.write(report)

sig_fdr_cols = [c for c in results.columns if c.startswith('sig_fdr_')]
results[results[sig_fdr_cols].any(axis=1)].to_csv('gamma_lmm_summary.csv', index=False)
print("\nDone.")

trial_recall_rate NaNs:       0
trial_nav_time_seconds NaNs:  0
Figure saved: gamma_lmm_avgband_neural_x_clustering_type.png
GAMMA-BAND iEEG LMM — Gamma Averaged (>40Hz) — Between- and Within-Subject Effects
  Covariates: recall rate (mean/trial-within) and navigation time in SECONDS
  Factors   : clustering_type (SP=0/T=1), region (LHP=0/RHP=1)
  Note      : freq_hz removed — gamma averaged before analysis
  Sig       : * uncorrected p<.05  | † FDR-corrected p<.05

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  PHASE: ENCODING   COMPONENT: OSCILLATORY (OSC)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  BETWEEN-SUBJECT (Level-2 OLS)  n_subjects=36  n_obs=108
  Effect                                                t     p_unc     p_fdr      η²p         d  sig
  ───────────────────────────────────────────────────────────────────────────────────────────────

In [12]:
# ============================================================================
# MERGE BEHAVIORAL COVARIATES IF NOT ALREADY PRESENT
# ============================================================================

if 'mean_recall_rate' not in long.columns:
    long = long.merge(
        subj_covs[['subject', 'mean_recall_rate', 'mean_nav_time_samples']],
        on='subject',
        how='left'
    )
    print(f"mean_recall_rate NaNs:      {long['mean_recall_rate'].isna().sum()}")
    print(f"mean_nav_time_samples NaNs: {long['mean_nav_time_samples'].isna().sum()}")
else:
    print("Behavioral subject-level covariates already present ✓")

if 'trial_recall_rate' not in long.columns:
    long = long.merge(
        recall_df[['subject', 'session', 'trial', 'recall_rate']]
                  .rename(columns={'recall_rate': 'trial_recall_rate'}),
        on=['subject', 'session', 'trial'],
        how='left'
    )
    nav_trial = (
        nav_df.groupby(['subject', 'session', 'trial'])
        ['navigation_time_samples'].mean()
        .reset_index()
        .rename(columns={'navigation_time_samples': 'trial_nav_time_samples'})
    )
    long = long.merge(nav_trial, on=['subject', 'session', 'trial'], how='left')
    print(f"trial_recall_rate NaNs:        {long['trial_recall_rate'].isna().sum()}")
    print(f"trial_nav_time_samples NaNs:   {long['trial_nav_time_samples'].isna().sum()}")
else:
    print("Behavioral trial-level covariates already present ✓")

mean_recall_rate NaNs:      1716
mean_nav_time_samples NaNs: 1716
Behavioral trial-level covariates already present ✓


In [13]:
print(f"Total subjects in long:              {long['subject'].nunique()}")
print(f"Subjects with mean_recall_rate:      {long[long['mean_recall_rate'].notna()]['subject'].nunique()}")
print(f"Subjects with mean_nav_time_samples: {long[long['mean_nav_time_samples'].notna()]['subject'].nunique()}")
print(f"\nSubjects WITH behavioral data:")
print(sorted(long[long['mean_recall_rate'].notna()]['subject'].unique()))
print(f"\nSubjects WITHOUT behavioral data:")
print(sorted(long[long['mean_recall_rate'].isna()]['subject'].unique()))

Total subjects in long:              26
Subjects with mean_recall_rate:      23
Subjects with mean_nav_time_samples: 23

Subjects WITH behavioral data:
['R1494D', 'R1501J', 'R1502D', 'R1503E', 'R1504E', 'R1509E', 'R1521E', 'R1523E', 'R1532T', 'R1536J', 'R1537T', 'R1538E', 'R1542J', 'R1543E', 'R1546E', 'R1551T', 'R1554T', 'R1560T', 'R1561E', 'R1567T', 'R1570T', 'R1571T', 'R1572T']

Subjects WITHOUT behavioral data:
['R1620J', 'R1637T', 'R1651J']


In [19]:
"""
Visualization: Between-Subject Neural × Clustering-Type Interaction
====================================================================
Reproduces the between-subject aggregation from the OLS script, then plots:
  1. Scatter + regression lines: neural_between vs clustering_score,
     split by clust_T (SP=0, T=1) — one panel per phase × component (2×2 grid)
  2. Bar chart: mean clustering score by clust_T — one panel per phase × component
  3. Scatter: mean recall rate vs clustering score (covariate effect)

All p-values / effect sizes are taken directly from `gamma_between_results.csv`
(produced by the OLS script). If that file is absent the interaction lines are
still drawn without annotation.

Assumes `long` DataFrame is in scope (output of Script 1), containing:
  encoding_osc_ssl_between, encoding_frac_ssl_between,
  retrieval_osc_ssl_between, retrieval_frac_ssl_between,
  clustering_score, clust_T, region_RHP,
  mean_recall_rate, mean_nav_time_samples, subject
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── palette ──────────────────────────────────────────────────────────────────
C_SP  = '#4C72B0'   # blue  → SP clustering (clust_T = 0)
C_T   = '#DD8452'   # orange → T  clustering (clust_T = 1)
C_REC = '#55A868'   # green  → recall-rate covariate panel
ALPHA_SC = 0.35     # scatter dot alpha

# ── load OLS results (optional — for annotation) ─────────────────────────────
try:
    ols = pd.read_csv('gamma_between_results.csv').set_index(['phase', 'component'])
    HAS_OLS = True
    print("gamma_between_results.csv loaded ✓")
except FileNotFoundError:
    HAS_OLS = False
    print("gamma_between_results.csv not found — panels drawn without p annotations")

# ── verify required columns ───────────────────────────────────────────────────
REQUIRED = [
    'encoding_osc_ssl_between',  'encoding_frac_ssl_between',
    'retrieval_osc_ssl_between', 'retrieval_frac_ssl_between',
    'clustering_score', 'clust_T', 'region_RHP',
    'mean_recall_rate', 'subject',
]
missing = [c for c in REQUIRED if c not in long.columns]
if missing:
    raise ValueError(f"Missing columns in `long`: {missing}")
print("All required columns present ✓")

# ── helpers ───────────────────────────────────────────────────────────────────

def reg_line(x, y, ax, color, lw=2.2, ls='-'):
    """Plot OLS regression line over data range."""
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return
    slope, intercept, r, p, _ = stats.linregress(x[mask], y[mask])
    xi = np.linspace(x[mask].min(), x[mask].max(), 200)
    ax.plot(xi, slope * xi + intercept, color=color, lw=lw, ls=ls, zorder=4)
    return slope, r, p


def p_star(p):
    if pd.isna(p): return ''
    if p < 0.001:  return '***'
    if p < 0.01:   return '**'
    if p < 0.05:   return '*'
    return 'n.s.'


def annotate_interaction(ax, row, col_neural):
    """Add FDR-corrected p-value annotation for Neural × clust_T."""
    if not HAS_OLS:
        return
    try:
        key = ('encoding' if 'encod' in col_neural else 'retrieval',
               'osc'      if 'osc'   in col_neural else 'frac')
        r   = ols.loc[key]
        p   = r['pfdr_between_x_clustT']
        t   = r['t_between_x_clustT']
        eta = r['eta2p_between_x_clustT']
        stars = p_star(p)
        txt = (f"Neural × ClustType\n"
               f"t={t:+.2f}, η²p={eta:.3f}\n"
               f"p_fdr={p:.3f} {stars}")
        ax.text(0.97, 0.97, txt, transform=ax.transAxes,
                ha='right', va='top', fontsize=7.5,
                bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.7, ec='#cccccc'))
    except Exception:
        pass

# ── between-subject aggregation (mirrors OLS script) ─────────────────────────

PHASES = ['encoding', 'retrieval']
COMPS  = ['osc', 'frac']
COMP_LABEL = {'osc': 'Oscillatory (OSC)', 'frac': 'Aperiodic (FRAC)'}
PHASE_LABEL = {'encoding': 'Encoding', 'retrieval': 'Retrieval'}

agg_data = {}
for phase in PHASES:
    for comp in COMPS:
        col = f'{phase}_{comp}_ssl_between'
        df = (
            long.groupby(['subject', 'clust_T', 'region_RHP'])
            [[col, 'clustering_score', 'mean_recall_rate']]
            .mean()
            .reset_index()
            .dropna()
        )
        agg_data[(phase, comp)] = (df, col)

# ════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — 2×4 grid: Neural × Clustering-Type scatter (main figure)
# Rows = phase (Encoding / Retrieval), Cols = component × SP/T (4 cols)
# ════════════════════════════════════════════════════════════════════════════
print("\nBuilding Figure 1: Neural × clust_T interaction scatters …")

fig1, axes1 = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)
fig1.suptitle(
    "Between-Subject: Neural Activity × Clustering Type → Clustering Score\n"
    "(aggregated to subject × clust_T × region means; FDR-corrected p shown)",
    fontsize=13, fontweight='bold', y=1.02
)

for ri, phase in enumerate(PHASES):
    for ci, comp in enumerate(COMPS):
        ax   = axes1[ri, ci]
        df, col = agg_data[(phase, comp)]

        sp = df[df['clust_T'] == 0]
        t  = df[df['clust_T'] == 1]

        # scatter
        ax.scatter(sp[col], sp['clustering_score'],
                   color=C_SP, alpha=ALPHA_SC, s=28, label='SP (clust_T=0)', zorder=3)
        ax.scatter(t[col],  t['clustering_score'],
                   color=C_T,  alpha=ALPHA_SC, s=28, label='T  (clust_T=1)', zorder=3)

        # regression lines
        reg_line(sp[col].values, sp['clustering_score'].values, ax, C_SP)
        reg_line(t[col].values,  t['clustering_score'].values,  ax, C_T)

        ax.set_xlabel(f'{col}\n(between-subject z-score)', fontsize=8)
        ax.set_ylabel('Clustering Score', fontsize=8)
        ax.set_title(f'{PHASE_LABEL[phase]} | {COMP_LABEL[comp]}',
                     fontsize=10, fontweight='bold')
        ax.tick_params(labelsize=8)
        ax.legend(fontsize=7.5, framealpha=0.6)
        ax.spines[['top', 'right']].set_visible(False)

        annotate_interaction(ax, None, col)

plt.savefig('fig1_neural_x_clustT_scatter.png', dpi=150, bbox_inches='tight')
print("  → fig1_neural_x_clustT_scatter.png saved")
plt.show()


# ════════════════════════════════════════════════════════════════════════════
# FIGURE 2 — 2×2 grid: Mean clustering score by clust_T (bar chart)
# ════════════════════════════════════════════════════════════════════════════
print("Building Figure 2: Mean clustering score by clust_T …")

fig2, axes2 = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)
fig2.suptitle(
    "Between-Subject: Mean Clustering Score by Clustering Type (SP vs T)\n"
    "(error bars = 95 % CI; FDR-corrected p for clust_T main effect shown)",
    fontsize=12, fontweight='bold'
)

for ri, phase in enumerate(PHASES):
    for ci, comp in enumerate(COMPS):
        ax   = axes2[ri, ci]
        df, col = agg_data[(phase, comp)]

        means = df.groupby('clust_T')['clustering_score'].mean()
        sems  = df.groupby('clust_T')['clustering_score'].sem()
        n_per = df.groupby('clust_T')['clustering_score'].count()
        ci95  = sems * 1.96

        labels = ['SP (0)', 'T (1)']
        colors = [C_SP, C_T]
        xs     = [0, 1]

        for x, lbl, col_c, m, e in zip(xs, labels, colors,
                                        means.values, ci95.values):
            ax.bar(x, m, color=col_c, alpha=0.75, width=0.5,
                   yerr=e, capsize=5, error_kw=dict(lw=1.5),
                   label=lbl, zorder=3)

        # individual subject means (jittered)
        for xi, ct in zip(xs, [0, 1]):
            subj_m = df[df['clust_T'] == ct].groupby('subject')['clustering_score'].mean()
            jitter = np.random.default_rng(42).uniform(-0.12, 0.12, len(subj_m))
            ax.scatter(xi + jitter, subj_m.values,
                       color='k', alpha=0.35, s=15, zorder=4)

        # p annotation for clust_T main effect
        if HAS_OLS:
            try:
                key = (phase, comp)
                r   = ols.loc[key]
                p   = r['pfdr_clustT_between']
                t_v = r['t_clustT_between']
                eta = r['eta2p_clustT_between']
                stars = p_star(p)
                ax.text(0.5, 0.97,
                        f"clust_T main: t={t_v:+.2f}, η²p={eta:.3f}\np_fdr={p:.3f} {stars}",
                        transform=ax.transAxes, ha='center', va='top', fontsize=8,
                        bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.7, ec='#cccccc'))
            except Exception:
                pass

        ax.set_xticks([0, 1]); ax.set_xticklabels(['SP (0)', 'T (1)'], fontsize=9)
        ax.set_xlabel('Clustering Type', fontsize=9)
        ax.set_ylabel('Mean Clustering Score', fontsize=9)
        ax.set_title(f'{PHASE_LABEL[phase]} | {COMP_LABEL[comp]}',
                     fontsize=10, fontweight='bold')
        ax.tick_params(labelsize=8)
        ax.spines[['top', 'right']].set_visible(False)

plt.savefig('fig2_clustT_bar.png', dpi=150, bbox_inches='tight')
print("  → fig2_clustT_bar.png saved")
plt.show()


# ════════════════════════════════════════════════════════════════════════════
# FIGURE 3 — 2×2 grid: Recall rate covariate
# ════════════════════════════════════════════════════════════════════════════
print("Building Figure 3: Recall rate covariate scatter …")

fig3, axes3 = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)
fig3.suptitle(
    "Between-Subject: Mean Recall Rate vs Clustering Score (Covariate)\n"
    "(collapsed across clust_T; FDR-corrected p for recall-rate covariate shown)",
    fontsize=12, fontweight='bold'
)

for ri, phase in enumerate(PHASES):
    for ci, comp in enumerate(COMPS):
        ax   = axes3[ri, ci]
        df, col = agg_data[(phase, comp)]

        # collapse to subject level for covariate plot
        subj_df = df.groupby('subject')[['mean_recall_rate', 'clustering_score']].mean()

        ax.scatter(subj_df['mean_recall_rate'], subj_df['clustering_score'],
                   color=C_REC, alpha=0.6, s=40, edgecolors='white', lw=0.4, zorder=3)
        reg_line(subj_df['mean_recall_rate'].values,
                 subj_df['clustering_score'].values, ax, C_REC)

        # annotation
        if HAS_OLS:
            try:
                key = (phase, comp)
                r   = ols.loc[key]
                p   = r['pfdr_recall_between']
                t_v = r['t_recall_between']
                eta = r['eta2p_recall_between']
                stars = p_star(p)
                ax.text(0.97, 0.97,
                        f"Recall-rate cov: t={t_v:+.2f}\nη²p={eta:.3f}, p_fdr={p:.3f} {stars}",
                        transform=ax.transAxes, ha='right', va='top', fontsize=8,
                        bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.7, ec='#cccccc'))
            except Exception:
                pass

        ax.set_xlabel('Mean Recall Rate (subject-level)', fontsize=9)
        ax.set_ylabel('Mean Clustering Score', fontsize=9)
        ax.set_title(f'{PHASE_LABEL[phase]} | {COMP_LABEL[comp]}',
                     fontsize=10, fontweight='bold')
        ax.tick_params(labelsize=8)
        ax.spines[['top', 'right']].set_visible(False)

plt.savefig('fig3_recall_covariate.png', dpi=150, bbox_inches='tight')
print("  → fig3_recall_covariate.png saved")
plt.show()


# ════════════════════════════════════════════════════════════════════════════
# FIGURE 4 — Summary heatmap: t-values across phase × component × effect
# ════════════════════════════════════════════════════════════════════════════
print("Building Figure 4: t-value heatmap summary …")

if HAS_OLS:
    ols_r = pd.read_csv('gamma_between_results.csv')

    effect_cols = {
        'Neural main':    ('t_between',          'pfdr_between'),
        'Neural × T':     ('t_between_x_clustT', 'pfdr_between_x_clustT'),
        'Neural × RHP':   ('t_between_x_RHP',    'pfdr_between_x_RHP'),
        'ClustT main':    ('t_clustT_between',    'pfdr_clustT_between'),
        'Region main':    ('t_region_between',    'pfdr_region_between'),
        'COV: recall':    ('t_recall_between',    'pfdr_recall_between'),
        'COV: nav time':  ('t_nav_between',       'pfdr_nav_between'),
    }

    row_labels = [f"{r['phase'].capitalize()} | {'OSC' if r['component']=='osc' else 'FRAC'}"
                  for _, r in ols_r.iterrows()]

    t_matrix   = np.zeros((len(ols_r), len(effect_cols)))
    sig_matrix = np.zeros((len(ols_r), len(effect_cols)), dtype=bool)

    for ci, (label, (tc, pc)) in enumerate(effect_cols.items()):
        t_matrix[:, ci]   = ols_r[tc].values
        sig_matrix[:, ci] = ols_r[pc].fillna(1).values < 0.05

    vmax = np.nanmax(np.abs(t_matrix))

    fig4, ax4 = plt.subplots(figsize=(11, 4), constrained_layout=True)
    im = ax4.imshow(t_matrix, aspect='auto', cmap='RdBu_r',
                    vmin=-vmax, vmax=vmax, interpolation='nearest')

    # significance asterisks
    for ri in range(t_matrix.shape[0]):
        for ci in range(t_matrix.shape[1]):
            if sig_matrix[ri, ci]:
                ax4.text(ci, ri, '*', ha='center', va='center',
                         fontsize=14, color='black', fontweight='bold')

    ax4.set_xticks(range(len(effect_cols)))
    ax4.set_xticklabels(list(effect_cols.keys()), rotation=35, ha='right', fontsize=9)
    ax4.set_yticks(range(len(ols_r)))
    ax4.set_yticklabels(row_labels, fontsize=9)
    ax4.set_title("t-values from Between-Subject OLS  (* = FDR p < .05)", fontsize=11, fontweight='bold')
    plt.colorbar(im, ax=ax4, label='t-value', shrink=0.8)

    plt.savefig('fig4_tvalue_heatmap.png', dpi=150, bbox_inches='tight')
    print("  → fig4_tvalue_heatmap.png saved")
    plt.show()
else:
    print("  Skipped (requires gamma_between_results.csv)")

# ════════════════════════════════════════════════════════════════════════════
# CONSOLE SUMMARY
# ════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("VISUALIZATION SUMMARY")
print("=" * 70)
print("Fig 1 — Neural × clust_T interaction scatters (2×2, scatter+regression)")
print("         SP=blue, T=orange; FDR-corrected p annotated per panel")
print("Fig 2 — Bar chart: mean clustering score by clustering type (SP vs T)")
print("         Individual subject means overlaid as jittered dots")
print("Fig 3 — Scatter: mean recall rate vs clustering score (covariate)")
print("Fig 4 — t-value heatmap: all OLS effects across phase × component")
print("         (* marks FDR-corrected p < .05)")
print("=" * 70)
print("\nAll figures saved as PNG in working directory.")
print("Done.")

gamma_between_results.csv loaded ✓
All required columns present ✓

Building Figure 1: Neural × clust_T interaction scatters …
  → fig1_neural_x_clustT_scatter.png saved
Building Figure 2: Mean clustering score by clust_T …
  → fig2_clustT_bar.png saved
Building Figure 3: Recall rate covariate scatter …
  → fig3_recall_covariate.png saved
Building Figure 4: t-value heatmap summary …
  → fig4_tvalue_heatmap.png saved

VISUALIZATION SUMMARY
Fig 1 — Neural × clust_T interaction scatters (2×2, scatter+regression)
         SP=blue, T=orange; FDR-corrected p annotated per panel
Fig 2 — Bar chart: mean clustering score by clustering type (SP vs T)
         Individual subject means overlaid as jittered dots
Fig 3 — Scatter: mean recall rate vs clustering score (covariate)
Fig 4 — t-value heatmap: all OLS effects across phase × component
         (* marks FDR-corrected p < .05)

All figures saved as PNG in working directory.
Done.


In [20]:
"""
Mediation Analysis: Gamma Power → Navigation Time → SP Clustering
==================================================================
Tests whether navigation performance (mean_nav_time_samples) mediates the
relationship between between-subject gamma power and SP clustering score.

Restricted to SP clustering only (clust_T == 0) — theoretically motivated
by the spatial memory argument; T clustering excluded by design.

Mediation paths
---------------
  Path a  : mean_nav_time_samples ~ gamma_between
  Path b  : SP_clustering ~ mean_nav_time_samples + gamma_between   (b = nav coeff)
  Path c  : SP_clustering ~ gamma_between                           (total effect)
  Path c' : SP_clustering ~ gamma_between + mean_nav_time_samples   (direct effect)
  Indirect: a × b, 95% CI via percentile bootstrap (N_BOOT iterations)

Run separately for each phase × component combination:
  encoding  × osc
  encoding  × frac
  retrieval × osc
  retrieval × frac

Figures produced
----------------
  fig_mediation_paths.png   — 2×4 path diagram grid (one per phase×comp)
  fig_mediation_bootstrap.png — bootstrap indirect-effect distributions

Assumptions
-----------
  `long` DataFrame in scope with columns:
    encoding_osc_ssl_between, encoding_frac_ssl_between,
    retrieval_osc_ssl_between, retrieval_frac_ssl_between,
    clustering_score, clust_T, mean_nav_time_samples, subject

Output CSV:  mediation_SP_results.csv
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import statsmodels.formula.api as smf
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ─── configuration ────────────────────────────────────────────────────────────
N_BOOT     = 5000
RANDOM_SEED = 42
ALPHA_LEVEL = 0.05

C_SIG   = '#2ca02c'   # green  — significant indirect
C_NSIG  = '#d62728'   # red    — non-significant indirect
C_PATH  = '#1f77b4'   # blue   — path arrows
C_GAMMA = '#9467bd'   # purple — gamma node
C_NAV   = '#8c564b'   # brown  — navigation node
C_CLUST = '#e377c2'   # pink   — clustering node

PHASES = ['encoding', 'retrieval']
COMPS  = ['osc', 'frac']
COMP_LABEL  = {'osc': 'Oscillatory (OSC)', 'frac': 'Aperiodic (FRAC)'}
PHASE_LABEL = {'encoding': 'Encoding', 'retrieval': 'Retrieval'}

# ─── verify columns ───────────────────────────────────────────────────────────
REQUIRED = [
    'encoding_osc_ssl_between',  'encoding_frac_ssl_between',
    'retrieval_osc_ssl_between', 'retrieval_frac_ssl_between',
    'clustering_score', 'clust_T', 'mean_nav_time_samples', 'subject',
]
missing = [c for c in REQUIRED if c not in long.columns]
if missing:
    raise ValueError(f"Missing columns in `long`: {missing}")
print("All required columns present ✓")

# ─── aggregate to subject-level (SP only) ─────────────────────────────────────

def make_subject_df(phase, comp):
    """
    Aggregate long → subject-level for SP clustering only.
    Returns DataFrame with one row per subject.
    """
    col = f'{phase}_{comp}_ssl_between'
    sp_long = long[long['clust_T'] == 0].copy()
    df = (
        sp_long.groupby('subject')
        [[col, 'clustering_score', 'mean_nav_time_samples']]
        .mean()
        .reset_index()
        .dropna()
        .rename(columns={col: 'gamma_between',
                         'clustering_score': 'SP_clustering'})
    )
    return df


# ─── single-sample mediation ──────────────────────────────────────────────────

def run_mediation(df):
    """
    Estimate paths a, b, c, c' and indirect effect via OLS.
    Returns dict of point estimates + SE + p-values.
    """
    # Path a: nav ~ gamma
    ma = smf.ols('mean_nav_time_samples ~ gamma_between', data=df).fit()
    a      = ma.params['gamma_between']
    a_se   = ma.bse['gamma_between']
    a_p    = ma.pvalues['gamma_between']

    # Path c: SP_clustering ~ gamma  (total)
    mc = smf.ols('SP_clustering ~ gamma_between', data=df).fit()
    c      = mc.params['gamma_between']
    c_se   = mc.bse['gamma_between']
    c_p    = mc.pvalues['gamma_between']

    # Paths b + c': SP_clustering ~ gamma + nav
    mb = smf.ols('SP_clustering ~ gamma_between + mean_nav_time_samples', data=df).fit()
    b      = mb.params['mean_nav_time_samples']
    b_se   = mb.bse['mean_nav_time_samples']
    b_p    = mb.pvalues['mean_nav_time_samples']
    c_prime     = mb.params['gamma_between']
    c_prime_se  = mb.bse['gamma_between']
    c_prime_p   = mb.pvalues['gamma_between']

    indirect = a * b

    return dict(
        a=a, a_se=a_se, a_p=a_p,
        b=b, b_se=b_se, b_p=b_p,
        c=c, c_se=c_se, c_p=c_p,
        c_prime=c_prime, c_prime_se=c_prime_se, c_prime_p=c_prime_p,
        indirect=indirect,
    )


def bootstrap_indirect(df, n_boot=N_BOOT, seed=RANDOM_SEED):
    """
    Percentile bootstrap of indirect effect (a × b).
    Returns array of bootstrap estimates.
    """
    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot)
    n = len(df)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        sample = df.iloc[idx].reset_index(drop=True)
        try:
            ma = smf.ols('mean_nav_time_samples ~ gamma_between', data=sample).fit()
            mb = smf.ols('SP_clustering ~ gamma_between + mean_nav_time_samples', data=sample).fit()
            boot[i] = ma.params['gamma_between'] * mb.params['mean_nav_time_samples']
        except Exception:
            boot[i] = np.nan
    return boot[~np.isnan(boot)]


# ─── run all models ───────────────────────────────────────────────────────────

print(f"\nRunning mediation models (N_BOOT={N_BOOT}) …")
results_rows  = []
boot_store    = {}   # keyed by (phase, comp)

for phase in PHASES:
    for comp in COMPS:
        label = f'{PHASE_LABEL[phase]} | {COMP_LABEL[comp]}'
        print(f"  {label} …", end=' ', flush=True)

        df = make_subject_df(phase, comp)
        n  = len(df)

        if n < 5:
            print(f"SKIPPED (n={n})")
            continue

        # point estimates
        est  = run_mediation(df)
        boot = bootstrap_indirect(df, n_boot=N_BOOT)
        ci_lo, ci_hi = np.percentile(boot, [2.5, 97.5])
        boot_p = 2 * min(np.mean(boot >= 0), np.mean(boot <= 0))   # two-sided

        sig_indirect = not (ci_lo <= 0 <= ci_hi)
        prop_mediated = (est['indirect'] / est['c']) if est['c'] != 0 else np.nan

        boot_store[(phase, comp)] = boot

        results_rows.append({
            'phase': phase, 'component': comp,
            'n_subjects': n,
            # path a
            'a_coef': est['a'], 'a_se': est['a_se'], 'a_p': est['a_p'],
            # path b
            'b_coef': est['b'], 'b_se': est['b_se'], 'b_p': est['b_p'],
            # path c (total)
            'c_coef': est['c'], 'c_se': est['c_se'], 'c_p': est['c_p'],
            # path c' (direct)
            'c_prime_coef': est['c_prime'], 'c_prime_se': est['c_prime_se'],
            'c_prime_p': est['c_prime_p'],
            # indirect
            'indirect_coef': est['indirect'],
            'boot_ci_lo': ci_lo, 'boot_ci_hi': ci_hi,
            'boot_p': boot_p, 'sig_indirect': sig_indirect,
            'prop_mediated': prop_mediated,
            'n_boot_valid': len(boot),
        })
        print(f"done  n={n}  indirect={est['indirect']:+.4f}  "
              f"95%CI=[{ci_lo:+.4f}, {ci_hi:+.4f}]  sig={sig_indirect}")

results = pd.DataFrame(results_rows)
results.to_csv('mediation_SP_results.csv', index=False)
print("\nmediation_SP_results.csv saved ✓")


# ─── console report ───────────────────────────────────────────────────────────

def p_star(p):
    if pd.isna(p): return ''
    if p < 0.001:  return '***'
    if p < 0.01:   return '**'
    if p < 0.05:   return '*'
    return 'n.s.'

SEP = '─' * 80
lines = []; L = lines.append

L('=' * 80)
L('MEDIATION ANALYSIS: Gamma → Navigation Time → SP Clustering')
L(f'Bootstrap iterations: {N_BOOT}  |  SP clustering only (clust_T=0)')
L(f'95% CI: percentile bootstrap  |  * p<.05  ** p<.01  *** p<.001')
L('=' * 80)

for _, r in results.iterrows():
    L(f"\n{SEP}")
    L(f"  {PHASE_LABEL[r['phase']].upper()} | {COMP_LABEL[r['component']]}  "
      f"(n={int(r['n_subjects'])} subjects)")
    L(SEP)
    L(f"  Path a  (gamma → nav time)      : β={r['a_coef']:+.4f}  "
      f"SE={r['a_se']:.4f}  p={r['a_p']:.3f} {p_star(r['a_p'])}")
    L(f"  Path b  (nav → SP clust | gamma): β={r['b_coef']:+.4f}  "
      f"SE={r['b_se']:.4f}  p={r['b_p']:.3f} {p_star(r['b_p'])}")
    L(f"  Path c  (total effect)          : β={r['c_coef']:+.4f}  "
      f"SE={r['c_se']:.4f}  p={r['c_p']:.3f} {p_star(r['c_p'])}")
    L(f"  Path c' (direct effect)         : β={r['c_prime_coef']:+.4f}  "
      f"SE={r['c_prime_se']:.4f}  p={r['c_prime_p']:.3f} {p_star(r['c_prime_p'])}")
    L(f"  Indirect (a×b)                  : {r['indirect_coef']:+.4f}  "
      f"95% CI [{r['boot_ci_lo']:+.4f}, {r['boot_ci_hi']:+.4f}]  "
      f"p_boot={r['boot_p']:.3f} {p_star(r['boot_p'])}")
    L(f"  CI excludes zero (sig)          : {r['sig_indirect']}")
    L(f"  Proportion mediated             : {r['prop_mediated']:.3f}" if pd.notna(r['prop_mediated']) else "  Proportion mediated: N/A")

    # interpret
    if r['sig_indirect']:
        if pd.notna(r['c_prime_p']) and r['c_prime_p'] < 0.05:
            L("  ► PARTIAL MEDIATION: indirect significant; direct effect remains")
        else:
            L("  ► FULL MEDIATION: indirect significant; direct effect n.s.")
    else:
        L("  ► NO MEDIATION: indirect CI spans zero")

L(f"\n{'=' * 80}")
L("NOTES")
L("  • Mediation run on SP clustering only — T clustering excluded by design.")
L("  • With N≈26, power to detect partial vs full mediation is limited.")
L("  • Proportion mediated = indirect / total; interpret cautiously when")
L("    paths a or c are small (can exceed 1 or be negative).")
L("  • Bootstrap p-values are two-sided (proportion of boot samples")
L("    on opposite side of zero).")
L('=' * 80)

report = '\n'.join(lines)
print(report)
with open('mediation_SP_report.txt', 'w') as f:
    f.write(report)
print("\nmediation_SP_report.txt saved ✓")


# ════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — Path diagram grid (2 phases × 2 components)
# ════════════════════════════════════════════════════════════════════════════
print("\nBuilding Figure 1: path diagrams …")

def draw_path_diagram(ax, row, title):
    """Draw a simple 3-node path diagram with annotated coefficients."""
    ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis('off')
    ax.set_title(title, fontsize=9, fontweight='bold', pad=4)

    # node positions
    pos = {'gamma': (1.2, 3.0), 'nav': (5.0, 5.2), 'clust': (8.8, 3.0)}
    node_r = 0.9

    node_labels = {
        'gamma': 'Gamma\n(between)',
        'nav':   'Nav Time\n(mean)',
        'clust': 'SP\nClustering',
    }
    node_colors = {'gamma': C_GAMMA, 'nav': C_NAV, 'clust': C_CLUST}

    for key, (x, y) in pos.items():
        circ = plt.Circle((x, y), node_r, color=node_colors[key],
                           alpha=0.85, zorder=3)
        ax.add_patch(circ)
        ax.text(x, y, node_labels[key], ha='center', va='center',
                fontsize=7.5, fontweight='bold', zorder=4, color='white')

    def arrow(src, dst, coef, p, label_offset=(0, 0.25), is_indirect=False):
        sx, sy = pos[src]; dx, dy = pos[dst]
        # shorten to node edge
        dx_vec = dx - sx; dy_vec = dy - sy
        dist   = np.sqrt(dx_vec**2 + dy_vec**2)
        sx2 = sx + node_r * dx_vec / dist
        sy2 = sy + node_r * dy_vec / dist
        dx2 = dx - node_r * dx_vec / dist
        dy2 = dy - node_r * dy_vec / dist
        color = C_SIG if (pd.notna(p) and p < 0.05) else '#aaaaaa'
        lw    = 2.5 if (pd.notna(p) and p < 0.05) else 1.2
        ls    = '-' if not is_indirect else '--'
        ax.annotate('', xy=(dx2, dy2), xytext=(sx2, sy2),
                    arrowprops=dict(arrowstyle='->', color=color, lw=lw, ls=ls),
                    zorder=2)
        mx = (sx2 + dx2) / 2 + label_offset[0]
        my = (sy2 + dy2) / 2 + label_offset[1]
        stars = p_star(p)
        txt = f"β={coef:+.3f}{stars}" if pd.notna(coef) else ''
        ax.text(mx, my, txt, ha='center', va='bottom', fontsize=7,
                color=color, fontweight='bold' if pd.notna(p) and p < 0.05 else 'normal')

    # path a: gamma → nav
    arrow('gamma', 'nav',   row['a_coef'], row['a_p'], label_offset=(-0.5, 0.15))
    # path b: nav → clust
    arrow('nav',   'clust', row['b_coef'], row['b_p'], label_offset=(0.5, 0.15))
    # path c': gamma → clust (direct)
    arrow('gamma', 'clust', row['c_prime_coef'], row['c_prime_p'],
          label_offset=(0, -0.45))

    # indirect effect box
    sig   = row['sig_indirect']
    icolor = C_SIG if sig else C_NSIG
    ci_txt = (f"Indirect (a×b): {row['indirect_coef']:+.4f}\n"
              f"95% CI [{row['boot_ci_lo']:+.3f}, {row['boot_ci_hi']:+.3f}]\n"
              f"{'✓ CI excl. 0' if sig else '✗ CI incl. 0'}")
    ax.text(5.0, 1.0, ci_txt, ha='center', va='center', fontsize=7.5,
            color=icolor, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec=icolor, lw=1.5))

    # proportion mediated
    if pd.notna(row['prop_mediated']):
        ax.text(5.0, 0.05, f"Prop. mediated: {row['prop_mediated']:.2f}",
                ha='center', va='bottom', fontsize=7, color='#555555')


fig1, axes1 = plt.subplots(2, 2, figsize=(13, 10), constrained_layout=True)
fig1.suptitle(
    'Mediation: Gamma Power → Navigation Time → SP Clustering\n'
    '(SP only; bootstrapped 95% CI for indirect effect; green arrow = p<.05)',
    fontsize=12, fontweight='bold'
)

for ri, phase in enumerate(PHASES):
    for ci, comp in enumerate(COMPS):
        ax  = axes1[ri, ci]
        row = results[(results['phase'] == phase) & (results['component'] == comp)]
        if row.empty:
            ax.axis('off'); continue
        draw_path_diagram(ax, row.iloc[0],
                          f'{PHASE_LABEL[phase]} | {COMP_LABEL[comp]}')

plt.savefig('fig_mediation_paths.png', dpi=150, bbox_inches='tight')
print("  → fig_mediation_paths.png saved")
plt.show()


# ════════════════════════════════════════════════════════════════════════════
# FIGURE 2 — Bootstrap distributions of indirect effect
# ════════════════════════════════════════════════════════════════════════════
print("Building Figure 2: bootstrap distributions …")

fig2, axes2 = plt.subplots(2, 2, figsize=(11, 8), constrained_layout=True)
fig2.suptitle(
    f'Bootstrap Distribution of Indirect Effect (a×b), N_boot={N_BOOT}\n'
    'Gamma → Nav Time → SP Clustering  |  Shaded = 95% CI',
    fontsize=12, fontweight='bold'
)

for ri, phase in enumerate(PHASES):
    for ci, comp in enumerate(COMPS):
        ax  = axes2[ri, ci]
        key = (phase, comp)
        if key not in boot_store:
            ax.axis('off'); continue

        boot = boot_store[key]
        row  = results[(results['phase'] == phase) & (results['component'] == comp)].iloc[0]

        ci_lo = row['boot_ci_lo']; ci_hi = row['boot_ci_hi']
        sig   = row['sig_indirect']
        shade = C_SIG if sig else C_NSIG

        ax.hist(boot, bins=60, color='#5B9BD5', alpha=0.6,
                edgecolor='none', density=True, label='Bootstrap dist.')

        # shade CI
        x_fill = boot[(boot >= ci_lo) & (boot <= ci_hi)]
        if len(x_fill) > 0:
            ax.axvspan(ci_lo, ci_hi, alpha=0.15, color=shade)

        ax.axvline(0,                  color='black',  lw=1.5, ls='--', label='Zero')
        ax.axvline(row['indirect_coef'], color='#e15759', lw=2.0, ls='-',
                   label=f'Point est. ({row["indirect_coef"]:+.4f})')
        ax.axvline(ci_lo, color=shade, lw=1.5, ls=':', alpha=0.8)
        ax.axvline(ci_hi, color=shade, lw=1.5, ls=':', alpha=0.8,
                   label=f'95% CI [{ci_lo:+.3f}, {ci_hi:+.3f}]')

        sig_txt = '✓ Significant' if sig else '✗ Not significant'
        ax.set_title(f'{PHASE_LABEL[phase]} | {COMP_LABEL[comp]}\n{sig_txt}',
                     fontsize=9.5, fontweight='bold',
                     color=shade)
        ax.set_xlabel('Indirect effect (a × b)', fontsize=8.5)
        ax.set_ylabel('Density', fontsize=8.5)
        ax.legend(fontsize=7, framealpha=0.6)
        ax.tick_params(labelsize=8)
        ax.spines[['top', 'right']].set_visible(False)

plt.savefig('fig_mediation_bootstrap.png', dpi=150, bbox_inches='tight')
print("  → fig_mediation_bootstrap.png saved")
plt.show()


# ════════════════════════════════════════════════════════════════════════════
# FIGURE 3 — Forest plot of indirect effects across conditions
# ════════════════════════════════════════════════════════════════════════════
print("Building Figure 3: forest plot …")

fig3, ax3 = plt.subplots(figsize=(8, 4.5), constrained_layout=True)

y_pos   = list(range(len(results)))
labels  = [f"{PHASE_LABEL[r['phase']]} | {COMP_LABEL[r['component']]}"
           for _, r in results.iterrows()]
colors  = [C_SIG if r['sig_indirect'] else C_NSIG for _, r in results.iterrows()]
indirects = results['indirect_coef'].values
ci_los    = results['boot_ci_lo'].values
ci_his    = results['boot_ci_hi'].values

for yi, (est, lo, hi, col) in enumerate(zip(indirects, ci_los, ci_his, colors)):
    ax3.errorbar(est, yi,
                 xerr=[[est - lo], [hi - est]],
                 fmt='o', color=col, ms=8, lw=2, capsize=5, zorder=3)

ax3.axvline(0, color='black', lw=1.5, ls='--', zorder=2)
ax3.set_yticks(y_pos)
ax3.set_yticklabels(labels, fontsize=10)
ax3.set_xlabel('Indirect Effect (a × b) with 95% Bootstrap CI', fontsize=10)
ax3.set_title('Forest Plot: Mediation Indirect Effects\n'
              'Gamma → Nav Time → SP Clustering  (green = CI excludes 0)',
              fontsize=11, fontweight='bold')
ax3.spines[['top', 'right']].set_visible(False)

sig_patch  = mpatches.Patch(color=C_SIG,  label='Significant (CI excl. 0)')
nsig_patch = mpatches.Patch(color=C_NSIG, label='Not significant')
ax3.legend(handles=[sig_patch, nsig_patch], fontsize=9, loc='lower right')

plt.savefig('fig_mediation_forest.png', dpi=150, bbox_inches='tight')
print("  → fig_mediation_forest.png saved")
plt.show()


# ─── final summary ────────────────────────────────────────────────────────────

print('\n' + '=' * 70)
print('OUTPUT FILES')
print('=' * 70)
print('  mediation_SP_results.csv      — full numeric results table')
print('  mediation_SP_report.txt       — formatted text report')
print('  fig_mediation_paths.png       — path diagrams (2×2 grid)')
print('  fig_mediation_bootstrap.png   — bootstrap distributions (2×2 grid)')
print('  fig_mediation_forest.png      — forest plot of indirect effects')
print('=' * 70)
print('Done.')

All required columns present ✓

Running mediation models (N_BOOT=5000) …
  Encoding | Oscillatory (OSC) … done  n=26  indirect=+0.0161  95%CI=[-0.0092, +0.0841]  sig=False
  Encoding | Aperiodic (FRAC) … done  n=26  indirect=+0.0158  95%CI=[-0.0113, +0.0456]  sig=False
  Retrieval | Oscillatory (OSC) … done  n=26  indirect=+0.0220  95%CI=[-0.0176, +0.0893]  sig=False
  Retrieval | Aperiodic (FRAC) … done  n=26  indirect=+0.0206  95%CI=[-0.0079, +0.0484]  sig=False

mediation_SP_results.csv saved ✓
MEDIATION ANALYSIS: Gamma → Navigation Time → SP Clustering
Bootstrap iterations: 5000  |  SP clustering only (clust_T=0)
95% CI: percentile bootstrap  |  * p<.05  ** p<.01  *** p<.001

────────────────────────────────────────────────────────────────────────────────
  ENCODING | Oscillatory (OSC)  (n=26 subjects)
────────────────────────────────────────────────────────────────────────────────
  Path a  (gamma → nav time)      : β=-21331.7234  SE=13183.8568  p=0.119 n.s.
  Path b  (nav → SP cl

In [38]:
"""
Gamma-band iEEG LMM Analysis — Final Version
=============================================
Fixes applied:
  1. freq_hz removed from between-subject model
     — between aggregation is subject × clust_T × region only (averaged over freq_hz)
     — freq_hz was unidentified at Level-2 (same 6 values per every subject cell)
     — freq_hz retained in within-subject model as continuous covariate
  2. FDR correction (Benjamini-Hochberg) applied across all tests within each model level
  3. Effect sizes reported: partial eta-squared (η²p) for all effects;
     Cohen's d for neural main effects
  4. Key interaction visualized: Neural × clustering_type for OSC and FRAC,
     encoding and retrieval (2×2 panel figure)

Model structure
---------------
BETWEEN-SUBJECT (Level-2 OLS):
  clustering_score_j ~ neural_between_j + clust_T + region_RHP
                     + neural_between_j:clust_T
                     + neural_between_j:region_RHP
  Data: aggregated to subject × clust_T × region means (averaged over freq_hz)

WITHIN-SUBJECT (LMM):
  clustering_score_ij ~ neural_within_ij + clust_T + region_RHP + freq_hz
                      + neural_within_ij:clust_T
                      + neural_within_ij:region_RHP
                      + neural_within_ij:freq_hz
                      + (1 + neural_within | subject)
                      + (1 | subject:session)
  Optimizer: powell
"""

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# LOAD & FILTER TO GAMMA
# ============================================================================

df = pd.read_csv('ALL_SUBJECTS_irasa_clean.csv')

gamma = df[df['freq_hz'] >= 40].copy()
gamma = gamma[gamma['SP_clustering_from_prev'].notna()].copy()

# ── gamma_avg: averaged over freq bins, used for BETWEEN-subject model ───────
# (one row per subject × session × trial × recalled_word × region)
# freq_hz is intentionally dropped here — between model does not need it.
agg_cols = ['encoding_osc_ssl', 'encoding_frac_ssl',
            'retrieval_osc_ssl', 'retrieval_frac_ssl',
            'SP_clustering_from_prev', 'T_clustering_from_prev']
gamma_avg = (
    gamma.groupby(['subject', 'session', 'trial', 'recalled_word', 'region'])[agg_cols]
    .mean()
    .reset_index()
)

# ── long_raw: raw gamma rows (all freq bins), used for WITHIN-subject model ──
# freq_hz is preserved here as a continuous covariate.
sp_raw = gamma.copy()
sp_raw['clustering_score'] = sp_raw['SP_clustering_from_prev']
sp_raw['clustering_type']  = 'SP'

t_raw = gamma.copy()
t_raw['clustering_score'] = t_raw['T_clustering_from_prev']
t_raw['clustering_type']  = 'T'

long_raw = pd.concat([sp_raw, t_raw], ignore_index=True)
long_raw['clust_T']    = (long_raw['clustering_type'] == 'T').astype(int)
long_raw['region_RHP'] = (long_raw['region'] == 'RHP').astype(int)
long_raw['sess_id']    = long_raw['subject'].astype(str) + '_' + long_raw['session'].astype(str)

# ── long_avg: averaged gamma rows, used for BETWEEN-subject model ────────────
sp_avg = gamma_avg.copy()
sp_avg['clustering_score'] = sp_avg['SP_clustering_from_prev']
sp_avg['clustering_type']  = 'SP'

t_avg = gamma_avg.copy()
t_avg['clustering_score'] = t_avg['T_clustering_from_prev']
t_avg['clustering_type']  = 'T'

long_avg = pd.concat([sp_avg, t_avg], ignore_index=True)
long_avg['clust_T']    = (long_avg['clustering_type'] == 'T').astype(int)
long_avg['region_RHP'] = (long_avg['region'] == 'RHP').astype(int)

# ============================================================================
# BETWEEN / WITHIN DECOMPOSITION
# Subject mean computed on gamma_avg (pre-stack, one row per trial × region).
# This ensures encoding and retrieval get independent subject means.
# Mapped onto both long_avg (between model) and long_raw (within model).
# ============================================================================

for col in ['encoding_osc_ssl', 'encoding_frac_ssl',
            'retrieval_osc_ssl', 'retrieval_frac_ssl']:
    subj_mean_map = gamma_avg.groupby('subject')[col].mean()
    # Between: mapped onto averaged long
    long_avg[col + '_between'] = long_avg['subject'].map(subj_mean_map)
    # Within: deviation from subject mean, mapped onto raw long (preserves freq_hz)
    long_raw[col + '_between'] = long_raw['subject'].map(subj_mean_map)
    long_raw[col + '_within']  = long_raw[col] - long_raw[col + '_between']

# ============================================================================
# EFFECT SIZE HELPERS
# ============================================================================

def partial_eta_sq(t_val, df_resid):
    """Partial eta-squared from t-statistic and residual df."""
    if pd.isna(t_val) or pd.isna(df_resid):
        return np.nan
    return t_val**2 / (t_val**2 + df_resid)

def cohens_d_from_t(t_val, n):
    """Approximate Cohen's d from t and sample size (one-sample style)."""
    if pd.isna(t_val) or pd.isna(n) or n == 0:
        return np.nan
    return t_val / np.sqrt(n)

# ============================================================================
# MODELS
# ============================================================================

PHASES = ['encoding', 'retrieval']
COMPS  = ['osc', 'frac']

results_rows = []

for phase in PHASES:
    for comp in COMPS:

        col_base    = f'{phase}_{comp}_ssl'
        col_between = col_base + '_between'
        col_within  = col_base + '_within'

        # ── BETWEEN-SUBJECT MODEL (Level-2 OLS on long_avg) ──────────────
        between_data = (
            long_avg.groupby(['subject', 'clust_T', 'region_RHP'])
            [[col_between, 'clustering_score']]
            .mean()
            .reset_index()
            .dropna()
        )

        try:
            formula_b = (
                f'clustering_score ~ {col_between} + clust_T + region_RHP '
                f'+ {col_between}:clust_T '
                f'+ {col_between}:region_RHP'
            )
            res_b     = smf.ols(formula_b, data=between_data).fit()
            df_resid_b = res_b.df_resid
            n_subj     = between_data['subject'].nunique()

            def gb(key):
                t = res_b.tvalues.get(key, np.nan)
                p = res_b.pvalues.get(key, np.nan)
                e = partial_eta_sq(t, df_resid_b)
                return t, p, e

            t_b,          p_b,          e_b          = gb(col_between)
            t_b_xclust,   p_b_xclust,   e_b_xclust   = gb(f'{col_between}:clust_T')
            t_b_xregion,  p_b_xregion,  e_b_xregion  = gb(f'{col_between}:region_RHP')
            t_clust_b,    p_clust_b,    e_clust_b    = gb('clust_T')
            t_region_b,   p_region_b,   e_region_b   = gb('region_RHP')
            d_b           = cohens_d_from_t(t_b, n_subj)
            n_b           = int(res_b.nobs)
            note_b        = 'ok'

        except Exception as e:
            (t_b, p_b, e_b, t_b_xclust, p_b_xclust, e_b_xclust,
             t_b_xregion, p_b_xregion, e_b_xregion,
             t_clust_b, p_clust_b, e_clust_b,
             t_region_b, p_region_b, e_region_b,
             d_b, n_b, n_subj) = [np.nan] * 18
            note_b = f'failed: {e}'

        # ── WITHIN-SUBJECT MODEL (LMM on long_raw — preserves freq_hz) ───
        # Random intercept only: random slope was degenerate with small N (5 subjects
        # in available data). Intercept-only is the appropriate fallback.
        # Session nesting retained via vc_formula.
        within_data = long_raw[[
            'subject', 'sess_id', 'clustering_score',
            col_within, 'clust_T', 'region_RHP', 'freq_hz'
        ]].dropna().copy()
        within_data = within_data.rename(columns={col_within: 'neural_within'})

        try:
            formula_w = (
                'clustering_score ~ neural_within + clust_T + region_RHP + freq_hz '
                '+ neural_within:clust_T '
                '+ neural_within:region_RHP '
                '+ neural_within:freq_hz'
            )
            md = smf.mixedlm(
                formula_w,
                data=within_data,
                groups=within_data['subject'],
                vc_formula={'sess_id': '0 + C(sess_id)'}
            )
            res_w      = md.fit(reml=True, method='powell')
            df_resid_w = res_w.df_resid

            def gw(key):
                t = res_w.tvalues.get(key, np.nan)
                p = res_w.pvalues.get(key, np.nan)
                e = partial_eta_sq(t, df_resid_w)
                return t, p, e

            t_w,          p_w,          e_w          = gw('neural_within')
            t_w_xclust,   p_w_xclust,   e_w_xclust   = gw('neural_within:clust_T')
            t_w_xregion,  p_w_xregion,  e_w_xregion  = gw('neural_within:region_RHP')
            t_w_xfreq,    p_w_xfreq,    e_w_xfreq    = gw('neural_within:freq_hz')
            t_clust_w,    p_clust_w,    e_clust_w    = gw('clust_T')
            t_region_w,   p_region_w,   e_region_w   = gw('region_RHP')
            t_freq_w,     p_freq_w,     e_freq_w     = gw('freq_hz')
            d_w    = cohens_d_from_t(t_w, within_data['subject'].nunique())
            n_w    = int(res_w.nobs)
            conv_w = res_w.converged
            note_w = 'ok' if conv_w else 'ok (convergence warning)'

        except Exception as e:
            (t_w, p_w, e_w, t_w_xclust, p_w_xclust, e_w_xclust,
             t_w_xregion, p_w_xregion, e_w_xregion,
             t_w_xfreq, p_w_xfreq, e_w_xfreq,
             t_clust_w, p_clust_w, e_clust_w,
             t_region_w, p_region_w, e_region_w,
             t_freq_w, p_freq_w, e_freq_w,
             d_w, n_w) = [np.nan] * 23
            note_w = f'failed: {e}'

        results_rows.append({
            'phase': phase, 'component': comp,
            # Between
            't_between':          t_b,         'p_between':          p_b,         'eta2p_between':          e_b,
            't_between_x_clustT': t_b_xclust,  'p_between_x_clustT': p_b_xclust,  'eta2p_between_x_clustT': e_b_xclust,
            't_between_x_RHP':    t_b_xregion, 'p_between_x_RHP':    p_b_xregion, 'eta2p_between_x_RHP':    e_b_xregion,
            't_clustT_between':   t_clust_b,   'p_clustT_between':   p_clust_b,   'eta2p_clustT_between':   e_clust_b,
            't_region_between':   t_region_b,  'p_region_between':   p_region_b,  'eta2p_region_between':   e_region_b,
            'd_between':          d_b,
            'n_between':          n_b,         'n_subjects':         n_subj,      'note_between': note_b,
            # Within
            't_within':           t_w,         'p_within':           p_w,         'eta2p_within':           e_w,
            't_within_x_clustT':  t_w_xclust,  'p_within_x_clustT':  p_w_xclust,  'eta2p_within_x_clustT':  e_w_xclust,
            't_within_x_RHP':     t_w_xregion, 'p_within_x_RHP':     p_w_xregion, 'eta2p_within_x_RHP':     e_w_xregion,
            't_within_x_freq':    t_w_xfreq,   'p_within_x_freq':    p_w_xfreq,   'eta2p_within_x_freq':    e_w_xfreq,
            't_clustT_within':    t_clust_w,   'p_clustT_within':    p_clust_w,   'eta2p_clustT_within':    e_clust_w,
            't_region_within':    t_region_w,  'p_region_within':    p_region_w,  'eta2p_region_within':    e_region_w,
            't_freq_within':      t_freq_w,    'p_freq_within':      p_freq_w,    'eta2p_freq_within':      e_freq_w,
            'd_within':           d_w,
            'n_within':           n_w,         'note_within':        note_w,
        })

results = pd.DataFrame(results_rows)

# ============================================================================
# FIX 2: FDR CORRECTION (Benjamini-Hochberg)
# Applied separately within between-subject and within-subject levels
# across all phase × component × effect combinations
# ============================================================================

def apply_fdr(results, p_cols, suffix):
    """Apply BH-FDR across all p-values in p_cols using statsmodels multipletests."""
    all_p = []
    for pc in p_cols:
        all_p.extend(results[pc].tolist())
    all_p = np.array(all_p, dtype=float)
    valid_mask = ~np.isnan(all_p)
    fdr_p = np.full_like(all_p, np.nan)
    if valid_mask.sum() > 0:
        # multipletests returns (reject, p_corrected, alphacSidak, alphacBonf)
        _, p_corrected, _, _ = multipletests(all_p[valid_mask], method='fdr_bh')
        fdr_p[valid_mask] = p_corrected
    # Put back into results columns
    idx = 0
    for pc in p_cols:
        n = len(results)
        col_fdr = pc.replace('p_', 'pfdr_')
        col_sig = pc.replace('p_', 'sig_fdr_')
        results[col_fdr] = fdr_p[idx:idx+n]
        results[col_sig] = results[col_fdr].apply(
            lambda x: x < 0.05 if pd.notna(x) else False)
        idx += n
    return results

between_p_cols = ['p_between', 'p_between_x_clustT', 'p_between_x_RHP',
                  'p_clustT_between', 'p_region_between']
within_p_cols  = ['p_within', 'p_within_x_clustT', 'p_within_x_RHP',
                  'p_within_x_freq', 'p_clustT_within', 'p_region_within', 'p_freq_within']

results = apply_fdr(results, between_p_cols, 'between')
results = apply_fdr(results, within_p_cols,  'within')

# Also flag uncorrected significance
for pc in between_p_cols + within_p_cols:
    results['sig_unc_' + pc[2:]] = results[pc].apply(
        lambda x: x < 0.05 if pd.notna(x) else False)

# ============================================================================
# SAVE RESULTS
# ============================================================================

results.to_csv('gamma_lmm_results.csv', index=False)

# ============================================================================
# FIX 4: VISUALIZATION
# Key interaction: Neural × clustering_type (OSC vs FRAC, encoding vs retrieval)
# 2×2 panel: rows = phase, cols = component
# For between-subject: scatter of subject means, split by clust_T
# ============================================================================

def get_between_plot_data(long_avg, col_between, phase, comp):
    """Get subject-level means split by clustering type for scatter plot."""
    data = (
        long_avg.groupby(['subject', 'clust_T'])
        [[col_between, 'clustering_score']]
        .mean()
        .reset_index()
        .dropna()
    )
    sp_data = data[data['clust_T'] == 0].copy()
    t_data  = data[data['clust_T'] == 1].copy()
    return sp_data, t_data

# Color palette — publication quality
COL_SP   = '#2166AC'   # blue for SP clustering
COL_T    = '#D6604D'   # red for T clustering
COL_GRID = '#E8E8E8'

fig = plt.figure(figsize=(12, 10))
fig.patch.set_facecolor('white')
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.38)

panel_labels = [['A', 'B'], ['C', 'D']]

for pi, phase in enumerate(PHASES):
    for ci, comp in enumerate(COMPS):

        ax = fig.add_subplot(gs[pi, ci])
        col_base    = f'{phase}_{comp}_ssl'
        col_between = col_base + '_between'

        sp_data, t_data = get_between_plot_data(long_avg, col_between, phase, comp)

        # Scatter
        ax.scatter(sp_data[col_between], sp_data['clustering_score'],
                   color=COL_SP, s=60, zorder=3, alpha=0.85,
                   label='SP clustering', edgecolors='white', linewidths=0.5)
        ax.scatter(t_data[col_between],  t_data['clustering_score'],
                   color=COL_T,  s=60, zorder=3, alpha=0.85,
                   label='T clustering',  edgecolors='white', linewidths=0.5)

        # Regression lines
        for dat, col in [(sp_data, COL_SP), (t_data, COL_T)]:
            x = dat[col_between].values
            y = dat['clustering_score'].values
            if len(x) > 1:
                m, b = np.polyfit(x, y, 1)
                xr = np.linspace(x.min(), x.max(), 100)
                ax.plot(xr, m*xr + b, color=col, linewidth=2, alpha=0.9)

        # Get stats for annotation
        row = results[(results['phase']==phase) & (results['component']==comp)].iloc[0]
        t_main = row['t_between'];          p_main = row['pfdr_between']
        t_int  = row['t_between_x_clustT']; p_int  = row['pfdr_between_x_clustT']
        d_main = row['d_between']

        def pstr(p):
            if pd.isna(p):   return 'n.s.'
            if p < 0.001:    return 'p<.001'
            if p < 0.01:     return f'p={p:.3f}'
            if p < 0.05:     return f'p={p:.3f}'
            return f'p={p:.2f} n.s.'

        ann = (f"Neural main: t={t_main:+.2f}, {pstr(p_main)}, d={d_main:.2f}\n"
               f"× Clust type: t={t_int:+.2f}, {pstr(p_int)}")
        ax.text(0.03, 0.97, ann, transform=ax.transAxes,
                fontsize=7.5, va='top', ha='left',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='#F7F7F7',
                          edgecolor='#CCCCCC', alpha=0.9))

        # Panel label
        ax.text(-0.12, 1.05, panel_labels[pi][ci], transform=ax.transAxes,
                fontsize=14, fontweight='bold', va='top')

        # Labels
        comp_label = 'Oscillatory' if comp == 'osc' else 'Aperiodic (Fractal)'
        phase_label = phase.capitalize()
        ax.set_title(f'{phase_label} — {comp_label} γ', fontsize=11, fontweight='bold', pad=8)
        ax.set_xlabel('Gamma power (subject mean, SSL)', fontsize=9)
        ax.set_ylabel('Clustering score', fontsize=9)

        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(True, color=COL_GRID, linewidth=0.7, zorder=0)
        ax.tick_params(labelsize=8)

        if pi == 0 and ci == 0:
            ax.legend(fontsize=8.5, framealpha=0.9, loc='lower right')

fig.suptitle(
    'Between-subject gamma power predicts semantic clustering\n'
    'across encoding and retrieval phases (iEEG, n=26)',
    fontsize=13, fontweight='bold', y=1.01
)

plt.savefig('gamma_neural_x_clustering_type.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print("Figure saved: gamma_neural_x_clustering_type.png")

# ============================================================================
# PRINT REPORT
# ============================================================================

lines = []
L = lines.append

L("=" * 110)
L("GAMMA-BAND iEEG LMM — Between- and Within-Subject Effects on Clustering  [FINAL]")
L("  Fix 1: freq_hz removed from between model (unidentified at Level-2)")
L("  Fix 2: FDR correction (Benjamini-Hochberg) applied within each model level")
L("  Fix 3: Effect sizes reported — partial η²p and Cohen's d")
L("  Fix 4: Figure saved — gamma_neural_x_clustering_type.png")
L("  Factors : clustering_type (SP=0/T=1), region (LHP=0/RHP=1)")
L("  Sig     : ** uncorrected p<.05  | †† FDR-corrected p<.05")
L("=" * 110)

def fmt(label, t, p_unc, p_fdr, eta2, d=None):
    sig_unc = '*'  if (pd.notna(p_unc) and p_unc < 0.05) else ''
    sig_fdr = '†' if (pd.notna(p_fdr) and p_fdr < 0.05) else ''
    ts   = f'{t:+.3f}'    if pd.notna(t)    else '    NaN'
    pu   = f'{p_unc:.3f}' if pd.notna(p_unc) else '    NaN'
    pf   = f'{p_fdr:.3f}' if pd.notna(p_fdr) else '    NaN'
    e2   = f'{eta2:.3f}'  if pd.notna(eta2)  else '    NaN'
    d_s  = f'{d:+.3f}'    if (d is not None and pd.notna(d)) else '      —'
    sigs = f'{sig_unc}{sig_fdr}'
    return (f"  {label:<40} {ts:>9} {pu:>9} {pf:>9} {e2:>8} {d_s:>9}  {sigs}")

HDR = (f"  {'Effect':<40} {'t':>9} {'p_unc':>9} {'p_fdr':>9} {'η²p':>8} {'d':>9}  sig")
SEP = f"  {'─' * 95}"

for _, row in results.iterrows():
    phase = row['phase']; comp = row['component']
    L(f"\n{'━' * 110}")
    L(f"  PHASE: {phase.upper()}   COMPONENT: {'OSCILLATORY (OSC)' if comp=='osc' else 'APERIODIC/FRACTAL (FRAC)'}")
    L(f"{'━' * 110}")

    # Between
    if row['note_between'] == 'ok':
        L(f"\n  BETWEEN-SUBJECT (Level-2 OLS)  "
          f"n_subjects={int(row['n_subjects'])}  n_obs={int(row['n_between'])}")
        L(f"  * uncorrected p<.05  † FDR-corrected p<.05")
        L(HDR); L(SEP)
        L(fmt('Neural main effect',         row['t_between'],         row['p_between'],         row['pfdr_between'],         row['eta2p_between'],         row['d_between']))
        L(fmt('Clustering type (T vs SP)',   row['t_clustT_between'],  row['p_clustT_between'],  row['pfdr_clustT_between'],  row['eta2p_clustT_between']))
        L(fmt('Region (RHP vs LHP)',         row['t_region_between'],  row['p_region_between'],  row['pfdr_region_between'],  row['eta2p_region_between']))
        L(fmt('Neural × clustering_type',    row['t_between_x_clustT'],row['p_between_x_clustT'],row['pfdr_between_x_clustT'],row['eta2p_between_x_clustT']))
        L(fmt('Neural × region',             row['t_between_x_RHP'],   row['p_between_x_RHP'],   row['pfdr_between_x_RHP'],   row['eta2p_between_x_RHP']))
    else:
        L(f"\n  BETWEEN-SUBJECT  *** {row['note_between']} ***")

    # Within
    status = row['note_within']
    if status.startswith('ok'):
        L(f"\n  WITHIN-SUBJECT (LMM: random slope + session nesting)  "
          f"n_obs={int(row['n_within'])}  [{status}]")
        L(f"  * uncorrected p<.05  † FDR-corrected p<.05")
        L(HDR); L(SEP)
        L(fmt('Neural main effect',         row['t_within'],          row['p_within'],          row['pfdr_within'],          row['eta2p_within'],          row['d_within']))
        L(fmt('Clustering type (T vs SP)',   row['t_clustT_within'],   row['p_clustT_within'],   row['pfdr_clustT_within'],   row['eta2p_clustT_within']))
        L(fmt('Region (RHP vs LHP)',         row['t_region_within'],   row['p_region_within'],   row['pfdr_region_within'],   row['eta2p_region_within']))
        L(fmt('Frequency (freq_hz)',         row['t_freq_within'],     row['p_freq_within'],     row['pfdr_freq_within'],     row['eta2p_freq_within']))
        L(fmt('Neural × clustering_type',    row['t_within_x_clustT'], row['p_within_x_clustT'], row['pfdr_within_x_clustT'], row['eta2p_within_x_clustT']))
        L(fmt('Neural × region',             row['t_within_x_RHP'],    row['p_within_x_RHP'],    row['pfdr_within_x_RHP'],    row['eta2p_within_x_RHP']))
        L(fmt('Neural × freq_hz',            row['t_within_x_freq'],   row['p_within_x_freq'],   row['pfdr_within_x_freq'],   row['eta2p_within_x_freq']))
    else:
        L(f"\n  WITHIN-SUBJECT  *** {status} ***")

# FDR summary
L(f"\n\n{'=' * 110}")
L("FDR CORRECTION SUMMARY")
L(f"{'=' * 110}")
L("Between-subject — effects surviving FDR correction:")
for _, row in results.iterrows():
    for pc, label in [('pfdr_between',         'Neural main'),
                      ('pfdr_between_x_clustT','Neural × clust_T'),
                      ('pfdr_between_x_RHP',   'Neural × region'),
                      ('pfdr_clustT_between',  'Clust type main'),
                      ('pfdr_region_between',  'Region main')]:
        p = row.get(pc, np.nan)
        if pd.notna(p) and p < 0.05:
            L(f"  [{row['phase'].upper()} | {'OSC' if row['component']=='osc' else 'FRAC'}]  "
              f"{label:<30} p_fdr={p:.3f}")

L("\nWithin-subject — effects surviving FDR correction:")
found_within = False
for _, row in results.iterrows():
    for pc, label in [('pfdr_within',          'Neural main'),
                      ('pfdr_within_x_clustT', 'Neural × clust_T'),
                      ('pfdr_within_x_RHP',    'Neural × region'),
                      ('pfdr_within_x_freq',   'Neural × freq_hz'),
                      ('pfdr_clustT_within',   'Clust type main'),
                      ('pfdr_region_within',   'Region main'),
                      ('pfdr_freq_within',     'Freq main')]:
        p = row.get(pc, np.nan)
        if pd.notna(p) and p < 0.05:
            found_within = True
            L(f"  [{row['phase'].upper()} | {'OSC' if row['component']=='osc' else 'FRAC'}]  "
              f"{label:<30} p_fdr={p:.3f}")
if not found_within:
    L("  None.")

L(f"\n{'=' * 110}")
L("FILES SAVED:")
L("  gamma_lmm_results.csv              — full results with FDR p-values and effect sizes")
L("  gamma_lmm_summary.csv              — FDR-significant effects only")
L("  gamma_neural_x_clustering_type.png — Figure: Neural × clustering_type (2×2 panel)")
L("=" * 110)

report = '\n'.join(lines)
print(report)

with open('gamma_lmm_report.txt', 'w') as f:
    f.write(report)

# Save summary
sig_fdr_cols = [c for c in results.columns if c.startswith('sig_fdr_')]
results[results[sig_fdr_cols].any(axis=1)].to_csv('gamma_lmm_summary.csv', index=False)

print("\nDone.")

Figure saved: gamma_neural_x_clustering_type.png
GAMMA-BAND iEEG LMM — Between- and Within-Subject Effects on Clustering  [FINAL]
  Fix 1: freq_hz removed from between model (unidentified at Level-2)
  Fix 2: FDR correction (Benjamini-Hochberg) applied within each model level
  Fix 3: Effect sizes reported — partial η²p and Cohen's d
  Fix 4: Figure saved — gamma_neural_x_clustering_type.png
  Factors : clustering_type (SP=0/T=1), region (LHP=0/RHP=1)
  Sig     : ** uncorrected p<.05  | †† FDR-corrected p<.05

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  PHASE: ENCODING   COMPONENT: OSCILLATORY (OSC)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  BETWEEN-SUBJECT (Level-2 OLS)  n_subjects=36  n_obs=108
  * uncorrected p<.05  † FDR-corrected p<.05
  Effect                                           t     p_unc     p_fdr      η²p         d  sig
  ──────────